# Deney 02 — Script / Ton Üretimi (Notebook 01 çıktısına bağlı)

**Amaç:** Notebook 01'in ürettiği gerçek `hot_moments.json` dosyasından bir an seçip, insan onayından sonra script üretimi için aday LLM'leri **kalite + maliyet + hız** üzerinden karşılaştırmak.

**Çalıştırma sırası:** Notebook 01 → (insan/LLM an onayı) → **bu notebook** → Notebook 03 → Notebook 06. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Tek kaynak politikası:** Prompt oluşturma, LLM yanıtı JSON ayrıştırma, sağlayıcı çağrıları ve `selected_script.json` şeması `local_pipeline/script_generation.py` içindeki **kanonik** fonksiyonlardan gelir; master notebook da aynılarını çağırır. Bu notebook'un kendine özgü katkısı **çok-model karşılaştırma döngüsüdür**.

API anahtarlarını Colab 🔑 Secrets panelinden ekle: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. Anahtarı olmayan sağlayıcı atlanır. Hiçbir anahtar yoksa notebook yine de tam bir **prompt paketi** hazırlayıp dışa aktarır — sahte script üretmiş gibi davranmaz.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **script uretimi** -> Rota A (`gemini-3.6-flash` -> `gemini-3.5-flash` -> `gemini-3.5-flash-lite` -> `gemma-4-31b-it`). Karsilastirma hucreleri de ayni router'i kullanir; her model tek tek, acikca cagrilir.


In [ ]:
!pip -q install openai anthropic google-genai pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, time, json, glob, hashlib, zipfile
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)
print('OpenAI:', 'var' if OPENAI_API_KEY else 'yok')
print('Anthropic:', 'var' if ANTHROPIC_API_KEY else 'yok')
print('Google:', 'var' if GOOGLE_API_KEY else 'yok')

# Paylasilan durum sozlugu (Faz 6) - tum notebooklarda ayni degerler kullanilir.
STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []
def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({
        'task': task, 'candidate': candidate, 'quality_score_1_5': quality_score,
        'cost_usd_estimate': round(cost_usd, 5), 'latency_sec': round(latency_sec, 2),
        'notes': notes, 'status': status,
    })
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} status={status}")

## `local_pipeline` kanonik modüllerini yükle

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['script_generation', 'video_analysis', 'approval', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'script_generation': {
        'sha256': '4a71343c730bf7563993694455a1890d2105b3984ffc2c4a0561a5c6cf728822',
        'encoding': 'zlib+b85',
        'data': (
            'c-q}PYjfK+w%_$Du<8d%U5Rnprgz3!?Tj{2Q{Osv)>gXH@p@!Pge<NpQp<-O)z|<1o(BK|BxO6<?&IFf#Fj|l05~{roC9~e-7{6@'
            'auF+WeEeSI%PiU!qidC@Ocqg^h&a8DmLgxvOod{x69?bSxzW686hzxyvgpVp6bA?MU0N39Ld|7aq#Jra-^jdB88-_$os+asi!{9f'
            'VB$upZ7!lB7wKIhHf3ByqivSnMxn}H@YhWms#q*kvRrRub`vGn;^UF%XJwMW=T5F}VIHwK-Ku1A^kR|}>nz<y%NOs`^cvPh-E)xx'
            'L|g#<#;L}tC=iwu>vAI#SbLhOPO(;-!D|7(1WaSIEyOZS@**pjxRRmx@AK1>QKs@OJcTAQ%K^ra0B%)gr!JN<jyrdm+`>~_Up4<+'
            'z$+0)w@R#4rbM2KMOv&yqOp=Qwndt#PPEy^YNL{ZHzR_XC>e@$CBkwWM@va7SY`4$f|b(T^ptqx5ZQ<x0j*OR0+KVaEf;Z=uNBNA'
            '*%fPO8whT+&7!2pUqH|N1x`E43v47NAC6$3c0s4x?RHjx6)~T$V3RX7p9`2`nibFt=1;TCJ01P(Z+V)S&qcIR+&7ek!f&Rh`97pJ'
            'zv8kx=d;P|<N5sO$??bOxfqN4j(~q%^$DgG5kBJXP;^6>cAReU!F(IbrCO&kARk|4>NZk$^GyA{jDU2zLxuqR0Xu}>MHjGHu2#`9'
            '!tO8&i4Rv2FiEeWVaU4<P-M6R5D%SBCsZr3D5E%>m$}O3OeFnW#cEloaK1@#HisgNmc<Zhr$FdPgNFjh2-eaC+JP~$(JE64NNFjz'
            'dUh@U3P6YLc&)PQC=oz78=&|bQSN~3q6}65^DSg{tzaD`^W;2N%R?ZT0y=(wuou3WU@VjCQeNw^{zsg^xX=@S4aDe=`1Fu5*zNua'
            'SDzD$s84cP2t<V8mK>nLtm!RCl%b$K2Odicjhj3mx<Fw3fN%x;ER&lTd0J*mHP2x+1unrnP)SH%VJD#Tivk+67#ajfln5*@H^5#y'
            'n04Q-cGaCE3|ZXQw&<dVk9(I7u+>icdb0qK+V9n;)f<Y|b~28x6L?lN91I>t&xWs8`4n{13}EMj!)HvNu?Ap!y}?5uW)^#gqP;_f'
            'U)_nv7|>7J(ZHev=uKVk3^%_yJ(*2UX7l68$-9q}chmEscsu=J^6_{!ufOULG@zRm3a@dqE6t@&p(w)kK^%y`T3kT02EdDb9WSZd'
            'S7HWKYX{0om{tVh7~~o#zaNgy&Sxa(fd`Sind7SVpu|+XIuud^1I$D*isEj%rha!>RA4g%Fioye3U`HEfocQ>nU$d0@-@kLt!>02'
            'PM0?@1r3n6E#A@Kpaw|~kN}77tR&YmUX5019H)2rP_P8!_XhkLG>irZ14ug&t@;CcmnjhLiN{CxdY>+O%;<B(4sef0eUls%Q7CUB'
            'k;^p*kn~rO>`ACVZ{D}arw3@3L^~BnH)0WG;+M3Xf#o31*FZ_Ro>3%r^4Gu{W|k&ExF{NdnBS)%d~pRWJUnL+3!*YZ#o~#}p|~!W'
            'aY-X0HpH^r#43+=u-gln$I>6{O>_fu2w^8-m^u;?xgjL<t2axS0X!ObP<rQ+x6?OM($FHL>ECyGeO+MSAP7+7;B1_)DhGv#q~{?G'
            '8vFso`X3)2{+j$X={Yk!uW)|&*W_NyL(96Xy0r18J^^*+eKZzS$l8kJJbti(9=Rb&lM(k3t4JBSuZ6wL!CG0RfYKz*HXxn=Sio#O'
            't6HHBR%l9G;R3IS6|UMBclBDJC9)INj@rs%O#ng3gs50TFz#>;8>C;H%PLNB24F@>!CD`>C1lX#@{XF&Hsc545(0N8VI^QKAV}iN'
            'Kui{-yg+c2iMEMvWeihC39u8AUynv;sS($q`9{E^<Q5ehsW}*l1SAfp5mx}V1A|oh&FDwN=7GLbw=rs2y(J|2m>>o?qi%N)z}sls'
            'XBix=sL!F@euj9#T`mC@ez-XN?%;rBTaoRY#s>M&4rob_d9jb;E(dzjO9iXx|D<-)EJK{nc3Y)C{*0KS&x7`H(3CJ!ubv|<RNww6'
            '4w@_4Hz|N=U%66?`)P8gl^Kr(?G7*o?#Rdi;|#?CTA#$m*2?NyM$mSU<%bUl40jn|kh^6406=0Y?t5BgYWTp$Ad1dtRykW#eHfCr'
            '1E>RC1SHaRS^^z{)eVq)Y=8C;G&ny47z&zZ&>loJ=o2#SfgW;bm4Gf_g)A4D@dczIy1TVv{veqHY7F=jSakzvhW?u<Cs1a}){64z'
            'gv=Pg39L6*fgta9@4JI)gOX6c%lPF3pzinLrELH^ts#uX=p_z|UH&lUr-z<RifdC274^F`Z_t3(Agar_ggugB<P6V53p2oN4amL0'
            '&AnjYFw*6~orw{*$i1k5b$6H)KBzeF!Zf_J9QV{r4HChz1AQjcO%4j}R_!D>cDXZSO`trE%TiPNg&H4(0z9h*B$uqe>R#OY)cSDA'
            '`zm)Ccxe#`U-9E)HaR~6%>n>`?tYXTu_WCU#W&)cg9C9RK?{Ph0ZkB3hw$w3ciBU8g2bww&R1fB-qI&xG4gI-z8@9o4bT-#YpL?r'
            'nuK#h!vz?c1ri+~2v#yZpB=rQ%%<l63K@O5R?8csKS;dya(09f9+Kl0?I@s9(8*bb(v}5q(D|xE#u>nk*((JH83DaHeg9!{c7&*S'
            '1M5FNeaDId9Q~w<GQcn(_oD^ID-=NTVhNx=wgKrY4pWG2w1vjNt(!70#L4vM=@}Sp)Rz?xsDLoVu6YW;R$wZNv@FnJH_WG19?OM1'
            '`UDG$Lc&(=4+Y#Uu2Y3tHA~CuHNd-K?^4SwyX|>@a3x-dtE^0#5g2$k1dflEYnVOYSX+vnuV_+i8{!V637C&Eyp_ok0}KKnl(<rq'
            'I^fO+ozoA~lgZKi{LR_XhuQr7>D%cs`rjA*?)A19y$Ze`At!=$@-jF$7>e)V=U^zRw|R<=`73()?E${M>`YE(Kc1a_IC|3tb{Weu'
            'RHL<wZpzVjqgNL6D+5}$047gVK@Gm81}|SRcv$4S)6;jy(?d1~(5o&YK<*iAiV_nY02(6GXZR=srU%*!#0R2cq9vRvki}90^#DL7'
            'VL-JtG61R|sId9?04O~MxfG%Y0K)a`!}}LA_<#1{t=<US$!m&0&1Dq)xl9Xrte|;zg^t1UCLfaE3BgzgdB7BYb*(m05{<qIUSTBZ'
            'ima5g>HE{4r*Cm}Fc3`y-u(@nHZb>daUlMVu!(D;g6yCOgI2(g0FmvWwFDl^%jHt3P%B-cZGH=Uh8_qAA04q<GM=D&zghup+Wc(#'
            '?&y4W_KS_%0V1v=q-B7+1rh@K%2OR104qDEuvLx%pbVn|jjBP=;r(io0|_D^CdR_IKu@b!f|E3){o$X3VZGI>&uKM^ftBk`fr#{6'
            'KAfHYeDrpDHaF75#-7rxN@N7qcN>eC=#q&&LbE+ouUj%+p;;R$DA2^tC@N{NL&21hU=FYiTa!{H+9agyYimHdB;BUK`UEE7a40=w'
            'v-r*A<n-j|&E(j^r+{Q1JCUTgjU*TpR*<GZEg6Vyz}Q!YSt{;$N0??NiE^e^O{hKm6j5}>NdBN>eDV6M#}+$?gtrP7i-2>aOJZDP'
            'WR`WOvmc>@0jae@76B;|K2{)a7m_?^SWjT~_=m~y@t-DdeyTS6^o$4Yn~{g^UJmW%8z)LHhP=&q4xHCFb0E`?49KF~f;&itAb2RU'
            '(7!INHE9e(L|~sF4Oy?wqr`jyHXPuJ02G>lg^O6NKndK3Gf=d^m3>gLatoSgw2HDE`V!ng{9rIa+sYAPBpb6UIz~Pk5i}MH($&)2'
            '40g8!ivm142_^?{bsjA!x&Z<aI3DoBUk%a;(vrf$)>;LF9dsb-(Pf$!*5(A$RigPj7JImmydg6i>i{8S-L*298rGR-yREtl49RTN'
            'P9;WH=khw1JFq`;?cviXU>J%1Lfyc`6}szrnt`9u-!s9%!=Uo^2d+1wCwKvF7}E|I_Gx^pH0=OdvYk&oL*5NH68l_&BFGf7SD)~B'
            '0K~I}t-%h^6v@!*#WNS8ve0e-%_h*HS!g^ar)8ti!s*rM&ZnK;|L@I&39=Ct<&I0(zRff>@^t{2D61OFLJt^ULThx5c)NWcL%Xur'
            's`EG|R^6Um=R?m$KoWvKc~8*c1^tf1F1-oFdv&`kG5C9-a1JzEXQr36n9)9i)FWF>sx8zz!~?JNZf+#QA-F|py!S|g|H)f0Nzq)G'
            'Tk%d@fad>&!wOA)xNJM9APGNxi^1s8+m9cPLF>(?Z>tah!6igAt#}>L`jF-k`Dh171v0*qJ97xayoRQa4yHHRE)ahKl|+DiY#^{%'
            'Rsxokv?|T6elNqBs4yNdCTP?}R{!tOZ#)AEKj)|rwC<*bgCVK?QI0Sp5{US8TuqxH#;pOgkpvf@kv8}n+glgM;LEDzyQo;BgCy~|'
            'X%$_UXe=&cJa;3NO_n08fd)%bXIF5b1&57fQK-!paf{$E;;lm%OMnKDA{qNJwhD(}!)<AAw}_<{%HklLJqfYF$8i;hbWR;=V#5fC'
            'W_KNFume^+{*^d4lww<>F^`KUyu2noDu_VzX&PyCXH_Pi#_;r+MA%|3^-1Z&3|*4jVlzN#I0#(1gj23@Ynx!YxU6DCEm%v`r=s6?'
            'aQ=kl1q1tA1CAf*Juj5h7^oqP+h@r9!VyS1mswj%2K<}sf;dgCIl?SO7_C-{;tyu5jxlGn>^pppAVQbsveKp$xM{WVFpSn>7Bp6D'
            'gh_2O0Skft-VXam9t=8$e)nxV2}KG*KDm+pP5?~q2Rg2~c^2w^E^!a3qoh>c3+g=Ab~As$mGLLWY12%nw*wDO3mU1_@W~Mj?sA0('
            'TxAjgNnsEl%p0KJi^>wH<QeH$OG9=ZH;Rs}?JH2qPI&s_r60+})&K~H^>Nj}6E+EC81|t{1%hX4R%?gx%vNrr^*;*rIlDmG3h1+x'
            'a>+~Eu7<`A4KG+yvrnA7iOOB_4xZj%pp29XC-sOW_&y`_Jz_;ECYozW6Hpw~M&dEVE%j-O@v!cDc9FfI3DypKsKMrC9>PcTNs{G`'
            '58r!M=MH;p+_HXvo%A6ow8VN37$c&X?h8zmF`+bZ#W=bmYk@UC<1pYw(RySNkt|<OJg1+c+f|^74bu`|&<g1&2~aHZGF!=|-YE*R'
            'U__`)__UbaHk+<gWmRca`$C`5v(<9`_9p@pMwvpL4|^#o9ts8mc%0z+$1&O(%Nt}$$GkVRJO<N~7c{!B$l#&n7@;_T#+cj_;E%o='
            'Z_c;yisu*LCI`+Y2TKemE8pm6zr>Yl8<m66tMc*1eK$)fsM19eL%&@g3&SIa=pHV8+Y{{yjHAM?jSqtV8d_qSGck?tyTxuxaPg>N'
            '&YRVJ7}lHw{HF=YU#}Pe3YzB@EoW0OFrXpDuq4QUKrb!O%W!0ZClEBm$JF6{*LH<<*w1*%j6{rUtDh&pZpa)ni@FYXKtuiQp^_&!'
            'M^<6mfWCApiIk%t?$gsiADYt<LJs8ka(-}dfd2>LYb!X!?@jZH6dvsYqNVff)!H-n`e6?d3(EuS0w4D+yy>`fjle)O>QE8%CL)DL'
            'F0p?7b=v{_;Nf|vLC@aIRTo7O@1Q;rYHs8~aPZ(tcpLHh7YKP@jM$NXBS~UJf{`FCLPLJiL^XbOaPWjL>PVl@mo>kwY~V3aHegc0'
            'f2cq(Twe)<AWF97b2#7W0@U<WpZ_q{_d>@1*SKGw^!yW8e?ND{bRag?aU+>TD=>ZZS3Ms(iSqMjO}u4))~v{5rMB}`GPcoE&%V@0'
            'Av(<CWV??>VsiYK$uH;TJXR-aBxw=>mh|~ymEx4?^Av4G8JjQQczU2AtGVXeeh>tME5bJ)wa5fbtW2{gALPm9GA+@NzeY<0#NP|_'
            '$I{hmC}@L0$jLm9R+RC9Dd}wPVNhknJVU^4WsGz|gp`DoX;(zQf>a<NP7*pW-c_8!oaW|8DX^!0R~-K#fEsBg2DFi(;!G$1pxvIW'
            'gHfw?xtFwvMl2_Nrm%xMq;Lu_TTm_MTN>F{4SE>&3?cNcptU|p(I>p%#?O8{I@f*z20(_!VCM8M&ddkgMSut7`P>VBvE84GJFu76'
            'm}jZOPdO(tNMmiUjAf7Et!0+xxjA<BgHfC_Ym#IwemgD7kYBcFLQV;SHr1glCIfhk>*fSF2AXBap;nP&m%+yk+*X`r&iwMRNmz7-'
            '05Nh=O%qe8Efbp$6(r5<{A<VuMkd~2q7Gw<OR{?AHwN$HmlY(--R8qk=yHX*H;6d^C4tiMqWO$V&;bS5{dt$-@LkNlY^q|@lzFRL'
            '#g(X*;1epYF)S|WJ7^w&zWT1-kb@G~vYzLsHPH~0&jx;9qf4v79?_o8K>u2O+0=`^vN1SxpI#1q?P-o)o3M;3HBk6F_IC*v8q7UH'
            ')(NnEol4gP{HQuM__4kAAgX$A)Gv>xcb4dD&I$NBmD32)6!kNv9OHp%^sBS-R_rlA_YLJQIP3ZXbnG_)xL0T`freHWUk@Ks;7^&<'
            'o=V5e6If0qY@P+8RfO>A5&)$`l_zJ>5HzM}=DVS9MLY`>)lE=65o+#8S!mxhjga@G=SDV<W1msNX-z+TJU+G=P0O{660I0NCUJD5'
            '!18DsabRln>0~BK5)<hjQbf{I*#yR_?ZJ@4N_R58;aBx^99&5~t)}f6rP}Bh8#(%J^zF;VDAK|rFVgLn5>^Z6bc1q~@;ngdq?f_4'
            '0tGuF<5H3lo>k|tXj93hN&e-`HmarI1VeQjr6t=JI?D|jaCYrysOEp3&LuVLP`v&0k1NsErxlDA<=T8SY|*lvM@fC$`n2lwSt`0J'
            '=6Y0!Jzt%-s=LbA(g1Y4_mcc2s{rd3fTIeV%F0oVAf1Q*CQYdE^t8^audSy2Jaids@gT$BQsaHT*4wvaqtPCtn|%c@ldOnVAX7%U'
            'T!AD8=R%R~Z@daE4f-ILumr%wi7}H?yYC<as*1#>3Bi-GhmW;Y*5A6So!@O^iK4u<pLyr@GE`;Ry4JJ#-tBD<?eyFFPS;<%eT~?`'
            'D%tvc{dcXsH2Li5XNb$`Nin1OMir7OdHs&Hmu|nRx`e#%`9gtyRJHzl8a)PwRM8V)xQ&N#mS1fg<&TDpH^&;l`ViHNnuVv6c+8$@'
            'DIhh)_w{#<D%OsVhr$T8p>Q~{QKhP@o}U#=<sNe`*fQnPx~3GzPHDT$3#WQ`t{Y+M{1f3N_ut=33FZjQ8>{^1FXykc(drjIgVrdr'
            '?(-n((~K<;B=vVHhIF#Q^Wb;_b8gIlmO||3fvfCT%Ix_92vk^RtD&CPr@csvt=})0!gI^gJ?ZVFd)3ICD$WULhV|;$D-cJv7!rxz'
            '1j{*X9OSu8RH!m;UYLS)%97DH34kV&nifW6gld8?30j$2Vj529Uf$prY^~&tvgNlrvgn@czL}hyoX&JLuhR4~?wJYnFlreUPK66b'
            '+W;n7RvjCExRtSx*>y=p-r@%k8f$20B9x&)xs%)0tKFf{8){F16Tg#mk%l`;Tf+Mha0)wBU{yuBNonFmDx6uaDd8YzOC1wFbn*mv'
            '-4t+O^B!=trM&ZcOwueue^8a^vb8T7-!xl&$o=DF^v$d9xlmf!HBwC|UHSq2OFHP*m7`&Goq#KgBFZpZMU>rC8Tk!)rS-AA)6Y;y'
            'u?@7RsZ3<N1G9JzzQNXM-Yt`xB)v=Ay8D{6omV!pYhlO)Yl!QY9q{u@SJl+6G-!6x+Na*G{c?hC<LZO%V9*>+UtRFTV5Zd*Lz!m&'
            'Kugx&?eLd%y=@JR*7m-k!KN{ougb@{W4l|;*=gA9tV-R9TH%AkJNnm2>pL^j1JuF-1jeSS>tshbFSpwmD;_|0z1Bf#7(W7*vmtc{'
            '9uLD&{Y0Rm*m3dk8v5rL{pM`%ZZ!g!|0M9P@vs0|b*qFyQ?ZWmOoHN9Ds%ZUKJ+F3{>t|D7p(|^70hPmLqf$ZW=wE-At^nK(c{qy'
            'ixiYjD95j{N=4s1!ocpZq{80cKw%H93x=+^j*=BBWy*~S+#nryr1L@k=&Z4>!YEYgKWa0znfjXaIB$4#UC)Ff0&~7MS+6e1aE$gh'
            'd-~QppcEUrw9sm!S@LLo>qRe(E&_)FdYOBH0+54!b2U&<7Rx~p<teA;^xggq6Ayqio_RaOM(x~CosMSXGbCDc-}i-Noo`;#2Yt83'
            'lx{Joa+a$rt!ynOW@qVSF~M>Mb+0iOoP)Qos>EFi{E><_)pSnG3zebhM+qiVBe8OgRF!Y2f!*Q3BMS<yl`;w)1%k|CRC1IAZ|Vhg'
            'hA_&#ix!U&(_E43`u2zn5OSgO9=H?f+PTr-)WZ><_D-flPD>v)U(FXgcw3Vq_9c-^?IlJy^GEs^Ud-1L3<R&?gU#r!WO5Zq;uTea'
            'rsh>stxb*3dP&zjv|0R!Hb2)*s|W!UQh;e2y%t9&=aUoo`DS`NeLp=v5`UWhbbR#7=@|xk@s5gJ7SY8fGK@B%bgwpYmfvDT3A};_'
            'CKAO_)~J3_uqpo@1b`i`aiFc=@U^*B#F*LY)0V1vxQgWyFawxWFM!^F^PMPqw<TH6Qncg8(fs^JZY&48m_n}BLTp#gVX)x|54K_9'
            '%+XML`oKYF<Lbb&uClTDTi#<G2Xmiwp5b~ud1cx2?<RX}xW}(8Gdj(%tv%eS35xvz21kN8KCN|$<|HSc>Fw&>ZYz5--mPiVD1v&9'
            '`^y>lR>C$;w4PqH-fQnzPCtA8>D>Oz8$PCcM&Cl_H~K)~GtTt%xrLeV7ar^3F@y)Ad9wG+?)y*k9w(D$9~<fgQ{KowwC_0ShGsUi'
            '{q}Kw_Xh|5gT;bG10dNmq=ny`FoyE@`Eus7LOk^|)0V?hlf`p-GUbvqyr_pVuJlz77>D4ZN)*64;_?R5ENp<NPp&}3F@(EyigkD?'
            'B2&i^OlZd53^Nis_$2Ekg%rxzf1^hr{u1d6t{#y9V^gk&?uxG0^A_q9?&4&)exe#koX^gV-pr1F5$DrCf1IAojwZ+O1+CDt>Fnd#'
            '$+^Aff{tM+cg)SL#rU$AXn)EnXK_wpIj&>9Q}+PYLaHCciasVPD!xIAY9DMJ2d(vvZk0jjlTtGV)e8B-Qhh5+?$@e2HR`&cm5k|@'
            '>sgkv@&;`_7v*xbUfgoB=SCUtKACE<&b2Ot5<a5;+Bu!K_rsbZ4C8fbs+^d>ouSM_UA%_22wvo2t+(9dEjEI7TvXDjBGz7%au9fF'
            'jEzX)YkS#Q6)CjWo^f@ijf8RyB3_t?wJJ^}=+)PAQwxLTo<JHSTcrZs&for|PsnqctW*Dz5B#*fTCEFu_PF^hw2*=pmJPY)h3{na'
            '5B!Z8w%C!eh0e&v1jxXPCPQ+Ai>cdk7p{-ny?kg?emW*xGx@sK3WnW8O?$C6#t^H$#<AQiLMb*dJ2TNX%Y5HdbcJ1Q-A~+V)sWq*'
            'ZmX$!-}dyu#ht*roxr@O((`T$TrNi!*fEw_Ri9s*u(^e_X~?&r*6(`JHp$CQr*RVHd!|m~M`Eg;2U{~c_xiF;^VG)4DmQIzyjcH1'
            'OGM3tuRlL@^3bWJs{jg~yx-Wr*_d5{)+>m=?6_xKe=z8@hTG|yL!W=#I5LOQ^%aZ0Pv`Hmvr@l&qblp5g*q`jUUwH44o?ezH(C&Z'
            'jrO7EfeDea7W6bxM)%o29@-;4&Sr1qv^;UL>+Jv&nER8GROm-zkNtq+W5GW@Hr(Uk`j{X4Uws(ufq`)Y5jIQ7@#>u`e6<*0;m=oH'
            'E%b@c$Z9J-#21s3Un(Wf_aufYaIYn)`=E9FdwbSxIU*GO8VI)M2B=(XHY#5Dg%jxN^%%Vf<XYW8<vIUD|Jtc~5Mb!sHRVMD9<&PQ'
            '3;aBGumd#<e_eUzjM`d1)<{e>4ZZ9+R|>gTHe6szIVUSp?Tk_Arkpho!1~*O4UOAmURKFum+lT$aM9$&Cr@}@k8d&T8f_R4THR^!'
            '#j&YD{dxf_gPT4>EePR99ESTmh@o@0f`8?X9rCw8O{Z=!Eb->2>QWlfzpu^zk&88cmpu%~b`W^M+-6Ca*I<OH7Syz$Y?%B;Zoz|j'
            'E!NQzgP}`#o#8do+bE_xMC(Gs2Cm0$)xKT2t{8mc3Wq(V{aB6X0;0@B!w%j$><w+&l5hAA^)e)L<70Z4*s7Bn#J-ob>t*Xcm8MI1'
            'z^Vq1E0lEVCK3$Ye}neo3yXZN?E!bOx`nC+<;o@5*4iEN{%jpi;9vY;0!px57#!$75_4kxKWJ2tR|&mUl&gMsMYZ8vK;;JZ8di-R'
            '@uH<lhAske>SQ`?x$s_PTdiVK=OFJ-(r;dw!7}o~klN+YH*0i+(2Ex~Z8aM+ti=$YOkwUkD^-8|SirAsHm!Nft+BpG)0dx4W8N14'
            '9PV*Id_SzQX`o>*JSlLwf5-fGPs$yCU-wMUz5W<)LFFXaPFvcTxfbNUEvrFKqn3uuCO1EG=R0`H_@@kQefgEmC4NqlSUvUEorm{o'
            'P~-hCu$ZgObSt3D6N@D0nN7+z4{S<!OTPG?eDHtJ7Jt+pf5eJ*Qvk;<Rw>bg>eE_yyESQJ+@AUB`9v+io^%EG(Kx72|1x7#X?&WF'
            'scH81<JphX6S^dYOa@dtcw1GayDMeWV2RGY%I|&r;qopMdrK(4HH)5~K`-Zu!*wlp`SUoT>i1xMrE|Gb1?Il3${Z%f_oveRVHg7B'
            'a#nju8U}!^sg>VF1(+Uui*Q<Kcnf9!Ov7Tgm3eOdFi8)Cf;3jB@M=Q5jg|a8dN`*EP_d;<93!%CQVQDEqg8t|?U^hr7qE<-E6&lf'
            '(@-f1QSY%|9s0NlY%OJf_Rfu2UPV!LE)&WvU^nkn4GqL`vlZ1D)sI(zQtdQ+G3D1Af&GhwQ|!jwb#%R+n=}|oHmkYZ-ht0!r$Mz_'
            'bL>&izB3@ded`g`xb>;ukvAh8HxK@FiF=xv04e=#ht@y&fPoyGru_qLE<+5OERW9Gqb!fkr(Z+S9XOm{z2+x%wk2UuH6r24K><e<'
            'R`2;q9Y}r?fn*B3mgboQ&fg!SaVv44uMSrf^ETAAuRa7XHXC@Bh2O&zD>N&xyrOE;simyi`}-)Z1-ie5N^cf5S5tD8diB(*bsI5j'
            '0R2zU+Kj3%*4l3Of1=jzJQ+b)b&GN%KpO6<@IfrIs(MVMaafA}Jr>j(oD3d7!6a&fu{9fuxl!~7oN_7TO#yx5(9KENgIB!!_J08m'
            '5~w}'
        ),
    },
    'video_analysis': {
        'sha256': 'e851221ad1eddac0b86c2a953e81ea601cfe4d476d9107b44130d64bd11ac067',
        'encoding': 'zlib+b85',
        'data': (
            'c-qaqf0Nrb(jff5J_S~(dPhtoSu=_A<e98;<#omz*E)8}c9Lgr-YE?wQ6pw3Qp+DRmVHxopX~m*y6<)U3m^cJlI$c`o2p%pNTShb'
            'G#cHF?nb}ge;rjtI8P#V7cZl16sBRaE8;>uQtPZ7ZL&?2mTDQ5(V~pAR3+I>yimnD%%i26?|PHd88w{ghBIi!t;X?om(Ep~F4bf*'
            '+htW*&7)aZmDwglShG!7lu^!~jeEV9SsBf<>{d-q)om1Q3l*1z%I;IOsgg1tEdWxy49iH(qja&}g!yfp-l(6StHC1MY{MLX713Rk'
            'he_|(bzE$tTy3&tl&B(p9~DDYETS}$!DKl+K-{t|A`>R+#fu-|i!6`w0var#n;rcd414pcROLES;i9bI%P_gg^0-`Y5XBmsLXAdF'
            '9{?BnsFray+D74RZyPTVW$qW@CD{VN@$icH2C=ZHj>zDcR#^_8ga|!}?;^F1@<<g~Z=RKFl?v4g1)vgw8odKNRS|9?wSeKPI0ZU~'
            'Fw&LUgmGHJ{}Al5+9vS==sE1g=`z|z@PBEU>_*ExUX}R8BHPAM0aNo9sEMSkfKO7Su!6Q(xP<A+)wY@^aj}jH7}l;_L&I;8ELocD'
            'R370+p$wEbrMHR!3R2)6Xb7$N-8PTz;^<y2l6Z^UhXcu@vdU9nv^X7YqD_|X6wH69iZZWYCh}<9dkTZb2^$zX1PXW)niktEEn?(s'
            'pda;xLjv;7n7qaKw*nZt%+xY2ZpXcTzu#Ns*+$J~s|tb7W-8unv%Cak0Xp$)(d)^xJkoy^)qI<0i>N5{lQP~!3~L*f>m;6QKtI7x'
            'et~?9pXoRH_mCRD&r$<^9To^e|3st>5^1&B?o?Q)bldB_dHdw;&u?bGJbCf+#TzwMA9@P@>qqZk9mF`|FzF9fe+l!KWLtbN+a}>6'
            'T4%{J%JCIX!7PvdRmHHr`$L8hZee_P_*?aXm#Wn&Uc}fOCj%d@LYPu|4IeHmV*D9S{SH7p0A@e`^wX=?Z!f-|y?OQX>!%m9U!H$|'
            '@oIMQUvDp7zIp!YrG@Qyv;7JIjW^jH|9i{--beEd{Sn^9{Nk%ShW_l;QyBK^i@yt`LBA-&yu^<-@p73&_!lrY{<EkG5I_B(*Xu2#'
            '6>>?I+(otU%ER;~8hj3-BCGO6G+S0V%c>$;41tlLJq+vyo=jiD95I19{zFc#0c>Jvq8Nu6Az42@2#LQ3Jf@SO2OA%reKiTx=sQ)y'
            '%8V{ousTaX82`CCXVRjqr{ThSTvwYg9j$Vh#w9G%Buj5bv^o`!4wnVU+boT7Z36r(U7{dBKo{@gg4U8=awK_h5u((_g&XJ6WHb*8'
            'SiWI?Q}L2hZ(&(K1=axhOTeDOn#y2}LT5%!6@ZkwzP71-eXX`h1u`g3E07rshbBvbWD1k}a98x0RN-a|T>}1=z~<S6e^ft#O-G$h'
            'G^9TE3<H5qqgz<6@bco-v$3Wa{Tr{GnFl=8fTyzG56;cR(F6hi5zz<koS~eY1m;scbMDvnd}0HfQqu&G;12$(Civ2T`0G?n&K>T_'
            '!?=jlFJV$e7kQrLgT9B%chE!uA2fUY4uL0|Fp2wh*DQv)`=RN$l{836Adj;oYvdY;t@`sgiPKHAi&6*Bx%y!L{7e2I_>}q1qPV+A'
            '4K-U9uRWT7RS_`Sa&}-AfI|ZHx%zrCo>1fH-6GnSYVdZqW%P&EPy&>~@Wr0N_IZI7u(cF1k{_i`{9ZpO!d+YduWIru+9v>{CbU0V'
            'cjBAfRqv~mw_KGi!<$%v)K1lHxJ$!Z<>11T5bv^Fs2s1_$aJVfcRb7RHogsEIAIR`O-`1u!spelx^)@Mngk%YKXt<vJc3rvn@*L;'
            '{pN}^Gnllsl1-x7%tuCz47LGiC}(@2;)c@%*$q)Z6;a*D;q?{?<Lx63vwi}@wohvI9`t7r;|T~`(ENE;53;K-u|BnzG`a!mtdNT~'
            ';d@IaX;Ki+uwdGR?*=gZ;T|Ff3_t{0o>l2`P&Y#`!BBk(|KnFR7k;TVh8f{Li&rz)Hx?1>d&>bRJvYGe1?|7MG4N;?>u7Ncs`6Cb'
            'M4&yEx%m)Ptp*rSPq3}wZfRaqP96+uRH}3)n&*sPE(eRva;OYK_0A8XAOr|;vQ*MU-LFv*UuP*>8Ms|8csD))V<O8yZG_Q-oB}r6'
            '0?&mvq5xc9gT1w>QZgQ3XWjrKswIpF7C4$D+jt9V{jWbfe{n%jb%%g2?A$+PNkUdloEIg*Lt6l~MSL2rz$Cm6^JPIQ!4e*WeQ*oN'
            '!VdB58SL#hD%zl77^PWtvj*ioyh$@qNEQWb`UDRQ46xA%jWK8|Xyt%C^7Gqgqi>)e(s{|g3KLKW(C$Fhgsm{d3>AhS|M3ff+Q7__'
            '$$>j=gw_<8lrWa?21ZC6e;e~~LiH<HQQ3V#{nB{AwsrIfqyx$TV=s(<0wSk_dWLcGigF32^3?w7`A--0Jj(ON^IBR&@5<@hyaKfZ'
            '4AKm#Iqg^FYV-|oR)ou}nD%A(ZJQ~x0|IatYeNe0@<SvwQZOjQX>75@0L>U2$7FmKG@8$IG>7dcBCD*?y=4F}e`t#|Q2<@7a31bm'
            'nhj;9?&xW@0hJkcn4kE8Swo94e~i;U^oAVRD#71C%e|mKQFXBeAVJvz#}5KjRBPd2Mu8sy%Yz(4ss-J`zmvjW2E9+CB0)s{9R&M('
            '3fxWKGoD1;l14GGf6SI~Zlp0kL3?-#S~E&YG=>|d3CfE&-BzV&DED!>2K*J<q`c73fsmqi17tfoBX%fRL^eHvM|#CyUpM*&X>|`~'
            '28)xT+QOI7a;#oKxA{G8O4xrI-A5vtzycaYn{BzHkHO>y(}EELVduez(E@mw<swR=MTv0Vy3lg;pTGy`9l{w$J8Gc@8!`M5Cq1@+'
            'okZHQLRq+l;gcK6+QR@H9G03#t1M@Y67ATA{n8-Bxy3U=t|`sPpz*0=gP?-1Ta3YAMtlcB+xGqp-n`7pXSkDUZT}y(@;R@P3Wf@D'
            'AGCc9^P%r_XS#yTxf}%W&kTKe#~3|+Z&}gNjYtIjiup+E5OLm07kF+qYO)^9=SURz$EL}V^tc7nG|MZOE;V38ne__o2Q2`#%U&%H'
            'fe+BtMSlQ$+BgBf5mJt9g74}nG;fgY=oLA7-i?#&9+eJ+c+ij20>qs+urKMGE1P)rfGmaCEH}`l(cSY~M%Dp78XUwU2o4h8VV+gr'
            'CNvpJ%g8mA>RB19@6jN+g@p<Wwt63?pf`j$PL@Xcp>M#;qzRLuvRMyzsBWX(6ja;!G9+@HtAR%~k`1#IKBX5Z8uX07AbX#wHz)|B'
            'O@>>~=swGrY8_QMOBT@ZQJ#T*cN3#;Iki0O+PRDy<+yk8<EytoxY@6-UVm?S<BB!tUPylW13t&U=TVu5yZSG7E8;$)|7Bkw=x*g&'
            '{xH5oQ0>TL7~ibPLkFfgkOu!M^NK$w2Q&T=&NJE;VPv{_lHHLzZw=(ro<}e@$@Pd-q3+NvM1RebEMrvi@CJcEFY6rkU=1!_hbi}7'
            'WHin!sW$vC=>GIPybBBb<0gKekmC_xFCzLF%@qx)jMhnfONiWt)Kl~u=m~4`kCq`07JZepae|_jhl_GX?gn6n866P}Rx8wQXY)8c'
            'M-Mj`A>q9xhKJ0EGuRE~H#Mt&@ADuk<NP^S;v2NRBj%yjm&hmS*)y=6Jgq=czBI6gHP|m4XN$ruJ6m}p01;AVDwK^iN4xPoEbt%3'
            'XJ3Ey=*g3(<d2elP-ui_2d7`({_tLHz)n%WKKX^U?h+V)K4*m=q#k^p7K=P49qbVwx|r=yy+$ceC@_O?J&^<;)dHGTDe5@Oo~X5`'
            '&VpXSlc9va#(FDtr$BEF?o*aM?&Rn%-KRkd%<(qN@$hVOivpW}6k4xG&+9C^rJrrxd_?1(aV&$n>ue-79|G%+XrYhRfa7c)rYSG1'
            'QIg#xc#c4fj@0be?Dv^^jtbm}&d|uM3Mvoq6yOqr5W>kqf3xgQuqy-h=KJgGQBho9TfpwK3XE1ZzIgH{lk>?$Ra<oO4iH@QE<_bR'
            '5T~v>`${WBa_nP)LxD)9r<(6dun&JlTWv%Z6CNyrY(<q-o0=d-!Il(Fx2R9oEP%#AMJG3~AU0dJVrlGTS!ZGfM~jGzT#)*!93{B{'
            '(}z*_aXIA!v!ntf^a|~65<p+vMCrDqKS!~A>#nb0Xu&Jx6K7r8A8f)c!kV1lolgj;c@UmY{#)Pv^ln9>5R}nqN(*KX-#<DFK0iB`'
            'Uu*j1v{{_fxlIz!-@)>x4uAj!OIuXhS7A}|<M`cdwV}Uv)!kQTldT7cWM~*J=<Dz<i<dL@3xV;$1lfeQ(d>PcXFQPXuBIAxBj-Vu'
            '^C<V%?9ft6z(~%>xum6`SaR}rK}rkCMe^6F*~=<Pev;RsNQtT6?xK-JH<XTyBI%i&>Mt$+R$GFB0;x5}cpfX*y9#JqC0chRcr4N`'
            'M_*20*<2YVlI6x9z4U9&^vR1{ls4GRY08?x>DPcL&ooW=Dg7->uw86s#p%~{q=&tD;?$nReKTMN|A$QhI=69yS;V(t9v_?mQUmmk'
            'ML`j$05Dp-w^Q5#>CB`IpcY)fhKW>PMtAX|e!SRL{jR+q<03#H&dS}kZUDP5x`cR?vp8L4=U^RW3A}q20z-EJb9Dow6TCIUC{#TH'
            '-d$G|;W4`BU<}xXsm}tWHY}MLdaPg`qZ>LK<T<#f;D#t%tn~|Aa34;LR6H4~lL&Ms`4eH_zv11@3^)^ToS*OjPSR`!y+(`mjGw9#'
            'RF8|3b9L#!tHIF+@DF|;;-N^nJ9)TzaII<Df_Pvv8JIrAm!Oj$aqf7Tarnv9c{8ZzR!`=X{~0<HHno2+bu_JJXwVH6&K>%|5#G3F'
            'r-38r>N$Z`i2n5|youx}dLD%vLS*_F)XQ**wje-ejln>{7obbPh6V46>4!dSv(O6cAN!DPQ}oZ(<H_V<sE~{^vl|v3I6N4RsY!-Q'
            '6Gom@1u?^v`Ok61T8)E0N;+#YqU^ge8l>U3)TML1{@Xl^wg9nVkHalS)C}aeI9y?nxCD(o8{LfY8OR9GT*1f|JZv1uedB%S3}cp~'
            '*V!M64Z?Er;>@_;>o#ON;?XvCMk@uK-j`RNa|f`&mt#JF5uc`#_G3J`Z}oG^#Dg+cMa~O@ktW!OM!UXQtiWM>GQ%feH6na$(`+eA'
            'yR99(>S~4Wp$UB^eGu7OKD9(eTPN0Q3IP~TpwF*7Ko1S>*+Y;{V&JrO(V~M6*rc5dedIRy+GjlPD@*FOfVSSYjJog2=a%;F2W=pT'
            'O=YHtX7&3A>m<A0=zOQ2vjs5(<<VP<%sy?P1AUf<;D**)3I@SBhJVa*hM&#g9@A#l#ykxcV|sLS>wp*%f3@@3LvQb>MJ3^|#Tmsg'
            '76aS&po1oBwF4G2v}RX+0;c6<OXSg&eE?agpLTB$^7jNwg*PEa{HVH{vDM9qbZ~;>ISH5H7WMrT-ix&!5mL|SX&6{e31Rktk>H4^'
            'V(jeSG(zr7Z{CF3k6{W_g{>72=Ge4qZ8SDPq`LVN!SH)|F~ra&9Dshn3W07nTO=rhKNvcm``@F@BQW8m5B^~c95K+)46i%j8-jR$'
            '{7n(ADROlg-2#LP)K?AdFZqKS%*Eq*t_0<w()b11s$g?uYEf@c8v~?*$j*h7oNbuc@V!m@LCYuOaJigmYzFoq2W=3?EqrW$MocsH'
            '+5kOg_3J^P=WC^A68}agZfF(+dtx+XiSeq{4H&pU2^!~hEoX-fP+9<JknB4}fU@pgNAu;t^rtrk^8iJ@s2UtOLA!Qp5$8(ef!07p'
            'hm6A8!m(1yY&!#nz;mgoI-LyB0W(_y0921G$0EKRKi=b^TmvBBt{ZfLGqlI2XCzvd@uD<R<ORhXTUG|BUj_MoDB_Pig@MEqkc6Oy'
            'A5KUY1;szzj?-nB=iv^lfqc_=L?dJaf&A`l72t=U)sDyGs|Pt8qLYdFPVvbdtQj3ky+r3xnyK<WQv{8ok<l0e0{hf4GD&0znDKd5'
            'gglXePl0Pg9*!~IdeBhli+oBJv*^Z~!fLUKLJE?Gr-R)T;MG)~)_(?<6IKL?NOp77_(3txNR=M|p+XLig?0mg8F2SS-UXP)fmFb?'
            '8^(#JrN+y5$aVzy25CxAFVD|Mr&sk#p!O6hJ+)R<aS5%jobxQOjtAkq7}O0$3iS8QWna%_|0=-PM}Ha^8Tb;hvC;KZU;C9&&T8O#'
            '2RHq5C{aHr%g9!4;0dEYR9^=V&LEJs4hk4d#>=eN9zo$?MO3s7m2gz&O)A)uf=7TT(!Vvr8~mGi7=GEk=yMDVI>Q64JKRt-vG@nf'
            '6;5kG3rjwLbwXw>(ST7PZ|gd~S(~}#PsbS6h5Y3{guVi^)lst%f>3`|r_t#2tfAcj3g~Dch7}}HRz7ydFK-wW#`O(q$bmVr+2T13'
            'Cl1W!IF(@@N5!CdtRl*X{=N55MrC(Vo&YC-sXH^q8f#N1_pS~!XMtU#BTZTN8k%STW1}C36p&dqLQoT<v~bUeV5YtvPYxj_YCCM('
            'JT$BIWc=mh1I<^kl;`2%whO?2Jnjf}AJTSFYn>M~Z=fbpUe6L^KbF8RI;ey)SwL0aY&hE`CHtIoy||Qsk~cIIK5&xh;&T9j1xi+&'
            'E|LoN06tgXs1G)eH{p(tvo_F_z)@BVjtyhnaL)==8ihHg8d2hqOeP3xj>n=C2N(iELLdY{OHhO{R=;8gCHc+B`=J5WpkVyyJ2Z4v'
            'Kv(>>DxxJT)Rf?3LgDDpI9U>)s3=DolA1^B5ZyI76KPSEr4AiLl~13vfH++wSwZ<&>#WqLfnlmL6KHdjWlPM)QIag?_}J{AU>%;x'
            'qM*UkA`tW$_sj%_>UJnf#HO@PR=S!at+}xa(T1b#3kOQi;(Kc0MrU}#ki~v!S@W<rjr7~b^09PA&sEhXLYxp>en#ek)mvbD$}G3J'
            'bPYrF@Q2aY;jrEyw08l9@Wr1!1l?EH0Av-<>;U0Qe=a|FCQ9~0-LhtZOPej~R!E#HnekfqQgk<kr+lD^uBr9;pveX`H|fRYuoIEU'
            'EIXr{*S)br^g*9Ci4)bq=F+tHfmxpcdCi-@PF?D0O)Y_)5a;IiLH}-QQ~0VC@tqL9Wk3PDd7A?zFRu=JZy>u97HAtqz)1ux!y}j`'
            'GT%gGJFGQo!TzsWjp^7!Erl{n;+r&D8a{0R>v^FYC14ZX;CTAxTZb!l{gxa8zZ=7V{$LpIww*T!_MqZ7==bXEjK)-qoHh0d#Nwsd'
            'gj^fcC><>HQX1mXRFb9d$T}P4FODOVk3|QOK2zUED;)0~#=PDE>xHEM5x$RD*f44&DNzL=^?|if+I3m45ooNDjMej!L-ILOm$Zh`'
            's3f8`;2y1Y3$2du9^Bv<poa5o=xw$^uZ4ubX4aFTv%v;#X#Ldkxc0Xh*wN?>?eq9YLjKV*&N-txOh!bY(cP(ieC*=n7-YWmx1HU6'
            '8?p9Qgul~9_5o)JIc=Rc`y2b+TF?E4&f+6?wEe*P(&j(-&i=_?Jv7r`yeY1eV4P{h$9m$#HX;_BS$WliIuxtjsL_TdS(y86m@Ye^'
            ')3ot&WzA#9Q`K#M@Dg(EjV~oTD91nu*QYhvV+s8(GwPEKsKchyUW7#zn-Nj%c4vFq#p#R;hZ&3(PsQ6EfL}fyPwK-0*G_wiv0BF!'
            'Lol@$)F8%s1(cx75mtQcPhds_hM|n^&Q6eN*I;ni5wKR>27MG^*Mbm#G1Y?z#_DOdna3&U1|pZ2IHl!I7<dGxQ2mHJbg=BATrW5$'
            'K-YX%FBIHmRyZCE0zl!OMDJ*1OV}K8{MrtlypJ$^77WhG=yWn+Bo)RX6iG}0X3WMCN_mq<(K3d8gYyy!j+|qcAFc}N13JcR2TK{('
            'V-pD=P<0x3^9;r=7!qSg0~p`!(A`mxDIST?8``1v&9^)-j8q8=+=**)ai9b!{weP6@Da=q!TLUf8G8BZt-3+#W33Rkt^^k7h$8cL'
            'CT}Sew-^Mj6-t~mlTp{MOD!X`kykrLS+mEj&lgXi`qZLD&gs6ue<r84Xfe($^jJClkRQ+rMQTL?eQlQjR2wJ8#ZO*NP|clI=dQjI'
            'Ewrla8q0uZje5}HRjT!#Hbc&_BIStNj!|gYEc|!_cy8I4>N{&pxK!x1Wk;*O>H!%*S5B6k7lCofhuv2;R;?q^sPO0Jl-CKa!13nM'
            'cm)(ge{|lzd@}m);pqMU{-2}&^S@56zUU7L1rx*XMrPn?Zj)m8>e7icxN@@>_T%07R!4f^89Uk3e2hNCV2{>V5BkXNziE8CROTa!'
            'O<3Gw8Z{p*fc;lHVATJoADhVveBS264D^-P;I0}a6g1<)s?%&dQr2Pc>DVfw;H>D6jjyo0M@Pn2)Q+NNe1&0qbiCtvy!*awBH_!n'
            '$1O1tpj}pUrc81KXV>pv6b#vloG;ZQhq!)2QfcaTMYXYH#b41USkEc38$N0AU>xX&=bZP@zGxA62DQcX3(UBQ1~s1y)qhTcz!RK!'
            'm}I;I{(SL%?2SV&3^yCbUNVM4$Zr;L)oJ$x&+5J19>^vNxjC<TM&FVG9hi$^Ff)n)z$8qHs1$74@x$@}DBhikKBFxQ*N782)6>DD'
            'yv5!|J69^XvorL$V%Xp>+c@z^KkTB_ux+q*h_97H0yZQdd2WuRF7GelH;eRpoH%FYU$K7;zQ9-RI3Fwxt8i^ok0n75pA4G|f^_AU'
            '1vVL<KHftH`JkI|;PqtKKL0R@k8Q(_&NI_x^loTQDN<OH)rORwfkO++%i`-)hN`wqtBFn&zNCJxkhwq=9jdc~q^%|VW#8`+HsNUl'
            'jYcbaOrJ7hv~+pGc-B8}z^J)FFq7{C#JG<1aLgZADJ#%4bNtYvQj9iEA?5JAa|Uc-sVdUgagdK>X|7!7zxmWY5U?C?YjiCa+8^Wf'
            'gNK!!0PU|@`tZ+8o#C+@(VY+RnDsv`a*3d2vu}W>%t7M}cEE?vwtn`797tPK;c6;94h1ou1UFouuK!2Lt>kKVSlLOZv`@Z;VbGEw'
            'ONUdStSK#IIbn<es%^cNLt0@9g9X?i>#T^<0Jgi@kLi5uYEyI)IWGn-)eO9#wh{CzhTklo@SX9T?lZlEhT_Knk$e`BtxQ!*@2g4T'
            '#vOQWBF8zTJ*?Wd1;p$;IBy0AYdUybY(1~nRd?(0Hro!`wI|YVn+~TYu!e;*>~iKzJDxXTHXS<C=A&k_Hh0dn?V;P$oqCIeXpocS'
            'L@mH#0<=a?yBam@cT4ulsjU44sk?_@)l#V)k`7Dmg5x_4EI?kdvoBP`wRr$&6Sq!~;uP%y=&uL@=|Ak8D{^~Kla{Dp)<&iu>INtN'
            'rgw66d1C7<Csz-kK(v4&4n7FVQFOXt5mz4u=BP)V&_qi*2+q@{0A7F3g0PE#iV;|gVLuwnStW;pv1HS;#VdQsJ3LN}oNqhk+JfTx'
            '&ovCJud*+a#>v=XxBF}rN6FGeaSI>!J5s$vN_0~HaW>gHjx=&M*1;R>o3yb~n+gl2N<0Xqb6K82inGHyM4TmdqFdsyPt*VdzV#!@'
            'FgyuN<*9lM*Vo3OdB2V^L<e3aaf&&_PQjPBP+7`RIWQApm6TG}RK_mFhPjeVq>Ec}<rP(>i#7W8fMnTf1vXY8<z>SqmRFc?g-brn'
            '&I(Fv!hj&kWybS0iY~$un7Xbql5eb@RY@|Es+8u{8Vk9yV-{m)lE`gEdC74}g^wf`IS=oxIyp)NSYOu{=7YI;ktLWFZOYYrjzSCw'
            'V-E;Jfph`RM)jDQ=(@n;z)){!dJT>ZG`IQiL;b^xGM5f?0ii%Dwy+ciMUO-6P@SJ*{f-*WgS9IR{a0GH|8a@FhUs;VzxAl`&9Q0y'
            'Wmbks%QkJHJtiN|U^3*7hrp_FIqly9r}T%qNZEANpFH|NA3VS+h~FpCMpyqrjc`eV!K`A-NVIyDG7r;JYliJ5tD_f$43B{3b?ey&'
            '{`c|(b&r#)^U2xrfvWu0K{5P6--b!sp5t9KU0AVU+)(%c4IFu5nB>Z?t=4d?{@!$FFLS@qc`$5bM0<JqZmzyR#KwnqBL3*?9OSZO'
            'NAJnC2YHu=oNVtw^D^tZ(&io1Jp8YEcG0lTjCViNx+B|q+NkQ($r=ru>cfo0w^@83JdIbnYMsu{KBfF@tFprr+_fv*QD~{*5!VWf'
            'cCA5Fn&TvjeS}YVKZ7M?WHzTVhfb*oEOgEeOjw+}@Qbl!GFuCqm*)Pp{Q*CEXj|aD-e>2|E>mC$v9iJ#%`uEjmonyk1i{n3qRy<e'
            '^g{1AurVPx*yHCB*Y1$wi&!N(qk{6FD%`KJkh91Vs@9BU8ab?0R9!yXB!2<|)KD<C)^y7?-eA*RuBrrS4hU{g0__UWM$m|jLR3GX'
            'RZ3Xd4LPKxjB_mVk!;4jXFtDqF?;j$$xEq5Jp2CV*H6f~`{v>)9^rq3p<@8%4bo?m(cXSTyEt~hRWVr=xDTmJq*lndet)$@uZ&Q0'
            'iKmaSXCOSTNo~l3U(fMVp--0LoJfcd(uByw2iCkJ-=^7p+A@G2e}3`y`A;vNU%UqN&#d=9KKU=J3i_$@>eY`IFW<hI{dDnKKEVTd'
            'MDthjg9(|{O_I%Fusg*UD$u<UY&!^QTsRN5h#S_COeHd$Lxgu<r|)yfb*&4*)_|8eFs;VwO#~{*3_(RwH&^dB*Y>FY{^Hf&p1hb{'
            'yu5h*_kYfQy?FljAKp?#!dG53B`PU!WF;l9a?+u)M;F`$k?7YSUcI;wo#aGvkx&z~sucb7^hY|2L#Y63N_5wsaCx0l9JE^b3p0(P'
            'CDt#c(`k>1dC_o3yk6@_a+wuc6V9Tfye)EFfu3IF1@=o6Xsv}L=h}8XBwY*XUZTr&4Ruyhm0TLaSOJZraO0|t$Jaml<Pt`0+xj0@'
            '!Njkv^Z&ep$)8<@=8q|VB1Q5FEcyL0)lZP^XSR0&5u{Rtc7h@0p!tb}lV)Pz(=0^q?du=kXnBQO#I<RDt-?i~6?jF7URu9C`DMs<'
            '9L6kTkzu`3zXvmNPFgUmu%(VD0Sif|#(_1SNx5Q7^c&#{69IsN;uhtBR$Uowip_v!3ERC|SLrgx<&O8UL^mPUp_033wjv{NX|qTO'
            'Wvxde&T|~WxJI1F2?L><JR+>C!syU2FfWn+V8d2!P66Fcs*pmT`7%K!%5-H=#VLE9<;Pix_u*YsNA+_2tEOvkuqeFww7Ud61p+v6'
            'f3@SEW;5=5O)?7+98)FbMtWpFpZBTKIg2Q}m^R7-%19LHbrY4TergAG#7h!i^;b9%^wg<~TSm@DH0<0ho{Q(Z1Pzwe*}>|i+7brF'
            '8qGlogI#4PRVY*SfdPH4zInixZqZai=9p_CZlVRAA--2xo?>qs5byDO`|1bcqZwy{%3;?j)}1K_)XC=!ey~m}u$XwYp~Kn9!79_}'
            '$uRW8Rmpu0Yy=v?DJTG;;rS8urFr#8q!cDE&8yWaA}O!)65$05?DXqk47LwEz(aZ(-j`>1K=tL>e|-IoRbAhdX%|NNPMtOpk!=$1'
            '5?}<zR0j3YBhwB~PW9i-Qv-vw{B?QG5U+qn#TxYB!RT~Y!{VTV5VlD$)*E3|HU>>otDSDNHno$~L@4vTK9bo-Rv>4++ofK*y0wXX'
            'DE^h#5fJpdf^A+?C<p=%km>M?sWS*Ci?!Z{bvmn+fGPpUcK)fgZ*#t@^|{8pvxVVz6VOqw0P8b5yu}L*;vDADDV~N}ZGni|;pRtx'
            ')LFMcHrX==vS*&tZa7qj(oyTGT|WX4AS~^gvSX662v3c}P__n&Z5--aCae%vDVryHtEeNoSzke<IBbk(BN)0*k>r8xUM%cVLI}qC'
            'iB2&3#t(v&xi@a2M!=aLx#h&2xiMN!IcF?~Hb7|OFx?G;pu0<_MuX;Qj9H^-wZLdQ<f)z+gGnr&!k@tQo2QP;+Z7|Kr6E>!+h+$K'
            '2bkOvI)whKu2AluVo*Hz@-{Z3#jS9Ug*0p5!qC{xlr-CjJr-5$8CatQC8(fTqd^v1(%bOl8x$qc?6|OJvN0Bm$%kc;S*sH}&5IzF'
            'v5HxVOvV*sNh=bEGanh4oLPYN`hv?&D}7xH7oTV2AC!!1o@OK#=doN5B?*W70c1CLKz~cO9p#vpOZJ<qc74RHM!MF|r(O|Mh8tbw'
            'y2Ju+c;yP+614w%Ae$jPt-yK{I3IiW{K(GHWZ9sx0<1z#E~G=Fn`g;QqzBQeeg00_1-7sV;#%t1b{Q2S*)*S70_{6B>9wV|EUk#g'
            'a?9m5P{pVNa70wqI9SEglZ=9AkkIyw4DG4%c?EP5kHAz)>eP-x+J<S%B}R|9!=5WIT1w1OtQeYG94=3-g7zzp5Ox0ibLw{Q^*%n{'
            'L9Z#k_SKi8j{4_<-xun1cuWntwu^(B?OJsJYhvmcljgYg0wy(pRhRY)1pB@;!mPR~jqy6snHFpP&Qf$bJB+C(`R$acAWPm~jugXX'
            '1za~0C;aQU*cQ-tM0vw^vJ>o!o?w0sk<PtB0-6gcrTEK1(xXm389zR6s<XJz+Z3>&RZ6L$cuPx53N$ruKPKJbH-T+MnN!?}%>vxV'
            'Lu2IMO}h!()cA$KQ<-Q-&<KWp3JT4yi*9FDoHc->rlA9hHSG4epjGJ$g@-E%Yo;JLJD}olnBk!SLxWLwjG?I=GR13h2ff3wZ?Got'
            '!r*9W$8eIDNS?aiwgt*HA^^E5h2|0&+JpZzuIIhI<vSU~;RqTXCpn$Es$>fzS~Ik_mQf>uRhK<8Fukmf10udN>!aCSOESkEJI{-B'
            'H6^JIK(tuA39aTfDj?;Lq($nx;rd=tNH1bs7s9GT0c8HfHEpcW=E2Rt{pklSIhIbHklSTRybjY_|A?u1zV!U%o9EwOG|!hLV5pIu'
            'e2t=K1p8N=KEMZ+CakPZ2c@%Jv?QGUOwP*qer+9>6RQ6_<DxjB`d1Z4=1h?559ycs@CFT)lcITkNRk%kx9);&)TaDXx#FD8LpdE@'
            'N&)c|3C6})g?H$Xrl=^gJ_P)lRw|duo8w7sQF6F<if4XOy(^FP#Z@%JY)H{v3SdKFRTv_^nHCs?K<6w}K14D{RQgDhC-3>2+PPp^'
            'ZG3P{w%ceVb74Cska9hJ<(oRikGRC&f)eiN&Oy2}4886IQRfV25+#+=Jq&b)Y>u2tGz!P+*O)F**Pc4hK&%Z}8J|>5Qeh&>JhK@e'
            'oa&Pd`t(}PtV!#Kb+nc<Yal%$6Q>xr9eKf4+VkXYNp*aVx4?Q`zP9##Q3paagJ&D>>ezf<_vG$Dtf*QVVa0!uq!6S*-pj*U)zM{z'
            '$YKX82{lIoMyU{>1@I4YdvPaDK^H9VHnqNM`OG8J(CAyKdLM5+nKU%m2Isw29Zh?k1Z6NXz2b6qNJ-1QHzSu?8u%$J63(yZfZuX*'
            'N_Sj~-7At-#$XEB$p|Pj$8%gC!r#qetY5k)9Wk(m1~&lv08re2z#o{C1)n4hcwZa<WgvAAL-!$^@)2_*wZoQCPRAUgE&3eSp~JWo'
            '^Atz1x8x?lBehk*<l~jjW-xuNd-?*Qk<;RR*r}E71>`EHdfdNDJvG-b9#l~Em!zViZ-kVSpi@CbWVy6KTjZpM))qEtSWV|);2*XE'
            'Yep`2Y*r};wg~ZbaM|Gymz<C3IU4)(L-iUz)#of1V9Z*sD@j67A4Ydyj$mB#tbZh+w#!Y-%RSeb98)N>vxfQ6<zfD0M>}Q{iNA$$'
            '`5(L+O*=e1Ua-Fj!IgObW>k|CGw}@cSu2RYutFp*UcP<)`og}ZYe=;ku&6ObpKJ9R`IEU$bLKVO#zmD<G70>PZ>p>+bSmoL?5jzE'
            'F$xllPYz%0QMckLNFR*pc1S6Eg{mx!X$#hf!C7n7Y8Bnf^<Q;}lmzbCdha$vx&qh`3w8RTgO(;LvGAf>&lw;Y1aqpr24B4+@h7@Y'
            'xB{uLL>=;6-}VCfJYVZuSb_YgbZfPd0(rp-h_TQnjF6OP%~rbpXr|R(jF>6pnr^<aSk5~{<(4QgQMd=9LzC%eoqY^xT3Ah^k}(uZ'
            'GL)ltl!02N7tS%xJ?w33yhA(z^5>$PWw&FFYdfRMy{W;)e?5Qm_W8@dBlMBXIbMlPsTrWUQ~rnq?V;C)Kxv9`JO?J**Vo5L%AVA='
            '=X5-~NRX}!1h9pRN;Fsv=Eyu?Kh=gLMpu?PjuF^Gj0W%A)0_<;;zMIg=xcX3VH&T{b^`-WZa;E(682jT0OsINWA)i}UEgh``WuY+'
            'R?G#;4&&nD*OolE_Nwj}^U%&^`mM-PT5%MuNSQWxD%G+`nd%gz3(M=;WIO^bQMC50J*R&y(L^#(sxL&QIEK|MF!I1NV*Y@@HEZ|M'
            'p@Y_=#y7`C3;vrPtwkmj%3LNmb8h*yj%`>qagNsJ)$%4ndgFVGXK)B;IT<WlV}<!-<!spA_vdO)0CJ$~-hKv)?^FL1@n&bX6c`;p'
            '5GGvkb*gVGLr;n|M53nFndG6nnd<Oh9gu4C*dvGK$M@k~Uk6Z<A7E;gNw)$;D51C+KZ1|_&Dx0>IYkGx?=}Q(d(hhgg=@lIvaxMR'
            '=dOvpVb4u<&>Oy`9Qb(|^^m(mG(BkCST3^|*a4zLYfTK*t2guq1?0ik=T5GLM4Q};KL$-#C&U=z2S5Zx`5o=2W=s7Rck|z>!oHIe'
            'r4oj5MCzuBMVg|o4YU%x3|5v0o`IRz0t_Yq*$FxZW@hLey1p~i$om4Nbdl9r8{Twb!Wl-7!2pBtY=$>B&t?yF_xQrfN)=Yef^>sb'
            'pA!1Qd6*`#hzqp~w-QQ=nf1v{gsKd1J_z>kCdMnL7qAuvAMEZQbe7yICZC>7VG3rzxN{Emq`P$Svib!u;bML6gr)xYH+I+ocIbTs'
            'Mdm(b1;Nc`SsH3eLd*gdz|L~U(&UZ1Z%gqQX<PvT(%Zo%E-(Yq72Xz)2_t#eGk_fF_1pvK&bS+&-CN^srrxt*Z;uHuZFx3y4m_rg'
            'yl&6sqq{X2$3Un3(CoLFI94K%-R=P|=GiF;e~1=wvNXXCzh6?HIZ++7KYNl?@aTIV9v#x3CHuS{R(+iCE`(*k0hV-ip#OosCd-wg'
            'jo0=qTK1=cJI_(kM8`{`)HzkQGeBt3<_kFHHbHko6nwPg>#}-R{lC%F>+CR>H)8B7qOG$f<z4^`KGe}Xm)2#OR~RPp^(1&0gA}@r'
            'Z^OU}7N#l*dULJ$1M@-U=LZdu&YD`2LESC-BcPZ)n{AzvN`K>I$uNR$DRl5c_U3+S#yYexK478SqQkG%BM%*w;A+n<t!o+9_ov$R'
            'F=ui!Xyc06X04OfJ7=d&zmB=8)b-S~lwz^3Fm3bPI9Jb#7}z^b3=y^NAZ!Ltbch55-S{?MwF-#NODnr2ibrk^4m0-w2FF73gwI~6'
            '&UJro564Io;|jyA(Y+(h;%Dt9o4Q`J7BJq802ptA(s1V9C{|qJg-%ywL~-Y2_Fa0OWMV|&+x_>B1<eR`9llpXxfFf<YUn(qWAonL'
            'YCkOzivGe3j#}x;3-vYy3MKYWpJ25esF3o~sI1*g5U3abo|@V%MO&K#Xq8K3ZBF+kk~XVM)jsjYF{SSYH)wI^<PrbS9`vIFI5e94'
            'yQ4z58_v-(u=ZAd97nrQZnS;y&LKp78B<B9<-n=k;A~LiYKuxKMtjkW*;PAvsi1jTE3fu@4Rs$UI<*AXVO~gE=PKbLUt$Z^gG3TO'
            'G}P@+G3i*@e3fEyxv}L$L*m;BG7PZDvwIcZ^EF9!=5<LvG3{A8ifWTqT1|$nWY~Lyl|_*K*4%?1X=>YatbhB;ic!F#O+;CeG^WcF'
            '>GFh9B+YX2%{Wu;m#sD`#nP&ZwC<9-_U%o89W=g<F~S^U%(Iq6kJjgLH1J?w@q>>thCG9)Rrlp7kqM}cB8hOU4Uc^(uU?tgrkf9h'
            'u3E>7d7Rr7(a|J}C4Uc%dQbO8mAegBa&s-?;199i$lu-KXcG@Qy4jupnn@_9FIcoHLB3T*oQKIr`P<wwj_u)MhTy~63~%#7nyXE1'
            'q~*M;;2J-Ep`5xHhPqASd8{f>IzewbegFq6b~TLxyJk-BxT8As(GEQO*(#o61{oYz=!*wnm-?ET@=QI+krF@F6qqqimbQK|(=Tf#'
            'cgg_Sa#T+atTn6^gJ(Ug>yJJ(jfV$o!Pdp(r~b_vQ@T(?*Jho5fAQ?e&oACeAjcca`{K}j+NkcJwF)B{r|$WwTh4FF_aWgk*Pp}1'
            'Wjz;=6RT<am56PpUf8MOW>P1bobAPK!aOdk1%2%^wEv}v?X+r1jPURjC9iuo4p%$#H9#j=b?+#-thkvrX!K>XE_FSN=-+O>*JqY3'
            'L*u~UeZ+a-KLdw@F%T9n^&qXWlMG%H_oS<Cst>!o&+XU?I*D*K)P9T8@Hm?G`FM7QXhI9W20fi@){(oQ)z40Se~+c5*gjzIBR+P}'
            '+7Nnm=!#hux5h{c<a)Jc@>BD?Zqk?+VfBXl)4%q%HQURfisK<QD!BY^d{@lm{85<7=nBmb@mRrPEh+eZ>RkQnswq?j7v3;6e8n<o'
            '>7>S2K-kl5x}MD`oR)V)LELPF4pa_XowYA@K9LzwP_B0smTri8C?YlREBJe6<(^1rxwx7_SZQc<5H4zbzXdLzuCq2}N(f0{iB7~)'
            '&rq%9Xf>`{L4n0qtgWvbxv|BqWs2O8VTu%|%0@n@5BtLfAHH?siK!41D!YJ`UBwctoxU&W>)=5Op=b%M<Y;5O4K>^oZ`=yFEBv^v'
            '8(Ol!D&rcG%~vL}qKiRvaof-NiWLB;D@9VBPivkimV>M{iN?2ruf1Umu(FY7?+4#<V**p3Md`2_&>kYJL=>({%1r0B;|AYZRV@iy'
            '6UgD9?P>`emyb+^2;{(kk?#)gQy`2~Jq4;a$@$>z7Hxh=+A-Q*bhkA=M6aoSez>LMC+vV?e`8xgZMa$ijHX*)p|Zl6lq$tvz)t$D'
            '`t`}{m(O4R{oKCW3Ij!=&9>aBL7LU~XO%4R`gZuE2H)%T773m<Il&<&bnRs}%C;(-(`j&Sf^*F6UKseLaGDE>ssYon%&NsYT2jFR'
            'Q@@@qi6Tl?6r@{M*6UjZan*P{HZ9mX<rZ3_S(+++pKBwve*`G<5-=qBmf{jSzHTvAu&wvC<mMFX@a*x|Gdij#X8?W$u$g|#_=ffW'
            'by%#yHX8HigP^XcL&bDhneMEh!v*E5^cJ_y80*z@jPto&#bt0hIs1wt(9gbty?);BdyWZ4M!R$fo|Die@7K{g{%fEsD%fU=#Bs{F'
            'ZQXK)7Zq~KkxK;NgEy1`LR3&-&PhF|b+rjoQ>-{z#8}fE0|Z#$N*Et-W0jQSI&A~XWnl<7hG31C)n;3?+E9cKy2mm8XBHNVIG)nU'
            '<&c8M>6#2a60yRVaf>UnyqFI9L$uM)F(qB1C#M+v%gXVXu^9BLay9y<?~u`(YQZ;Xp(`7OVt$%jvO%wHAd<`FgsSVNsDzNdITSol'
            'jKb^rjsuqE>2Hljg=MUXolN_biQ4TCO{i3y8}s~}igw7UkJo2da$_%?wxx_1@T>%oG9!#M40Y|3gre=PVGk+eRk+aC#bcC(euR4%'
            'IpHbcDih@el{m{WRMZx^zSK2(2aO$~nr(<`g<<{9Z4%N|%UOcSVpM-K32~c5;b0|gaU=}K)^0)K-ysu&l!Dzjr7Rb8hQTPcw`19Q'
            'AA13^a}#pmRCMRufr>!Nu+0(tQDgPh5p*_?y|tnnF^xq;@pHOf71&D-um{8JG+DXShqZVPhYGUCxw1_ckdj9Gg8BH^WA^&um**G1'
            'UVK0M+dqNyg~~8K=l!=P-3+9~CdSBNAn`3MTCj*??QU|suV%3z!%N^^+X8X}tKROmO&5IA^WH@N&Bcq0r*8poJ&q?Y|Dm?o`?$ml'
            'y5n4-7qdYBq^^v?F{}W#XB|la9W}xf9^fW31m~I#?jexR5101z_OBicdpI<aI^$l?dj{#L`E5g<esKDDu)ML~Htd8=bcYw0;fG0>'
            'Bgb)akB3oUg>(guOtbET=Xz=IvB?}E$9=2xLr%TC(l@(Hou!!t^pe`!fXqv({nkmmD@|!1=&^O%LBU_*O`;pz01aJyXb*K^CwEFM'
            'O!qM>1uR8o3l4W_jFK^Qy62VG_tK<HNpLgfO9zFkUQ6k6lO`m$99o5@-SX|cciQnm<_(z6GZM}ATYX=>?h3`bQy`vHvoTmhq^PwN'
            'S9jpHG({DBfd9-)IW(_h3&Ifv0arfp^|rIjiX94I5PDSCvqr)`t!fa;RYV)rx}$`O`ncQp(JS0Tm13*W<DW*zvU$B_nnmuR{+Im0'
            'U-tcH;d^yTYF_+4PUA<${KM0O4Lnh>5b#PiG%@g|h}>TO9R=xRucG_fz*IW7ix2d*vOu9R=;ZPBjJ|sXIM`g|7v#-+*?g-lvU8=G'
            'S#~5mN}_v7)A|HV-9`59g@J-Lz?LluJ!kfs>i)dx*4*FTHs+n`C(Z(Gd=Ruls7sBsw%4!y*3G+)d;rZax?psn;J<H-fHFcGh|^R5'
            'TQk}2$MSgZt)p%W>s8~d!Ip{V&C>e~Nk?wtX3ARM6%YW*ebDu_kzsZV^RUYs-LkimgO;B=pBgohK45jQ?SrtaKfqtOSECw?9<?!S'
            'f)v>E9>zVc$^}aH3~Mc+jBY?PzPDOY<wu^Ts>?1~0J<$-0QV7QanJ!%#7GE4BfLDQ*i<-@rI8IsPEpt}9CS33^GF=phX)4>G`Z?t'
            'KeY3Mo&fm5wUoW`Cv<yezUlfftjY}LCSF*g_MF3r8!V-3ZyLL&-99!yZNqq<(Y)V=tqY?7Ggx&|Bn6DuI9}4U#hs;~qU{<3v^^z|'
            'gJO>puHXrVU$}~DzlApFF0X8pC@wJj5gknf18~Srltv}Y%f^PK(x^x=MW@K#Cl%(6-NZ@%${z62xzo(wAcy|0I5a!ouwzr&vkMK~'
            '6+U#=^Kt8H=<%j~9EqAZ-;G2HE|1TtE#cgpRP%@%*+es8O64|*iu%nmQYRnyBBK)(-4^j}t!p>TYMvf`>?-8wypS!$p?iRZsjVW$'
            'E>2C9+~$~^Q&P=pyHsSeQ+H~q&!*1PiG<TyHBe$+>*`tyEb7x}PzMHVttoD=9Vtt{H=JP<=U%Ti_XVDK@9f+5U4YiD3r<>0XiOK='
            'k;9@QUNvs^JJ-@N;?x}Vy4ATo#QbLTwNHs=Z*5}vxDH!d9@%G`gg1SXpzmh0W>aoqdi~S=VuFzbkvbm*dsaCn6|?vgx8jT(+|!TJ'
            '3_8}&9tlZns=e<{A~)Q8NL#R}e21Dgxx{L*(q;dA$JCK-yIRDw?M}fle!T8(oMJ+)tHbTj_({7wF^8OM$MhloF}r?*Y*D-HwjUg`'
            '4_S_(rp#<p-ve=u?ZK3N0t8n3i=q@WCsB`zFgP>vy+*U5+H68f7ur9WzD)H22<kNBxE!2O9!uRZ;oeGe3=oKlBr5C;w!@p{;?&~R'
            '`gHFUv^?mGYVkI8OjV1Esc#p(>ZG%0OcKSr9ZsKw3kw|Ba3ZV(n&u%(DA23>$9S-g9kFpxxo38=Wb9+P)>zHXYhv5NHKkfm9nE)K'
            'ogY*C*STCk0yT9>$HQ@$hkJ2DGfhv8=**+hZiHwuw(ePA^Ca+ZNN+g^+*@H|tlqFz2zmwI`9>g4jGDv-!3p0L81r{ERwFrhWA!rH'
            'O2K%kQdnjqt!zHx*wB*0%fjW(7;&AuntN7nY?L+^*0AZd8n!s&h7I?cx3IXbjCRA*LZyCNuWqr-Olmh&13WL5t14PD7}}2t_25xP'
            'PCr;Z^NpjJ@`}FU+H%5^t|ClWl{;4-sPn(%5AH2Gh@sIWn%9<pr~G$zN!Npp_T%vu@Va7%KDM*9-!_aJcbR{;r~Km|iLbuat{q+b'
            'VyUdav&&h=E2fqnd@Uy3l%6I2J#4g5L7Oqv3*b-vZ+cE%AIT0EW>Ye?!)^Jl5k+q~`XwIuT=kP|5hf!lQ^c{nqr0<qN4_T~_WJV)'
            's#-@?j%xD4D*k6l5iED~EoTl{rmMg&)QcBCq87tpF8YXsV+FR=9JY_OjY8WN(m!wAS*%u0ta$yy8JZO>UjMLR@ZN#|_^&oEW8-|m'
            'W&`b9_4tPlNu5x@E_b!hk?8#O9(4uT8-5Wa+87QHX+G{mrj4Bec-vz?^4{4}p^f{X?Q1WHyYb9FRxlBz6~?i3#Fe(5?>utl)Y0*q'
            '-?r2)XiDNop4=wza%Z~>y=5j^IvDUx^R(5|CT#9^2N0JYT|GK~(V*g0TNhGY+T83OeCz7zsM>N|vPJ>TFbT`mXXYTv@cKZkqf<oD'
            '?bz)nXg@Nst>5!zE|W2Pni^G0bZFxv3PYt+fyS$#BLdXR|1q-GEoY8}lfInLwb-1lo)#GOijU78<70|9b)KI}g~xuh$!IdXm#*AH'
            'uM>Pg0ta4WR|zaM5XG})OwLbfIRUT_R^d*!=GsS=>}*wp`SOfzrMF8>#+!n_;@qn4f}O=COp-m7H2<^`ogZ7BvuVBHZN_^E(=cA}'
            'B2gm9fwGz1Aj4(2En)tx;zIh041J749>b<>@dNB<Bf5chi1BwfNi>pZ&QZOibD203aB3<j4pOe>$R!YI8e#uV7i+MLG3z5Z*X2xf'
            'tw><sn<|$(%ZK*;-YOOYifJ#Gbr(Zje?ebwOa&N*N)kLQm(S2-Nfx#<f{Q^7rAm8Uu<(E-r6VZWNa9$@l+~BXWVXwyvZ6vJ?3%?q'
            'Qc64=$J<>xr<m`;t&5Ll&+DIG&VG6R{l%-<lb26k{PWH8H{Ytsskb)Kuoi{Z2F6!>-$(g~>s{iM8JvJPa}1{(H1cVNZf%V~l8g$h'
            'h-J-PJ=ga9VO7m72DDSasE%ldsaaDZ1_mJ~H(4H+YwyD4&(yW%+-usQKo#5~qL^wUk@6hfEg7`IBT2TFo6byM0*<OZ=?Y*tmGfvF'
            '-br{W-n&+a;`ps*v;wQAa)DT^<ce3}VujJHSC>)y`8gGI1rv<#C*T--#22Jr%7yW{@G5MCRQ9#crQfg|0Olx<|A-Qke`bwIOR;QJ'
            '#u8-*mq%UoXkSr6y8#tAYEOiw2yr<&{Rd~G-(pUUT*&+pS<jzP3aNRq`axGutES`@PHU|Q<PMxV!y&&xe!B%rORdBA6mi?~wd_!o'
            'E`CJDG6#5Bl>0r^mX5EoWSX>BTRNh;lIMBfQ)S6M#r{k^NhL|g%G`<h>g!clh6&gjShyozjA&Utf+aSRTplTDO6znTYWZPb5kJy-'
            'c_2neDvnTVWz$$T3X{AuIX%HBnD_=wSb?Q${E~2lCZO(utl@xl##$;3Q)v_RobDRo3qnkRUjb!I_d#2x(^U3sy3(aCd#Lz3%Uh>~'
            'B#bv=krqXq&4!X5D80eVC#J|93Rl>g!X&$?B2?C?;4j8lkN4HIBVUD6tmXJbt!<8{Qzt+F^U3SyVzA%RoMPbmGE0}08r)`S1zH~v'
            't=Pt;qQerbN&*|g4|U~=;x@|@E?|4&E$WjZTt?{Z!J<w;ncr{?egV>bonqc(vBAm_>5j^9$2%Ak7qwi3x8F)Z255aN#BD-nD*pc`'
            'SjPy+w?q1=O;n~FsJ=r$<CE6a&RZ`D2aQZzA{Oj!ua1|99R;&&skgUc)=}{Ph)P;c0Tri&kn@8x)M$#1s3|GU9%N35c+0BaL32|^'
            '<c-Y`R*WgJQXU^k`j4&&_NleN_SXPw>`uY(_|jgzy85b-a7Nz)wv4u-8-pSN<XeCtQ#?AI6h`Ffn*kAo0HZV|V2`Jwa%q(*dWW^~'
            '3oF~46BhfZt6|D9<_E1a+DMF67MF#&+KLJ@kzu1f3ZM&AlxkheB~nJzE6g#vM?Z0>&L&#s8+(OwwW)g8V-k-Wya9<qDmZhSGObv{'
            'Lp2RXEm({{p%AvQ!$EK269w}2fK~WdF1|r$Tc^Rs$-oUB_|87UtZh!=4K7~NZ1t$4gpK8>Zyj9Y{`lHv#b~)^M<WX-yzXbv3U;JU'
            '_uV1nbb{2i+_lZY1y=drXSf28LHSCBWgf5aRs}YiLM-sL9fx$$s}kY110#lRQ=r`oojsvbPFS#4D}c0iZngntT!AJ`YoL(Z3kp#<'
            '*vD-F@@|qg%#Ex8%2`Dz<@N1j<}wB1>yG*6t6m@Lb#D>gx*gXirQM=_X|?ENTWNi27ssJHNxu<+;#ODMQz-D$3j}t$iFsAi|J$yZ'
            'sm(c7O`kUJKCGV^7Tg9q11~|#o$*iG$zp4(`tIYlv(A>1UHv1<aq)%@+yFv3E<)I}b|LCDpp|2`5#^Kw{eRa=RR4}usQwKLP90{^'
            'Kco1R^}4GB)j@-Jy5|h$-=GQ=-glATMCFL|jgj7Ju`td4t+$JxNP+6F;&;(fE|D1-v4Bv6k1B7)GPt>&9jbF>x3*+{OEc#{({5%7'
            '{+4EcMpdjoxe7^Vt>aIwNODx2<A1a2Bu+L+OSG?7ccMFsBcyle?bHPaq*f{e(kU7PMlqIN16EYtpi`J`w+Pdn?XF<A#6X5vdo2QX'
            'Q2=Ky3ty37mprP%WCrBX3JeCiGb_&)5xY@Bph1F3O{AtJ-I*k4z1~JUjyICH2q%rhK5cY^3$d9*s1g<T=1Cx?1?)~c^oxU;L0J*!'
            '>f{4Jc{p*ygwcjKSx8R*9my|dAHpf3;7_~qE-uhXXSCmm)xillT}Y##FjOaa$JNPo9Ejd}DQ1c2%7LXrb$s*|B@7Ntj-Yd_X%)ey'
            '(~0G~_R9?QV4Edq8#ym7%I9sB*PTol{=B_>tUt{bF)u#=vJY3ygcvGN>ZHZEft3FjL|E++'
        ),
    },
    'approval': {
        'sha256': '8e6a6e6d92d0b51fe6ce55d06c96ec76680530d63e2335bf2d6961293a8f8357',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;NTXWmSl78o}m;*lutfF{ocPpt=bm>$n5}k0PNM1=!)~j-nATT5mf&dp6%d|@WeZTIW0R|u`Ih)=1!IA(>qo*%l-v*xNT?A>C'
            '#$k}CZB~@0OP%UGDB~<ut7;RZr@?lcXP<F<8I*e1>;3lAM|J<x<WBu^bNS(Ftj7O2xx2sXogVzxJGZMhRZ*(BR)M-3zdye{zaOij'
            ')LT`qf>N#U5}g+6{N4Td7MHZD3Z3I}p@Oiif+X3gLhH0gw+x3mQbAtE3p_AX?}MU<)1{gfVIFVGsW7dhv-`Z#GZm+0#$*>&8uH%g'
            'wLMFz;Cr%EZFG>z1xsGV1$YfMTfEY<lsyoYWx+-(h%f2!R>E8}Z*i$kPqE=emV;8d6KSe2tC9#C`>Yinq{oKpp4Q<VX;$J%Wd+_N'
            'fe(lf?gc8Z=5xq|8G#z^WJ8X+bXw*+C=r`2h&d~Z=8@rcHyfx~Sr+sg-WBy`xlSXUPpdLc<Z7goc+O&h(1S44kjeE;%@w4wtTuQT'
            'Eyuo7m8E!4f7W?VZ@?@levb9yG}r%K#kr1V%Fl8&TLf`JfBap|;<Tt1i#UuyWf}!#fUCV(1QJQMCDPMv5`=n{CE#q<hj#O9lZo0g'
            'HM1I+)n=o=gh^GzpNZp@&h?;2yfI&oD=lgw4Uzb&?KX*x#_xVQKmFJ5zE|0T|I}Yd*<qHJv^{J^cA_JQ`-jQ(<>dNDb^iYS?ai;}'
            'SL*&oUEEyXkFW2Kn7-Km7;@P)!pn7-<x$aWVrpJDEmi?ofqICOC|+u~i{Z<%(VOqcR()frrT$VvoIw@EnOZ=)_~UaNY54<Un}sVK'
            'uBZ0pet!m1r;sTmRfx5k7cC&*mdT$KDy;IH<X{*as3OC1YfSL9z<A7{2da6MrgZWC{%)Y4Z@R>mMHh*(a28Z$w!!j{T{=#|RvgtH'
            'xTT$DRne>E1yd=>O(2N^n@UTp3WGe~5m{&m-t`#dq@e4pT+v0+zpA!L7DUBBZL4__7b{&bQ?wOqj=U+=<ztqwfhzguI0UYFp4VeB'
            'Os9*gta3e_s(7=_ax!#yCOc5k>zQkSrWO3e{`q^6rLvrMQ^tS<bI1M-#p3_3S*qo^ZBP;c``CN@+3OkW`r&Fk{TT*#r_Pk`K@_28'
            'cq6|^y@B#X@buf~4BW$za{5Cg?)7`UHvkoqr(LONrQpcr_+oN5xw)=&JrLjl@I`S@Umd}90=mq&1%hIQ1rcEr)nQzaOU7lPlZEm('
            'S)|EGsyxKcO-7DfusLVs0qHWq<*X{VRS9aqL9!{~m=v>!mwmP|(5rorq|7f2)^KCmrkc@C9z6Sc@6&7ua^hx-SOZ)Ih8btW72d05'
            'Cv0Hx7U-6O`wS+>At`|~2$bWQd6I=|;5H}%vc;glask^lI2>t_J*^^;VL~Lo#+F?SANHWoi}BqZuw(lE>il|Y#XtRd{1NJHHbP2U'
            '!hJ+bi(Eroc{T^`h04K3H4h5j17I;r;&3-2&IpH7!eB%kV&0$&89jcBDh_&6ia0w`w}KCrc-W}eVB;1+R8!~}_O8djj&G-qInpna'
            '%keFl<de5V2=ShY<8!ORO|TQhr5JBt&*)1E1x!wE$MDOd1QQ!lOf2?j9wjA8t{~7Na51mq#Gn&4Sfw>#1A(N0z?IIa6FES!_73&P'
            '_K*Y>Gh42*Lc8?h6lPgSo`5$wQNH#Pkb7Lk;Y!W1gLNlEAcM_06A?>HDJftmvUv}%tswQCUtcOpO7JdrDvrMJuy_F=e(<c}c%L3T'
            '5RGDn9q-dy?DLli+mSJI_8<tbQ161I0D|3)|KkIMd^x=xUqe$<gs02NJxq;Zq^I!DD|NWyWN&J$mCS<m9(e<Kq?19}%e5$9xI^xW'
            'y_ASM#nfU{4{&sIJNa>PeSS4P|8P0E`I8I^v`pazTFlyE2=D4edVzE#NGkZ^)CibTDZhcXC=ScMI{hdAG7_?$H#Un3&SrL`vRVZc'
            'X^gRofeK*+9$I{>5iyX6Je^eOMwcfe^*|O-d>W{eMvJ)oG#n0}aZ$3NY3tqt?tOY5GQo4CbDM3sQ-&a5swge=-dtaOq&%=>;1Z(@'
            'Tt9SN&@lJ|qNBlPm<-I)f)$y5q_?z|f-52>yT<|37BMMxYLgeBTf_g%+&on}06RAMQ;Q^6);_=q79fJ`-%xw)pTCJEfSvtQ?0jHw'
            'u5r=+0mk;^J0sQ(`67$Cp&$Wj*cjhXjE~ooTsU6TR*H=lRl<Snl(Q{tncV&xcZW=B#qtK`lsUNLfI}!Mc^@T<Mu^r(`6nC2h>hX_'
            'i24cs^YBSD-EdA$RaZz$UHYDE&g)ZHczW*3#b$xM!F1w<LrR`BZrHcA3aN2a@058Oaz-BcFZyv`IJL@}O4L15ZDwmEbh&C2O3$|l'
            '$SMaj-E}tbPbo{;7mVZ|9l^BLP<oXj@g$c4sD#1R*qycXX^S<#7RzhUS3d}Wceq4$kmd0*PO(MK@~1^LFXJ-Ng;y`r@CkJs`uREG'
            'ln{*L5<ues*CUsc<}%BImbKK*k(Y@{?qe*nW%ezDU<#u5J>NW{%Dv}|#K9iFw4mZ3NJH)KJvm_4=r{blQ5T%WOQxA`j4KPK3eqAP'
            'P;OFEL}k`3(gbdn;>c*9!Knc&?^~tv0twkGX}4MLYH)95`!yt_m(}x<igJ!;>ZxAf@sr-jNT#{&Xh*`T-(2ZC&+RL7w7&)C`2z;('
            '3E=q$Zn<x2yX~m$RPx-j$r7aIo>C$0(|;pG)tvJJ{H)W0cU6)oAITPK+>rdgUpEd``ioPm2eb^jL-G9K{@v+66xZuR3))7grg9w0'
            '%YA+^;g09^hdK?xr0-Sb;`AS0zdytagudTz^Bmf)-IPjCKb!x)$G6knpf1h)cg=PE#rgHk_2lCGO8xsCsz81vg&n^@9g1MH*1Ljh'
            'gkp<Sy6Bq>qG=NrnG{P1d8nRa!24HQ<Z~~qt_*|ZG1wJqN9k%&-)Cw@Xt`dZdIz2r#3kOWDB$HJ4x}{@9<e}?H={WD7#Gw(<We@*'
            'RN?DL5QPixxLBLAj|@71M5X+rBtDH=1sdcOC&?*aaB4L-#EL2+U=s;=7*(5X(V#k{Ilv<Y^$CI^jN>ztoRAQwBtq^y;%~hT5Hzy9'
            'IP<*$MUjz*Iv1J@kT;w`Ut*Gbx_`lB3~Mh~2Q`3v>M;BnV<sZ7m$ng`2hO@Y3BQjq(##BK!Y`+e#2i5t018Dw&21zLS5>+uaDt+9'
            'KM6MTC>WV1hg7Qi-+%YrzkLsB&EXqu@DetL)t2<>Go_Zq%>8`<*>)S3D&WFjBmZT;AE*QMfh#(j<R9o9+J1%UmT*w{vW36FfXd9Y'
            'PH59it6WjxYx8Snz-+G5>fx9K@^#BbGNqSWEd$$4F*Hr7T>mpP-6o>6aLn^f&D?t4>AG{n@!jNVEVg0Rnfjyqo&}{Zrkr=yNZAy`'
            ')NuF}WtSf3IeWcn*EZz`lCGG0EJlpMJZS{J)1~tDa4EhUkN^?PPdSA)I|PKRgQX@jLfTdVLP{>oC7a<E4;B3q;oBI1AWGV?LA2YQ'
            'YfLhyjX3-87v4NZ(Anr6_Jh5^-PaIyl~<?%iRRz(XXVfJI)PYIT<g+)?%OokGC}EJw-wlMP=jtKjee>9)OWL)PD*9z4U8{P=7A^d'
            '<+eBOZGSJ(=yE<JP*rhA#iu&4R8PpjmzCOO>$fO_f*qa=!eFiDG0=RUXj~*30cQ<_&0_^D2C8o~GN@tu;k}L87fPbZi9-L+ZppBb'
            'Mu9pZiYLJS(C|aML8yect*L(pm*ATST6!5LQJf!T*x=dWsBgD%z*0>2rP-8e>#)CfOD*S~w<jy-<{uW3^k|T(36Yl8U521=<R6IV'
            'p*AiIw_QV>{kMno-Zc#XRvxdv$pG5&<jl`u<bWWX279x6DaI(o?N_b!t$j&4%Zxiw*WYh0_lI#IvHLY^TtKuie&oM7Wn5XYgH@?{'
            'l~l;$gD3`~G-QD_-Z2UA+Z+x|6M^8`Ztz>02#oU6W?8#bbI7?HZ!RC)_<>EE7;|vH4Yg*!2lnc`)T9K`(w}Qe2c{BZUdaktT|=7|'
            ')I$ayU8h<{JcX5nwoU>DxE3Z=WHV-Ni7_djb>PxY+EhARWxxuAdnY=nYpzH)Rawu>YUR{HD_vyNq;D!B+6)RI`r*_`cHxvdi4|2>'
            '9Z-<V{dTeE%qp?s8nD(X+H8QvJxPe0o1D9l8+BLl4MNdYsrT^7)@qzG<vXW8fgL_{nV}xo!Q`&dV827bVV73dT}l0ghu%(F67Y<<'
            '%YoeyErzN1cx2Db=*gErMMfAc!E=w9=Xhj4$0sqPm$$?6e}UsRTmw*oWfJTF&g$f;#n6dm=%oLAqPAHA9jyZ-gs9$Ph$ac@SiK3p'
            'cJSKskIeP^4n+6&1Y{<eZUg7O9wJkVn+rJV<sz&3IKJe9!lc=L+cKzAjw-a!GQtiVlBPJ*%A2T19$V0X49_%Pd8mdrtjQQY$AP(Q'
            'M+)*b$35NvkNDUC`!Atf(`u%)*|w25fkk<n=w`@R560rcq1Ozkn#L1NGN8Q|2OO=)Lw6ob|K?MtN%XqibZn>Nru01bWLhujA6tFU'
            '^4Ln=QHxaFT%Uhb<Le*CS636N`q?B?sGCojK<n$QK>dm=YjhwjSa?q-19dX|J&RLe$P7<D^`D38Rze4nfzhU%I#j5gkgf$WrRJMM'
            'nFbQJ6q&_mt#%diyZAI%qi(3#P$f)$10Ex4(}EOS8hWMN|8PpzLIsXKM#EF9yUvtqpqx--4}x?KjmX_2BZvB+$$V$1z%q5gztjBL'
            '+mC2++(y$W6~oJnf{+A+vlC14gbz5u<ar(yOZ_-I!sAStx!^0uOl;-eo~s3b!QYrk<<oi|*V=xxM8l{*JWx+k7b@Ku(J!hZNCX5-'
            '@3A?U;||aj5=c35z(YS(R3Rn$iz=Zs-mpe+#38#r4+VBm`$yF5*-6JqHv0&`({e6r<Q_$0n4ptjOfShY8XlP(!a$kiTFRFh-!|u?'
            'UCjZZKB*;}mpn@^hw7Ykln6a=zN|FOTk2C3x&EEAakGPWc!Z4W>_jl+gef1M)HAO^B!16lcbpj`Q}YY`(i{TX`S$UC&*@vR298Yl'
            'on8fR4+^F;4^^4R%T*~8yFjw5_Wb`3P5%GNo<KWqc769aUC((;ea-SJO+ZnAZXIvHBm_hNg$i{SQQ!_YI*EtvcOAo)-nW%|-jVf|'
            'p`p4-Lvq|ubIrc0LPW+mjmaC}ZZVYIO&yk5Q8J#Ub-xYuG94+S)IUd9P&*_Xj6wq`0<lRv5b8Bp*e91q)v`*flm!vD%WhNoyipfs'
            'jQhj*-3=U}4$(YllQe^Ds2E5iTw7q_c}vkr>fgw~#GR)xO4TRT5)A+-99TGUT7wmgvZAbH_<Dl$f0-A2LuJpQ8lHYRzMNcLPDZx<'
            'CuT<_J|~FlB(9@n9nPbebap6!y8+)WgLAZkIb!*NKyAXi1a@w`#La0hM0r;G;b?xe!EhkH+eB9){Gp|H(#43sBD~Nd@j6{@^jeXf'
            '@3Oq$`Ne)N>G#!|ww&&z4l?_eNXgaBHxxT$e7OWfadEcdCfTzlb;NOpG|Srh{b7<l>Kv^7?X1Otz5cBLvDh~m0Rxt;nsXbBecg0('
            '=p`qy@}G_{_%QmH?_mtXVZWn>g=fJPMtJw({QBpK1&191vH#?5!=XYAm108*8mfu#ARI>VAvh_rGzS3b6rn7&y(gGcmWXw{Jp^{J'
            'z#N`nF^^3V6zpdA)ZhPquO0ue*8WVL*b$*yHXf@JNuD`;dJU0vipHLs-@+q!{M^Pmd-uV%GpPP(V*J5p=$K?dCrk0@!jpkc{gz-s'
            '=O#^hhIsqr1mEW_JzhI)j`X$LGBp<?Zaob4?|>$?WtzVZJrmGa-kvfZrrD!!4;l}vGNjI026RC7?YpfUE&{NSe^QdwY04ZqpN<u-'
            '_qZ(@Q`*!yoSC<^kDwndZVvU;>{0c-lA%>9$ae_Y2BkbfU8&5c9&*>PW)4yXWs~u<xVp#p>e>c{Bu|Iw7qN$vXX`Ze$0b}UxA7ml'
            'FsDYW#rV#YshL6dyR>9n%(wY>p;dEh`W+CLNfz2=(@WG0m!M_^XXfInAcU&l@s<dG+rhQwe14(s?$58rtOU33!Kc~C3XWGAb&1gw'
            'Va<^sI-_`gb$3Ij3~WvlQOYGQ)|{|b%Z$$~%PgPghcpj#XPhp#Nnre!tF6;hY50BSsbf|xSI)Nz^0fv2-V{0^GZe)d;kBy*IR?mQ'
            'qaC+4X9DDio%6Q=m=TuOhwTnGUn%N36rJuOz3;(mK~-H-Fs{ZjTtZ<cb$bV_<?2ZvMMvN3b}fZ`ujjt1=AzXrtzBzxPS63r_PpR2'
            'ymU$uq5a~vJ7vgzlLFC29eJJ90Wxl!SJGnYgeXZIw3?gD6cK(v6%-19_$y-J4naSlVybs^*5nGa)yK9z{Edz4*W1zut)i%cbRBoH'
            's+BH+t%P14o`B5ri&j8q0)<^SA+#DgkPN=2DU(_JIKDtSGWoBOWssB*!M`QFA`>n)drA7j^KI^so;_JE>Rb*u!J9s_sa})i4b?}J'
            'v6<`>Wv3zaNA&nc{pw&um0Cmvr`I?4s`<Xnw2%b%Ms3{V)`}s=*jYEVn{)%P+8j1^&h|Rb)2Z6Ox<GtQyG@PQC|FQcG^!<2E}-GS'
            'EghZKD)F>_sY;|wz1t-|UNjyX2)}}Q0^J?R+uJ;6vdZ-|0~s-8_zV|G+&Tql-^EGN!D6tRD!thJZSLAO?+bza)dujb`AV^Lz5r9d'
            'o!?*lG`^J6OTy|v{WiJ(>E^?|^JPR_)(uiV1%*1Nu$>ZaY8av>g_ch^#d8SrlMHrUp8~S3BK5ERW=PE58IGt4*5q_vkh-;Ys55KH'
            'b7l@t(MPXxV8;2(mA~R5$K`&Dnn@IyQOKIL;~0Mj5Y*Emd4+Ts`I{n1IsV=>CMl6|KV0Hr)ZRC;NBiAxZ4K|he%$^nRQkL1hiKeq'
            'uTwd3Is=0VeWJw&CHH!otrp<R?MCOZrrl_3-MZ(v3%2Fap5tq+42xdf5cqPhQq?oSzo45jxxN@*jei-dpU*#DPFfYR)63uu`cE_+'
            'KEhf5g+7J>6w%?)z6A{uo!-S>g<RXo!3x>leehcD(k)i)WBbLbV-DWPwV^}L{O^j@BceN4jJ{UgKAV6mE>3-athYmrPQ$gvYV-ZI'
            'pfw#lo&rXuT-%zHuI)m$thR}M5ZtE!Kh@BBk)<d$kQsz}#FWU{`5<=P^=(cwQ}h=eWGd`8pX_$(YBGerOtzxzF*OOs7Glvn+LGA_'
            'm&WpqgN#^cTI77$Ur2^gA0^p8&g%{81LJ4*r1GKjz4d0y@oCzrFAuc_p+`5=+#t#Jd;bGEZ&}L'
        ),
    },
    'google_ai': {
        'sha256': '6a2cc8b2916cf3485b8fb85ca1b4b4ad217c00dec1f962cecdc219a2c7bed49e',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}d4JnDmMHqaKLwg~-&m}YWM`>#x6|r+DqBfu;w59tRbB1!P?SX3Y)Pb=r0ghe-_QQe0)PM)SxF`L&D`JAZ>A%Y00<l$oDFAj'
            '9OuxF;wT*Z^QOrBbk_3k{Ui`?<M?JCh;Cmr-Uf>>3dNp)9}C|VNxaH}L`>sEWVc}|mP!1NV4Ssgc7FNkS26ghe<4oJjxLXTqW8D{'
            '#o%J6Rr%k}j{yu3@ED7DvVzYAcR{iiQJe*%IG%~oDx7B`%u*5GM`CyrM1I&loQFY_HQMdAI~0C2*}3(T$vB<_6ETEgf_Z26hGyJ)'
            '(SFsM&i(XucPPSW9!5c13~v2QEc{6T!-{aRjMFq6!I!gGG+=GBD8AphzYWK?g2qZ=dV!cGL7<i?#(pIHakld3^R-9<e|*~%jS+MW'
            'f~9WV%EBPoIe&k$H-P^?e}4pPncM^!yo?uO6ok<YJV_#$^k^+;#Bn%*b@_8K^`jPmxQUAsjj;<4!Z|EJ_&duan1tiZhZ)1`)g+{!'
            ';@)55%5pe{Jnys9o><QPF#_4o4t92gc-9tUm~<AT!WUx#rD&u9Jj+^PGzpeLgwtpW^>e(2=eY1-;zIxDZQKhUo*)DuxCzq?w(UDH'
            'j-zRKvw}(HKoI|2#hDMoP5sq8OJRr?Z7~VcaSVH%0yLNYD4d7TG!=KDFNQp+HiN%Chp`*H(EtL#JD6)mpI?akFuR5E;6DIh@L@R*'
            '$FQleFY!HggI6!xVi}HSxYkJkoe?e4EF3cqizQ4CaUNQJ-=+=3mhd!%Rs@CK#h`!E9rTXmGUfs7Vgl`7wPE2|g0tQeBOkvJPp9x3'
            'p3x}dFd464IwOGgjJo`>E$;kzI3eHz#8!Yr2r8J$a+Qg64ZA<i@FW6abx)5(8Z7)sK7jszYKwV%lgdK^=y(ShlL$Y{g2ggJ=)|-w'
            'ev*Y#7zw)kxh+yZ0^oiORKFle;NXt<?BN-}1B{S`i6naMlO6GFzl}hRQ$m$ASo#Ufyr}^VlZ^ZYbQ{M&7GQf2nv!hjU0v_&oXXP-'
            '$hk^0Jlp`e7972Kut4ndqcwnzXFdX?KuiwSbT*UBqz*t}e(s(3-}GTOx^KJv(~E&Pd-JA$*zX>Tw`XT>f#LwYIyyVNJn5Ydx`Y1N'
            'DdKh@&U@XXlb&$i`RmA^iAj8utmgj0cibA<^!fkow8RM=i~doHkO}z2-)7k|J=og=`gKDfYEOc@JxQnhWw<wq$LSv9JPc7d1VH!#'
            '1il9c4DqtTbPh;hQ^-fD{tHfEhlpbn5TGznuowjsJ&GowwTF(#a{%MO^j5fAu-!cW77=5St`b1C0FQNJ%Qc2KxD>YyAEn6-fgG;^'
            'A@StReIRp4;b1(#MxX}SdKskcZx3bDc!Z?rwZ$}+eu_AH)IS_th%`o|PBVX<N)$p0xg%(qi+DU^f(euZjvXQ0#nC%BJJ4eehr5ly'
            'hry#6P-t%sM`k`m+8iu_=s38Ng+F)1zk~xshW>>U5Uv*V^ESM>bwmTsL5l$C;tp&F74=E@{NdU8dGC;r_bzSI6cy_k0lk?Xzy#sI'
            'BSHZA7&hV0c<C+Gj~O#X+<N$zs10oRD#~~}PtOK${1$M|CXAUle)oNU55UkCXVWPhZQwySE9e;k;{Y-NV@VO<1u>QwsGxrbn32x{'
            'h!C-9N#%jd?Ar_pzLF5hfx%Kd`)hI9gVQW7dZ$MhOwLiDP#<c4Q2dt;@X$k1VB(nv0r7z#%<LNYa0uJQoQVLnn-a$W5~H>o5e0}N'
            '@LnI7>O2jQ%O%0s&%!urZwGt1bI;q~vux1<Di;h9FbLW?5l{FKFl(Wx;YHvlFvA;QI<O<~b!MDf9vFGvQW}I7%s4K;2nPnI$%6-z'
            'RuXVaW^tdkC?XFy!`Aa59QlC1kgk?^ju54gS$cRRhp&lo(!oJS2JlOPGN{?W_{fs7;6t`OWzb(wUE%@!1=ycy_yJZ61Ho3s2?~1h'
            'Gm6vEGy<Pa?;9<6n9^6Yg26%DP)A_78o^fG=0o8OUh7|C83XD7s>4TtrCeknYQ;oCfKGtmFb@}q?l_v`FM%Hh%>E=?o5bbt0h}z5'
            'E7lJ6MIj6ZbZ|l>PNV{NE5p;X2qzJck4(Jx#~~a-k%f!E*&cX!Dq1`(fL;$-J<^mLSRZa22;{wE@4WXG1oHD=hf+Y~BMFD$@a*(W'
            '|Lx^@_ebENN4+=Q%i{qo0wo3#PFHY#65!^*g6Y^Eh}Ci)`;)Y(A&Eaxj$mN{<;f=en*cE(;Uj+tr#y)fis{ZM$nJw6Qs)HdEE1}8'
            'PlAzX%_Ihi3%V1)+B=Tp>=51Zyy*%_kLM{-!H*&w7g2F%M?M?*Y4GZ$`s)_X>pUE(KmU=&k@_8{>Q@3=6)*H(f%?UWuKvR9;fWxH'
            '#lO@vCL|@Pr%h@J$364;=f^mb9hZKFvr+xt!=F1l--s_MjC>=00_6y>86>u~NP!NHVW;SHr2(NY%q$!cO$Cz7#JWXn4@C(;cGwQW'
            'W*BO^is+z2zZ((;;7uA6R*UI8TrLsaV0DOGKFIQ{Sf%kQ0X{Om^`jfsLrA3f2K5HO{uBpHK~N4zu>xY>{=zWW`Q|{J6JnswWn*0T'
            '@bK*Nba1iv{`~CUdni8u#>q2TQej|O^aBSZ>3(qsKtRqR`b3UF0br$V$c)B|Vo&0<M?JknA}9)1aQIdzw*ZlW0h3iE!YK;F2tfoV'
            'aIdT_4iRoRb2`BnNa;Z8Cos=J@dAuOdZg%i1GqH|Huq6M=k3P6uqaS~!s!%FJdib5?9ct9-nomDo`PD;+Eogily^s|_{aj<JKf{s'
            'vtK+3D(^oo&jwxZ{7lN%9q|SkDM2|Wz3FPS2x)IqCEdYVN!OK-hbaNxgAPNoK}e)?vcfY@f||d^0ihdcPt@&p1v&=&br%NrNXVZi'
            'i>U5tT8N`*Zea?K_dwPHR6`Te<>(cldk82C{YJ1`i}uc&^Ip#z^n2&tQ5O~u45WY9gQYh}`*ftxg0y~xG%gK4ikY9mgb#ox0#{DL'
            'Ip7?OnS?VKRsh+`=+PRLv$?PJCWi;iK-GeM0SOG6uA({g6E^|M7Gbgofa{KdYt2^kIdq?sqLtvVfE&=`CjJO&yBg{y08|Zd2w{vG'
            '&sLG_*LV_u1Nib(jd$zM<{=Q$g#fCy_Sa!?$-Z3jB3Oq}+u3pX0Q&c$F^-q(1eshgaTQ%a9X8(ciNB&mLqZ`l7seMUQoaV1pVKzZ'
            'W7d!ntSH51K(Ny>Xx%910qMR+jXVWGfmHKFkoiE1eIj-#>Npag&%3{PC%vP7*BhL9#s)L<%klaG5dHGP`?-63*}H(p4-5y;g5moM'
            've-CH;Y^Tb#!GzQk;&yYp2H0B6(g;e1pi#YR!p2Gj{tH5qW=V&z|O5m^v3QuU3`dUAAT**1{^03&%+vqc!Zf9{?t3^dH}EsK8eqE'
            'c6P=flco}#y8ZV=_<N}GH_m|!!G!4#_kic%IOl$d2lM_GP{tRxC2oP6au>!(Q6_C<zB@axBjRZSCt80Mr$DCkm)^i~QrNxK1U3&z'
            '9e3OV@kpp{;@)RFg#hhxMEV$YKP2T$Na_S?4q6O`!G~ZBbjoA&^@wNzWhJf@#knnFJ}2G+)C3KGG$2~j$h=5PN7MjS9f@kjMmaqp'
            ')C=SSxZ_l8qKCX%{4YVkR&IoDLgQ{4tushg0g%Bk!4p08KhAx%Dq!GnfLT&XLwP#&pm5StJn2A8{aZNqK*^&CP@M#b2S}fLQZ!~c'
            '7Rd{cFF#%QGciL>R~Rk4oul1~iOgul+y<D?+E4u%kfb*7SK=3Ie;$P&$G+(PdfMk^M5`u2aY*83aD;D&rUgWb+Ibrw8ed!zx*+)z'
            'grA@N`6^)H{XT(H2-{6!4B+9IWJtsmX%lOrR=PkF+XN+77zTyLmKJQ;d@F6YUqf=~eL#+N2nTQ^S3p)bR&#-;f_y$FngH@Cl01;Y'
            '=c|+~lJ{_!TVqhf0F%)Mf-G<zf<O$7Lgb}sJYO-A1AZm#or~V#d2ir#kNe#Vlh(=e1w6qy5Dm%h-hoyFq>1fr9`HWzUGG=NmBeig'
            '2(cOkM9Xx?U-==ngPy;hT@EgP>@iC%_6K$LpZu}&v;WX<o9gu#$q1<5ApR@*6Hci8;qlq!(VOG$c~76I{!w*DP-?s^o&`~<<K^Hd'
            ';0o`0r+G*JBkCIIG6=@E9%^*hr~CIyXnyg&*E{^lo}kqnE#z@j?t$vq8Gi%Fx#T<^?7>m*bkOe}dmyO2=^vw{;Yc><?5Ky{Y4@ZD'
            'L%_=<XyY~mw`e5Je_nN4fA?D-_gjCy&VPFC*7d`F^VN&T|K-5pIy^h>{^-g5{;7BTzHYPiiF6nR0cAjMi2~qlvrto)hCEv?+IDV$'
            '9w8wOXF#oh3G99+ehq-W+=L(JCZTd<?_F%vog9c=jaa*)aTlz85v2H$3B0keY}ajyqu?e;4$R?lfiNX?T}d?nnmt$zvKfDp;N*~)'
            '`twP6gIs_o(KbigRVHfz#~fA^?y6=a<{h4Ne+5Y|T7kF&=N;spC7i2u>eG2djBnor8E`;jNCps$`v%Q4#0U1&SGwad=t1-hWll{9'
            'H=U*8zQ_&Gpv;e236@po4k2pHRst6_z!)KjhsFRWeg!@-fDecxl&w(my?dd7^dvCHAO^?t1wbud>Z2B34nasl#mb+b2U9IRbeUkG'
            '#K)EdVB9fq*gK@=iodgxvC5R`LSL$Z$dTU0_mPw*hDoqYKvdWpf>uxeGMQ(+3}pZkH;Krq3lwSlX-ZR0AZ^hs6_C;({{7<YRPzSD'
            'q)Sot8tWA)4l@t@8Q<8Tc||HFO)?q^yyX#iTS>MOy4phfGZJbb{HafBGzdF4w@A3d$WPXaOM$YNi!1cQ^U`^orJjfCiZcV`N7_q#'
            'z(Z*1zk<}m>ntU2PJx=>Pmkb>Gy_LN(a-bX6l9*)NP{`(yEx<l8b_KUAC(@6e~|##6xR4n-m#`7572+|tK7FE>gl1IOIbLFHckOL'
            ';6L_@d_wR#|D&RjnjN&uK>G+9)>GH+AtwrL)qS3q@31G|HsrsBz6?+93je|eBa9fj|1Wd_ts+}%Smzh21Z@(k;VjJ!F_W<sA2y5+'
            'xOF4a+kCKmoAMz6chLHpfXc=vtu4;*Fbt&)o*4w+7~z3|gS7%VnlyFPpW8}SHs)tcEypA}Pu5&Y>vD1kI4FD5IL0+|fJS_WYz@(k'
            'lMkK69vLRb9^NM9-@w1Dm0qaR-PFghmHJaOjImNl_MO8RhzKZY;`B^WAGEvRL9bfvgDvdRsXNGV)y92?z@XkUOx!{fbs!Hc1S~W%'
            '4)sA1|Ern;c1zPf^`J|?@`*5}@@NtPq@a=r=-m_I8#u-O&B8xG1&prvMGLRQ?*LLoC^e?eUo2WrC%E&V;q=HDBA-aeJQh?5{BNoN'
            'B%8{=9{J46hdD4F|ElO2P{%AG+Nc&}E|H0<rgR|9jX!g##BK;wQh)ZOJm)bsc_1(VIyF$VC;BHzoZ>}kJBaSW1W*wr3v8tZ+JXY{'
            'fYu2Eoc&q5BKT~>AETWYRU1NYMdLuY2<FUJi6i1A-NKJnz{+4f0U0Iq5m2-ZnW`E9+B=R5y9bA<;VK;(d0<hJZBJKo5DK#KZG&l8'
            'QJE&itxAVB_^bNtv(~GZAekkgpu)zmuz~8UR#9*L1ojl(`-`U7ZSTVWc)Nl`rQOyp8S56oJT!GuV@D(<qb-nfi2}Ra(uTnp&5h81'
            'r3XZfDi~q@pM@PAB-6By)e_P<&m#qS0jQXgUU)JfnuBN+-k@y*gbuWH@GhcYMEi(0SUfQ)BhmG(w)wL_#aD}nGq8cTeBnjlMwy74'
            '$h#fkPZ99)-i8Jw&y%(os`cZ=|1y=xXCqvyCYk0cuM~zzSCu@RST&l0D%4#ib%0nnkJ*Ht$;*lhz_)y{rgcYy;uBcA#Ac9f>AqyF'
            'UxS*B8|>o5^}y6L%5gyNzJKSVEtkZKlxT7TkI~GfBZtDG&;#R$509=}e?F!TnpC1ium)YHd^`@sgPPGFlE>E|SKf_R89ddqcyxGl'
            'gV%QStglq(Yigq)-9ZztYTk4aG6qzPS4@zIceAvWzq4Sf2Bw`~Hws|(<v8$`8Y>z@0Q1Sj$Ne`ElBr`tLas1m3p=WbAJO_IZUP_&'
            '@K~awKS6tZ63l=;@Pzq<;?|ib7N=HC>8TUV6Q^j*HJ@shtPd>Ops1_l71SX|0ag-3Pw5KPeM#6<4?`&q+D#>evuF$vGbzY7b1M={'
            '@(L|F3Xm41tQyBs3)G+GQEc*BCaa%E9qH;UzCpJ-wVz6zqC0)sknt^udrDa>^JtfD(3M}LXs{AKH1&V<$FceLkSvIFU<6ReOiW@R'
            'eSyo?$NWl)=_ua{r(&q}ZurCyr-;Ig=n4aoh^i54r;JpQSVd`&?akxyELWKE(w}@k$w9@zLo-Elgd=@<{VYGu`u3lE852LfwJBwO'
            '=54DX2`IQnRR{TCOM<1gWtFUeCgEs!HX9>2a6uGANWGXs+TV5`-Tyg`pL6)sQPUF47_`QThrbcq8|p`eI8%}<yORldduXePV(fIo'
            '^#Q%L&}R=X*^@is%C~ft?T__cA9+T!e<x4QyKOQhloU?2RC<o{fX||5hK7L!U*H}z*E9<jk8RCK9_3S7@uRvkV~uiT&7PH-{K0{}'
            'WhLnD8vMAS?WOLn*b}=_3pSoNg>5*6DZufWgzl~+ZKrg`$4a}!tso>S-Ff9RG_>zB@Emh`4@M_L`Enf_f_mByD1*2E5r<Ku*i$Q`'
            'k9r9Tss=D`gZa`!pR=4v*Z@?gqG<irM5|e}r;@FkEym&e1|<V!<P=DZ+RoRwsp1F+_R~k%yHVkb{~EQ<+i-$2V^a)~t<egN?`g^$'
            '&i&#yQs(t?j;SZSg;iI~=$tEes>*y+i}_8tDi19(ak6IiuFt`bI2O_w%Ntq7Gmv)`#y%KmBy}2pX6Q#K2@I{T4$+=386O!$G#eDl'
            'IG)S!5q6$q;w0VpeT;+fQ#8>60SjC5R!Vvf>D%P|elhrfBvAUqAor)?qy)e|uTR~v4_zgo*r00o%Mh=q*bCeeRM6QGB#%_yki(Sc'
            '=>NPI1=)R^%!Hgq7~PSjy+j5I5_k^ZW`S-MIsIV0tkm2qZ~|sHBca>24H5HCL<29rm!Q4htv;^y5ESVe01LD=Bk5ofe01AIrCrYa'
            'DI#Xv*q3p8$2K@A2eSA5oFc5_SzA0*5+vi(Bpf|fOwry&b*Et4W+ery*>K}4hAcqGfkCW%4f(i;Z{**A8FT+*h_--OB1jaFvWnPY'
            'l1?iGtNydn$fZPgb=c{j4htX^$Gd6lka^xUY_@<&nPHJzk=<eYjdo50^=%*)#-MmnH=l&B<+Y}|2}Q@Mg^w;)=m@?ThhnzkSd)+K'
            '8dK6{vRDF?{pZ(UKPOgL7qAd`HcXa_;odN#|4o(?4koB_8@?bpnxX)uSK@BU0d(jYsC-FRQHbGR6zw)fr;H1b2GM#Hq{}#>kfR8Y'
            '5*iu)*V60y9yBx1f7pqhV!*x;Fs05uxfU-M@B{IL-H7Jl4OxSvTI8>?c;V{^urJOZyX0r@4SIh=7c!?9xZ|h>7lZT5!@(uUqu4AD'
            '2J<)=Ot*V`es+0!#H}QXuno@B(FD5fV1PawPL`$mQSanO?rR19=-1u1APwVd{|2#b?5W@ZC0wBg!?^Kn69XR6nX`%TZHf&42SDBw'
            'cBo9+X}C?;(FKr_j>E2mj%naDz2-XPU`GfMq#ehBk1WGZ8DzF8{K4=a;5j)5>dKD)HZCj-anyr}Lra`~<?D=IXLYY(vDDkRNO1~l'
            '*YhYmH|N8kftLKkcw_k!9`Jph`OF-JAm$t}<c#O+8vy@nJY$i8oh{*|BhRE~tWCL=a4)=vx1$BLZwQ?g(LE3t=+ye>%EtgdDW={9'
            'QUF3x{;6XkJyY!<U6>uCTW%8N4ed0HQ`Hou6Qm!h!(zY+$-(!8u$|}M?|b|E`-Hfi=lgk^5)SJ)uO^3iHFcC%QwDj-FKBNiwUqDw'
            'yDx>e_(!?LAO)@#ff=hxQsOq0uo5m5EA`cY^Q;Ow-|v5&oqqNuR`h><0ldG52l6JTrNYL94jWV9{?FUr8xd<nMqE+7zt8^*;{8To'
            ';D3_s)P?BL1%M0`juvd3#hAT{ki522i6hUy4AvKMWdCb{q9FJlUH-jyJUHncAZjeaQuHE4I+8P;`1R~v6a9kMD<kqi52R4cGa?f+'
            'bWirt2#L~iXo|)a=z4S@P_}6-*^?ekI(YXu4rf^?3f}du*oGdd(+jdvOOCq51`RHnE=fS07rY72j6HZ}?13fbpbvMAY~xYfgHu9E'
            'wNQfeYq?Nd@<%_9g3(KU6943IxZ3|~R>_zF*qKL-ex4+6!ORGNeHv0ETq{^T+m{FPg$XVCnGb-_u_f^*4xx^>1?}o$(0g~>J3Z<B'
            'T~Z+<UKZ5Q7~mx^cq~B&Tor*^IE#xQi_kfD7Y9BN?RH!I3;%Z)ZDF%;Jx2$Lk40zv3cyn;7}1EAaNcF0cFcjac`%eKwgddaZ@p~2'
            'cs6Q<*_KpN;im-ovsVVlUwSX`x1oeA{4(w-J=06ax52`<>_@N3k5fS~q;x8<Ssec0uM-McU>KDS9<Am1t5M5;`QLs$ywh($qX6CT'
            'Ch_l3+P9RFrihoAHile->bG0E#OI@EinRZI>-qlkS5Je9G*tzsR|-!17G*L{nu3<`O{ZBwu!>;B%Y}6c+|B=oFA+O@4M$=bqN#!@'
            ')Qz9uuyDqR@7?)H>aRnP62!ai`|feS5{iEhlDNgjU^*vZY9$pqkmeTjY|+6eXkp2dt;oUfzuw;tk1x?E&`_?(!H4mDMWM79d#myS'
            'xlQepv%>H#(W;l<U?N{ldXw8FH)_8Gbr2nN5TS}K+NVgQ*CJ+n&5VSGwJ*9C$6axQVUQnd_#Se1808cAdY$04dGZzP?=#l(<WiW0'
            'gN%O@d>S=*3hKZyR#CR%U{nzBzTB({{QsS1oQq2T)o1-kL1Ba2pd#N@yDp87OfUtcp?^B)onQ10F`*S(P7jC}y%xR{>)B_JY_Tf&'
            '(})t~_}cnT@sirYig&tgX)&T$#^e6QK)cb~mLzpCaO<aD6h|L{DDs^aCp7Xx<I*;FbH}ZK0qvpa%2xHVZh5eI>2iRBc}<6OXmwSz'
            '4_=GE!d6h4kMipYzm`h~Q%~EHIr7v~w&snCp-2`6H3r6GN~J2pG&|d(108`~?=i3k%nWub6U-oz5g0e5Uek^ga^-4vco|qQ^x?|u'
            '_vUNdH4HGt3#~X6jrZ{FK)i6<;tZWURTjG`tR5Xa+4U1qHC<82)O0l$(=bUhit)t|&%xRIi#&k=XOUU>$XCTOT;ff>a<7{DQHlrS'
            'i{AhM3mo*kF8Z2u2mPNZO+eY!*{m=cHtP&#&61ex%Fq1h{djr&&i+J2*9wzWvr=4D;~tt94~?>&r>Pnq$y<lUX5G5^gjTAC>**Vd'
            'MwLpYKl?^sT{JezmC#{l(7kx)Q8y!`^)}8HF$L>uDWSM=l0|S7h{`<0ahx?xF<y>D>+$-@VWst)bmOeP*=RMNdE|6~NrCL|^XAKj'
            'cVXhgAw_ZmK*%<<R2dPfmSi=|6Dz<B0zX<UtJ{?fJi1wqMrug1Kp3bapXaKqel1a<SyCR#pOM(0xa0dEX*E#6Z8EKMwM|TrO{^`t'
            'Bg_OLI)B9*ZrfmnM2FKv5Z(-a>Yu*rpT6~udp~3Pl}!lrP<5qYnB6LF3H00gAb?sUI@$FeH(=@4na`FJsyqu{X+~R4sd(hR!fdvi'
            'NcEw9sPAkT`RgtqbeqS`1mO#RrJ9(0;g3`jqA&c7DkAn!b-|%_QcB_|zkq;;GDn{Dl>fx=<VeYSnEF&_4VGbdlE$3({^PO_3a7`)'
            '#YP%q>L2wdzuc<wIF_3gk4U*iHSU(%)u3OwVKoMo+f^b#xmndd7OWE%FEXs&6|Bgz^ls!tmYvE>=Wx>fn<u?+rIIul^iJNB)bvs+'
            '6E+LNxWfh;Xvy8WtWFf8b}768g?kp54PC$<xk<X5QY}bMR>_-FOqgo=A)*mX;j2o33d%@>Ex)LM7`#({5)<@52Un{c%u+j|ikFfG'
            '$87rH(XH3#XpE9Vr#<Jl&<;6V%G7}*tDmYLtPfJJl3PbpG+Y_1#G%?>-rqMT94aw3X478yQ8>jo5vdqG|MKp<Ig+0C-}Ej99{R*%'
            'IHuuml#?}2&8Uq@aGY}Q!Qj~Y>Fn|xar^npw$hcoX&B5Wjd=8rCPnf!m7{;bsrM)xXDwFfsE3%x@ocpu-!072%Lx}!m4G2k%rKf%'
            'I>X{1;!XejVxa65<2VH(6OIq848uBa4x;jq9T(H#-mnb}VnIJ<!8#psuuGOD;RuyfO6V}WL4uYg4Q+L&Ku)pCgY3FPaZe%fDaH(B'
            'lqR0gj#c&%QYcJJb{kJr=2DdbTT>r#7odY*$paT)P~p(X!%L{DriP;~fu4d9OP%1A4l33g<Jvx-yxQ{qX-CfpBc(APlE;URqDB(`'
            'z7)SOESRPeNwzqr(L9_5Vwaojm6G^tszTAA9fel#8DRAI%;8W=+!*+@W$npi+*UhX0wy<{J;%LjJ-gOn$-EAnm8U}>l}{$m$M#<R'
            'c1s6VJLdo_;IMmocG^Ge9*e`X6F4|W2b^`0vhCK`@7NH>;j4MQg&!E-72$u_jb|7@!F_iR#8o&+ui*iO-R`O%9=adzKDObUC`#kc'
            '(Geaez%e119+wg`Dw`k^d$u#ttp#6Xv|8FMcz-Kp9?a90khCKm$z{fVvNm$gUS-M>jY2|+GM&*L?uT&7R)`_Mb8<O>=W!s4NJaTa'
            'b>csqlYNY=P%fQSrc>juT-<4MlNrfmtXGXgF^14Vi_*b+aN-#V4$=$K;Wfa6E7&xL_R1lS-+<1pdgRGm5=O|or?Ou>mKUek2`x{?'
            '%so_})&<ASE_7-z^#8X2`{>pzfiFMu735zvR>J4QUIjjT+^Gtes|A{UpIr+<yP1Zj3`$cdN6!(y)}B^%Rx_58@47UCohG9q-ldG)'
            'pG8qN>cPf;`5s96C5KB^5jr14oVme62Fl*1D&yt0A1Dmsp(KaiMrD1ZS*<m=+kPpE@3$wWBMuC*6E_)}nHFJ`w4kUUj9s8s!@{M;'
            'kiMV9OYt$k&6Um%t-s1>`Na~g_ml%YNLra;-X>a!ce+}vUZu!pbV{o7&d{~u5J$pPBuLdx`ltO9SZ}(BXEL>=e}l1mnK(N={#C~%'
            'EfdggleHw^AY5Rb8kMr0Dle2b&j^0P`QP8ha~=#f40yziN@kS<b9@_&XF8|}&nG2N_t(2ArkRvk+)+lOVb#nqRt;VCDVsa*706mP'
            'jfPDJjKu$imMWQc8~$#nB--s&Hr9$l3Ej&=QZ8C|wqf*71YT^5yw;J)k8+8Hz{#tgs!3z3znK%h?J0Z-UXOLhaSH;zQ(`s5F!U;}'
            ')C{j(89SceW_85XqH0>Wsf+Rz)wSXl^A#%jfOp>ZqsiPG8%NCVy$XYwpOKnr#Z{Th!|}=9JVJGE>E_X&8c4v2fW<uuX|m?S9aYX@'
            '1g5iiz!jO4%4nP|IU){1l{!H8z#BEr5h`TOAY}*G<x*QZYz8x-!>LCSMrytv$>Gv01lltSIIZ3u=v5KsmWiiR&Rho~`^^m}8dyab'
            'ECs^4j3?9!^VD+y6a_pO7z0RuZJ7bez4G%?3GU*R^k$GVhI%8+n1HA5QKtkj?@Lu;7SNZB-E_sj5Ks-ChHIv$t1?bpsS&SrJM-)L'
            'Yi$Y4Q-!H~U0xpi!g;&szFIxJF3K>aIGF0|@A_U@b+2r5FOHh2zDqCkD#WR9AM0VUM>+h)fJE_S2_iZYhuh0POM;^vmBVeVhs7Rs'
            'P65z+8<&CNG;<OL1DKd-$k=ud6&%EK<-e{=Ioza7Vj());@JHFzF^#&vQ4C684uViSK0OoHtZ`L>)OnGNb$!YLCiu*xZEK%=PGYf'
            'JZUtIVkQg<utq(Nfv23_Jbt5r*woW&O487*ARn$ZcsNz#mjs96D%zMcI+_%$mD(-L;^ew7uUmC}V2)_)DdmaB?_-!5{V>%n+ARiZ'
            'nJ6$caLMTOsZ;sU<03wBTLube3FM6_kQtsWHsD{yQ6zXJ6{0CifR|Gnt`HlMo?BTtIB{bX%ek-=;*{`IGc<G64m-KxPNw{MI~BeW'
            '{-&zr;S^ZPGrnP*F3-;>pzgjMrm@NeYsUIzbMF@D5U!=(>{&5%j&FuyoIkAWp(HMY9rB~=`sPX^EB4{%X2)Fg#6-eI>~<k%W+{JH'
            'ZLoixC`^0&pweHOnWdX~)=Os<VbRUVF(pt(b617U1#a$E?_!k+!i=%378{{`-K(nJx%gmq%%2*4Oyo8MbSWqp4Rs4s5-{-(0|A%}'
            ';Ax{DcBHDQ&nao$jFqj*Ac5cqNKvH)^IV{j&qEI~&N<EXP|(s+RR_iBmmhb#JVvpFr%7EQ-eDbNK@`mNu@A)>_Hb?(4ZkCA>^YXU'
            'OTsw@W>r-zkWSf)Ii~FPQ6q%`lh-J}&ck9vIh9u#`8RXuF~%|n<=6*FjgU!H*iUjXPEPx(v>`~gv8%EM*{^LVY^Q%#Ym+wpRrBen'
            'A6()$?NsXj&FcIFs;^G!ur2eI++iaRuDZ=cT}?>>l`2~~OR&ZdAYG}<*}0AEO<doat@>g?psq-Dq;Pc|&qU*)FpSD_RIe#2;i0^k'
            'N(A2*jiPffN6&rCzf^=8!@c|g%c{#!4O}f`ExB}shQ|DQ7Nq!EhDG`qyJz{9$_rA&6NNS&h`m5h(%Xi5BJ-YVyB;8=;;sjvH^zP@'
            '(Lp9iD|paKkZQ?5rhn9iJ-WoTIlH^T9~i!k-pXnQbTOX)k>?v>2-siArGn-Lsla{0F0ZXyn5ZQeKXPJXFXr0a*_>dmv_#83*3LeL'
            '(KN2NkMW%H_DMkXj2hgpQ#QYSim6Vee#X`<<14QCODC_C*C3#F2p1XvWLIC%#s0(tp@vQCRW^0NG%Pa>bRba_bhCkd=hKaMJRWS*'
            '7{zWv?$5KKZ()tkPsz4pZaz$(v0l$g%wIm5wn^R}V^z)ktS13gxf{xj*?YpPU?>v=hPto<+Tu|i9UMyobP`~(ym$&fb*1=mFrV`s'
            's;n<f!UA7OlJQz!9&eYtf2SS{h~?@^+?0upstFrP7wqKx)yytvpsg$RY7;Tm`fKQ<jF!#JLQ%5(QC>o%$hb&%Lw3tC?_-DR9kr>}'
            'XM;MU(*)<V3Zik0t_mGzl}%fJa$L7!1TTkoqrs1ub{cAkbkJcRAPq+M2~34PPr=zx?i<DFbj;k<m7gkeFdl=?B>ca@xH|fmY5uc8'
            '0a79UOR}JI8)oG^_9~MVHQzFoC??oLgX}m?lGQTf&Uh@7fT(0qjehc0a_7s+p;gyf70+^@8Yje%r*p{RbvJ{KYBzd_puiV_b&d%`'
            'o4VU3F?NG6{&lWpwJa`0p8D%jw@g(}*jrV{+w%1v-K-U_|2ACxsXI@ZWff@a*_$__F+9IKRW6E$XQ!t<<y3j0%ODO_nPf$9-8SbA'
            'HLvvKJ>F|7;J4;<@8nLe?qTg4J|=m#VJGpU+@ByLbCzd2Mwku<l4urg{K@8tI;c+dp9W*h+&Kb)I)Yye8LEhh01PIa$Sqqfu!JQt'
            '3r;eMzTRz}73sf{;|POE`90LbsYV#F0r$DiCYIfp@B>l$z&2VE**8UXP3BFDM(zQMPi=N!Su9;MS+cZLSD9<ox6S(?)1IBXo}zDO'
            '_(V9Ldu@J}1dJ-{G{W>s!}dD2_2QQ>I~aXXTYlwz>ZXmV3wKZ(<`inc@{va^?$NEB9Nwzw(xwG(jx3B;fqf%|x@3nJd$oVPQyk4k'
            'yz6?qcuBfYj+0cp&h|AbW)EqUph68hxxGO)#Ec?{nPo8Y)l=ueS@<7hf;}$$hbQ1qk9G%l%Uvy>s_Jbv4E!4|^78}^5un9lnXOe4'
            'I8VPJp-cVcx<%hAYs|;`Oyni9vK4}Sc|g<=r{CkO^?43k2EcT^9$e*DaP{UYf3_!-_)e`=rh8_0?>~NdckkZ$`PsR5-W~M3<Nis1'
            ';GMreL6M5i8`(nX9=NLAQK{WTwZn3mltgls-KuA&XP}#4xCKdU?1K^mC!}%%vdu;B9E&|;{!0otm-im>?Vx|s17b}XEyG0+V^U1n'
            'uluGq_?7zk(}2P-robQO>V5z8=k9U;$b;VRd4Mn?UC*n?v&mfy1>&TCanV10i`^>No@{@4`tJ1X7Yx<W*B4x^6;~i#y*L4M7Gqr+'
            ')<m&pe3*$5s8F*Y*-fdyB9<@??}C={&!BEtjdgI|J-z6|tUUhayZ-4Bh1#0XYYGcmt~m>?zys>^_|`1yzKHLr>O9MHKvVc+ws+oi'
            'kB@)s9=<Dnj}Ar^>NKmK9;v4q1)9bHm9tiV+8fuz(%~vg`*HtW&wG!dGv}xIq^!cH-Tp`aYQObou6gSH*1CRpw%>eIeuh@t#t)(U'
            '+XxdFUtz-^Ky3XNqHmnYL-?;z;3q-yuMaJx8(0p#1#O_uZ|S#J2c2v9=eLXRYR7I|{qS$sjUNtZ+-2zT2PkNrIgi_Mum1DjeoKFI'
            'cCWwNFvWD%GK#pGYjSnH%nOaxXN1(8XFIR0QCI9+Ghu1vq%%Uff||K0h3av!J0)GZQcX%3C6unC49(qKkDa??fYC+oqzi}X5b)_&'
            'P|{*d?oT`|cN-wr{U!$Bx8i9F<k=e;&4YE@o0yUc3(qIv%}tQx3EL^BJLtKntx>{Jj9ij=r&0_GU`*yR<q|kVBF>pYY$}9KsRCr?'
            '<EgB7CYZp;+;?ePmZ#6uywvToPVjD2NUvu+#h8c_tb%2g`E1C#pTsOML?OCy@x)?YCpf|w7m;GB2yTF!PO5gl#K}x&;*!Ne_xR3j'
            'Zw!kQ=_&zjgzJ(5DN>9GvwXzatxgD^*XxqeHb$Hc8RTNjCqlWyRS6I1RSH%yZjXs?mp7wqnKx;C?Ko8!UpJIZRqBSU(ZT8lEi;TA'
            '=>R;DHDSm;NJBgL(2<o^OzR=Pl~<x>BUUz%Gy{VX4%fuUy=kzzo}Xmw&WXRMIG82Q*q@M?poVW&&LDrQ^}zf>5J84tGK@3x_+DNX'
            ';7r@}XT#MOr349_*USB9P4RO71uflKBKoFuaR`tOsjO$D$HEq?2q`h;qsCH2*8AUf3Kl@jd5lJfoPd?alUeFLDFm4BYc>}J8fzJy'
            '4lLj$jh0I>#qZqvsW3_^ZJf2FxOw@_rzfS^X=!H9|4d_|B884}Uhv`8U!_?EXrbv76JxiAVn3Ao_^!NqqA(i|WA#!GBe=W?h%#$='
            '&nerwl|_xJ#D|@y+nkhe43p%F@AseM|GdEedCAyD9{mx}G!`v4uN@Fid4!iBfVWg=1g`)pd~u8IW5GvFT#K~D!RQK1RK1x5{sdiX'
            '@#Z}UCR^c*l&|zxnVpq}9&wql(sN*Mx8W$vjC_@wAqb2)l%6G9KqM#e)^jOaD|ds<cvY2RNd;R*>&C)QW|&AQpETDmRZopN>Oh##'
            'PlU=}V-+|0C7tS0Z%Mu|YfdS^1`72~wKg1*aGvtKV24#SU(IN^rE;^;H=00HQ@o-9Nya0A9Sf7iAZ##gt`3y_DUOIU%S>G@DThoj'
            '7%*A4x+M7Tv3ztC%>6Zsd{lHRQlT%$2w05RZiKR`ri((yPlLgGF*F7mid#TEEZyMaEgnvEf@5MK2>pRVoGjf0V~{O;4h^5=bRI?='
            'Opahx{m_W@;#um5Db~`Z5+9LvT9<6Atz#VN=8=|Sq$x{%_IS9pil5`ZJnz)t$2Q+n0&vdx9NDFwTNfHsh!e6!Dc+XKcmk(Fv~vvj'
            'GM?d|zWNh1p7UQ@-8Z1^I0ecH-;#sK@04$wO4hc*@mu&G8vAX`DN6V2a-1s|_S!99x=Hg<Ls#%=ijrzks^pGEpL9t*03I*eKpwql'
            '#saHe(0x&W!e1u5mHii5O6xBnn6U@;o`KK|C(;mb3Okd37TULEYqAmJAXVQi7om~Wdhz7qg}^utkQJ-5tmBc5p;YVI;jHrM)K*@K'
            'p+<&=(>7<H0A-OO`vZaYH|GaKNp*f`{5JUxRMFqs@UQ!WoBrS!5+07y*H}zLmtCC*P3T=4HKAqWnQK|gRNT3w*cEF(s5mDSw+*x>'
            'lZfb+W$gl{;z(}YNO{rh0+6_(bmIIcfTI8lc#<Np{+EN5pzp1nF2G_dc;pD6GXzj@x_n+F8XGa7!e(9t6FG)kviELvV2jB#%5Gd+'
            'Sz+a!CYsmhSe`S)<XRww?U!`j0nu?_&AGO_M~V(~LH#zp1$o#Ux<vu!tOK`I#QOwm28o96%H+53r<wFbmJj73o0u~Jxduq*C~nWu'
            '=~`d;H%7?w*2Fv5YPUdt)3rE0J9~G*`3QU@xO<j$NYdWDe46sQ{_^}7xE0WdT$k5LD-xA@FwJ6|C4BcDQ)#~Dge(-=uQPQP0@hg-'
            'gh8uR*SwhW)Oo;U;dHdy9!33m-s6FIfCrCNT6w)sH&N6l3Z}_vk@}~+#EP}R0Q<lF_FK@FyO?woemW}Pv3@4BAp85<ezfj$tSMum'
            ')(Ck~rCG7`4cVccu11ZyzZgw?u{aQog+BG-yYlCVc5sWdA@`-&XyZ*Ee!SWOKx^g%7B&IK7gq=J_+5X6Ym4{(7_Tc@sGgvd%sEWZ'
            '%K%f+uTXG9>GS;&)e*@MFdRb(2jd2-_VJa*t1o}0K^$kv)AQo#6u!=WI1QOPYn6=;v{#}l!@M;;y1Y<p6PJTS!2=wK)*r=Wg(OED'
            'T@0F{^(R5)?(f4?g==4;Qycm=k~1aCq#p$(oWS%GN|9{JD(5J!qLqaz`z2K45k{uce6oUbAf5xpUm#dXygQ*%8PE0)_V;OO(qR8A'
            '&5TB;pqDO20O0*U%XThPj{agf59GGwImBgZaX|`7i?)hQdoX3&vWVA&;+HU*#P_L8TJu30{)X8{T<DOJ*I>qv$%gzIRDDnbO4c9z'
            'JN(^;zjZ4HJRp0%Yi&PO{-X`mzTtF%K!;<0?>J7qZgdmOgVb?dbTH*18Wt0m!`6ye^85fsG`MLy1FW5E*~Jxmd*a0_U}R7D+-iw;'
            '4I-Jn416L5w}fO8M8*a_ZHgC7@yrw*yp*@LBO8Bb_{75Vw9|McTKukk9|SXah7Jt>kN6`r{iAZnl{jI2>De>=Dax>ve@VBm{+N1Q'
            '_#O{6!GSWgK;~}D4QnZF{=EHc5l85!Wj}=f-#{0!?hR(zu5&`hnFNvPPTerqDW_pQlC!X$ZJNZNH_t%g5|9q`x7L99i-`>4FM6Ux'
            '7g~SPW&oyO@O?B^RuZoY*IP@pf`h#)a|2Nh2Gr(}B+DB`F$(4}mPyItCZ?2NN}#DXqi8sZ!kgQyrHCTY2<<y-ETkihIc|3R&ee<j'
            'P}Di0TVF8RKk0;C*U=y%<xcZ>?9Z+C)*)ce&ZoGG&5CKrHXS_Ij3t!&jQ6d2KmzWzTdiK@tP7X|i%*%#9>%x1#XZz)`M<+HYDVxR'
            '*tg$79zjw^+yLt3DL|Pa7~^;`3ISKlZ7dVUXYMDZ|IniYdu<bGa-XInRTFL1E{(CP&LI}LH6f9eGO|lexYGg10gI&|f9TM-P@RNH'
            'z~zX)6LG4xPi2Bh{8TwA$5cjsyh;)@;-Q27*pH;bGY@>qCU)!3r!D+k7DQ4q+|$ng+CSg$ipSr%uY7yyCzKs+2D57LAE~Uo5q!Yl'
            'uXsj(O6eKa)8URNxB!KpPQwrE#LmBAO;e{0?H*z>_bkIn!+msg_cp_m{I19iezIDCP>;}rFGeY7L!O_G!%&qnr9$yp=Q$|`zhUO1'
            'l8-zCE6oqWV%ZeDPIYDNy@6dgj>og2`!4`IPBE!^JmwPb<C(Y%eR0NF!~b>WVP%*mXD{+m;G737Y9+55r0n`Oj)DI~@#^N59XMgP'
            'c{_GNNru@%H(Gkq!X!pI$YO=oSW?We45Ta-KH;0XIn`5FA|o|A4x?2rGy)J4yuX%}?8Cs&ahZdoB=5mWGtUDVVm@WKG}Yug&&cet'
            'jOX(X{QADh3uLQ#=M_9M6K_EOwuwr7X$oM*t!X1r3};emEkDzgec?0mr|}ul5xSXdk-xNu^QIF?zRl%Py$88IW4h{cM~*Lv1UiNp'
            'Gb=Cd6Hf&B%u{p?KC>m^N~5u{5I|Wh_C;{i-v=@V&n5Z+&!RsN2i5Oi&e1L1tv74Vr#ltKW;_5roPSD2t4}a-WAvT@z?K0k)W3iy'
            '15g&kV*gs&-Gf@9o{zGMV4r3U=?_+hZ-wuv?O0n_>e)rj5-qJ_!xI16(b!Wx$T#e%FF3UgMF^-;7*i}72d_xm!`H6`5tm?6GZL(x'
            't_nqiGK3+G5MhSUAxD483B;U7>Y>|iJZ&njDxC+xvZ0TlYaN~s25tBj0O%nLb*+17jk2{mA0Xy<9;ZP=HFbAta9*25KMe(}_{%)U'
            '?s4zvZSTA)!!8Hq6LpoDb<vCO&~_J8R`%5N4?XOkA71tc-j8qq-@)L6pU!(1Kb;+;+vSVW+ru+>bN0(ATEC#dv+u>X;?=(9JO6>x'
            'I3_-zInR4cv4MHRf#CHn2K|#RmJUL$z`jy!<&oJ1aue+UjhU2&^Y{$Y^Qo??AeoPzN8v*FBV-NIwwiGnX?d-ua*VR#O%WxW)C$L)'
            '`y-6tn1~@J30R<?2zyo!+oGQ_?MvC(fft-0{7}#!6{l8pb1EYT;Y5HSz$sbLWUP_|r%M6>vNg0B8}5`UB^G;)p_yjklhv4#^U?Q-'
            'zmlM;W~;}gM5_cUw2S~k<p}|g=oVQjZz+5ZhKmV!xs0!jS53aIP`+4(O^yZstvIc;;)C4pO{2P~rn<1u&Zc0z4Sbn-GI<xl!#vBj'
            'eH?X&G0Y4_O=*rjcv4%@LsZ0ZZjJygKPrq!w!V&>Ah|c%wx@ilR8UHjkBaC^Jtn%CC?6^`l;TN~dHaLUwVUQ!&XXk3!;UbkPq4am'
            'rXBXYSBJ4O&Vl0C@z_&3o^o7=^K0+CcPs|opHBLxZM1T(#_<1<SO?h*1Gp#=!eo`K7AsiL4Tf#$oF3(1O7wOTnD-;YEFf6x@)PoY'
            '7`Z=ew+>d?J%wuMzffV41f{xgLS3hrDC|=*vJ?{cD2(I`PF2WV=(<`Wn#(Ll<&&MKpLuvJzVV|K3`?snM%q2*<b)7cW0qsE5@*Ae'
            'CVN!bLB{z(**emghcP3K`AIR**xNoryZ%4>+KN9UJqmL2RSrL{?SsTY!Ji$K2qtww$;GvH>M+2q5$7e@;&JMmv|+!c#L*}q0p@LU'
            'w~S7%rIcpKvb-cBdnTGpslJC270Af2QzbuY<V*d#AV)r1;!{Nd6;@T*|I2j!$~-aGdhW<%Y^$DDt5D_Eio2m9+21Q0XmrbF{`CR_'
            'XeqaJPU-pi41h<V@*>Rj%auiJp73;S4T{_}c83JCjtNK|9at{#p)>F(v99WZMvhn!X11Bz=tOnozb6e--U4(niVohdXw5IZd$M7$'
            '`M#1|+Fu7jDi0{!?n9+%O~@0js;RsXsGmn(3(-S~Gp!kibI5LN0!lze4eHoL-U?Xj#zx~fo&yXc54D1MeLH5U+F|*r=0?1fNA-YY'
            'cr`^%)h$bgU3og;9BF?Or*3UB7>(31PgN1vums7i^PFZC(U{dVF+Utu&R53y!lA0DRoXF%))Zd4n|f!PdS_4d=0d2o-=0+Tw$r#;'
            'eXUoVLj~`zJY!7Bg^K1n4;3PLR?`N5s{HBvlqMj@hNuAQ6`&`TwK4lm(WqT9XdURzq(`ni!`D0IFk&52O1qj%<JSC{9?Mmfxfts|'
            '1{2CHhjfQqygr0B6(m-s#*!4p6@S2sk98Hnz0$3&>2_pY*`^u<&|$x3$g0DBT{5)D`($&@Rg~FQw(!w57ENroDeD$BaI1`NvTN;?'
            'SM_I9uO~0QiUK2N$dcXNl3&7KJL1)TotrPiy9#L6tVUPLFk5Z0<ZI<!u0MI3_+xpT&GO`zSl{QZ?f0y!oD=v{*I)>QNzj_k`8x3T'
            'S_H@>OeB$WI95o7O;NpI<>a|vnZi+;n|`@nD?`}H<;8$<*N+19=SD36ty8oPN~Hw!75hu$7yfq^>B?Ctg{x9}X6^o8u&9Ch_H*w-'
            'S0R#BDRQcUViFpzkzi^XgvldVQrT0{*?7wVFra-Aqtyh>fn@zNEZXX>vMlo&B6eBUt*7F63nxQj$+SuS@{JmbsY6QGAd;<MKPKUm'
            '^;8-~sf;}1?-ESflZN9^01uft#q?Fw)TYKX(UOKs4L{+ghHvca2{Dc5)?M;&@2zOD^)cD+lc=#)Bk#pu<N)<w(&<4TXd@p*V-#i~'
            '9G3@Lkje}CX%w$?7gS{F1sSy3%Uko|LTe+j%W_g+yhYiL#2{-LOR`lwB@+O<f|k|p^bY6v&HP(9bq7M?Hks(eB%ZG*yDE*oTQ&Ob'
            '_2b?H4E~6YE(Ul(4&JnwMde}^n-`<0JFRE2)p{oWES&kgp76;9@fYNp7PzJpV@=tnHD#4+%H*1=$vD0G?RPV~@9KvCD}Mh|z(oCQ'
            'VEylh3?VVFAYnx+XhTrsFVovN+tR%838cH)I$la`^GW9LdgH{C_*eI#Cu|X)xUEmkm#r3?Pk!x8ZGG(EX^rivk|uR}@>4}k>NMo1'
            '%9_+^ip72f^MmeYJuiQ)U_i;gX1%yB-BBoWQAv?2+3i`JjMMCQ%9BO=B3VwVWyx<r&YN*!tER~FG*x7ltr2FP5JsEAHrs5<%~*)J'
            'mA9Yrw%yBaG2<t|x>I|{UNf#5LGKg}zUeCd_fe*Lq4H@i_FYFj+xcvo&lZ&ZCb`eo(0{U;Y%wL45-R6onI$*rcmIWKJ}3`ch*+)t'
            'ZLl};LRA$v`(?xa)7`N`9%j|wD%pt(;4YZ^HrZLr3bK|*OLPrys2p_s^iZ*K>b3b&KW(V3nW9Yxi&mr>%bPibWfCty0~VB%C(_2Y'
            'Pk`la*lcMy--O2YrvZFb->^UNjX2`MaO6Ej+N8_?j;RiWk=R1&bM;Ol+Z*jFT-nj=xW$?<8D~{;vxCn=#u6Hv*!Wau5adfF|C5lE'
            '(7L`u8BNKuXS$S2qh6<qak$m+r0B^iEYVqFYOK0N)laAA6Qk99#T6YeWzS3ZH9SE}3dV!M*C)PE$wn9cN9QT)Ho8}N3R?R3*UII^'
            'a^{~Pm)CAgPIPv*B_4YE=7o3Fq@}FVltw8j-LPxOXstY=9H)Y{Q^GGCotvW!MyZ?GX*<%<dUCRKI`JEZVN841MTX~{PK*{u+NqnJ'
            '?-V~I*Er?r*0ekf*jOf4pt6dj;Y61DX2&;}6BhQC_wpm4B`Ftp^eeLEx6*rmNq*u!B~Pn1o~d-YY}%8RJmh+1Y4&28hA<6`c=ie}'
            'BHTN`XR?i9ZnI<#SS<jgSjj#KX5&D7j1x36Oje1|$%%IZG4t0G%my7MZqXGeKY5tYq+Elos0mU!oMNDFvBH~LVyP9Q5w+u=$~s!v'
            'k}hIv926>w16rxE+2x9L@KAv(c`Z_Mz9Jx|_+~=&L4gniAKO-yHf@>k(Qz`RoXh$y)2vBDzA;>(v3&n&TS9p(PAN^J9z1n_+Fr)X'
            '1~)Dy3fbDDMJ2SQ9WH79^i<`JXNn}fcUj$GAC?2tiFRA<=ZOUJUr^;iGa@?_>MA4Ty1Z3kZAuqJxmMVGuQSh64hA(zEAG3U=X#eE'
            'T|T!o#Zt~`1XCxTSuo0f=4-2lOsD#B2~PH(X4wJsMwMXF-;xt2SEE9zVCH-r-nUU=*KoG(N&rR|OmLHcv%yIt81-oqffdWSPenhW'
            'ZzC_!eR1?o=9h5WM%qP=(<vFni^)E0Pd53rLTVhonTL&|*E>dfL1fSL677%`HLSUbOq(zZ)^;wd1Dzp7e%EY4&S6pfPyayl{`lgV'
            '3|bE6lv#xegaG>SwtB?=@G4`B6|?XzAuYb)t5fUYPBCNkEe1nnX?aj88X~iwS#MRawcK#>QKCuG(&QycezAkpjMwmJV$TE6gF8+Q'
            ')r22mWQ?e)uHr`BnehydiXBL-QwVS2J7tR~<vTb`R>0K%_siD!)=yFj?_y+AV>FqolSrat!|0}M@cu;Hto&qRq8jIJsll{;vlfkk'
            'YS^imut(;)Fv9G(RXNPAP}So@<5?3<XC&9Vw<lh@Um*j=IaL(k@}-=!JNQ6~g-+O=iWlPFzDCw=vTW0b70aQ}11eyUa~33?TJXeL'
            '27td9E|7o}m+AyPW(|^B9>KIu_6efAgr$dt!?A3*#Xjl>!i*%`QSuDUw(>$oe^aGLWQEVcpm*|qaPisva4Tj;`mdSr0?6!Y5@w_#'
            '%fcy`5s!`l@;sw_3cl{_B7}E&LhwDaggOU?U|xZFMkxnDeA5*$zEh0E7)nX}yEGJ?yrF~%)vRE^oc#<MUjL|f!DWDluT{=Fm5sU5'
            '4+RTHQR0u)2$q?xxCnD>DH`M3I2==cl65L>VYR6W6#WHtl3^tVbYxu^9SMU}AOo?QlD1+P3?^%)KKwz>azVkRLfX+NM3|#xFbO#p'
            '2mw%YCSJP4FC`SFQX{gup;6Dn%pO<J52g;5#)uZd8i<@6`Ub<^!UOSu!~G$_JKA72$C5AbOqOUhMt2`w$2Ds#su?uYIdc>&?L>B^'
            'soVr(lq)_8xxF+#?Js;7M=nf8V%fh$jw3h~jv-cO8-p|TZ>pSPy*FRyB~~f~0M)8UnT>(2)bHyZ3m$0W(;%>mgj0zS6a-N6*F#|l'
            'kL#AzK-s?X!f>lX6H*0hR#nBOS~v8#5XX@F((1a`)HW|?8^xYlpQjd4<v_Z*ysg%8oYMZAQ~=f0P3M1?@b+={x+3UBf+Y$Ub5`2s'
            'uZ6A~%7hK8AuU2Kqr31ww#$RKRo<(%bhmM9jkCkVN}_DB)asM9<8=i#U9NuHVnXIyqoxpz%C&^(GQb+Xw%(9#XO(j%`&3plA-nK1'
            'ef>&<f%$_(Hau3>sDSD`Wm;tpqpBnE3FUE#<bF!!`C=j6LK&e_FmrwV+`6*6l{Ixgd7e*K-`!&7TNmd2bTC$z@ZDxoHMoZMx8kmE'
            'Xj=B@kTu73!c-H_ta8D>Yu#!}APy8tQxPrN1^_Y`XhX^_vK4AmLeYmYpI5b@=$5Yv$b?&V75<c!^N&Y&=gIhgI{oERS*EH~EpnF%'
            't8rbfgJ;(}6;G9Bgm<%LG2C^pcXldytHo38`t*vo?pPin#<3$f*tNfnwk3Y7e-~S+3ioE+c+HvRuga?PvV1jzf-^4y=QfYWrdsq)'
            'v2&p~{cSiN=^|@Y=q(y}&i5o~(y4943`i*jp}Mb&h3e&)WB(@6EE%f~RurHoJ&^6XvO*o^aA;_iRF%~(rDgq=lG}P%&d|533s>e4'
            ')8!h;)i~F3M}8DZyVJr?G4x%>&~pJusWcn{#-m|S0;+$+BxxCgc(bM~hyiCjZMAeZ0M6K2s1GR-+M$S2&VZ>GYw&8#UCHL&s4Iyl'
            'eaDPINuCPG6d)OolW=5(fz!)Iv0&rV0ihc*sxm^VdQo;g?A-qC3`nhi&`Z7A#k073{rH=4lQnh7AOY<N28DO|hoM1nQH8c$zCqF@'
            'Y!Hq3lCbbS|6ULZiPaK=M`7e$CRCGMnHif=jMurVM%k5#XH|}>x_Wj+RP15CAWWP~mLbX_31YF#y?@9CxpP&)hzjPIRcdm3s5C5j'
            'WPH<~qq3PFvRtpjze)xye8^aH3)@DQ8@ZUK4UUvhwlsWhk5mY1r-D~?TVzl9kM2cfW4Pu5=B3Bh6$6n)KMcr1D@VYV_Ua5orM4CI'
            'iD`L?DN!Q}Jso-cxLtWC*^aNb(kjDHhD^cJMGuDtnQ`XXMcqmWQna0)8vz2;qZBA0@3&><8aCAs=g~N#-gOGQVN6;You@%CL2gW&'
            'Lfy2_wZbGhSUHV)ag4Uvzw;pb$>Z`nF^mWGB=Td@rV0d`qVeliFM#N$<F;3T21%n3Kb8-N#N4i}VZOEMYdn=Cui<bJ5Hz8@KR^4a'
            '|6_m9JHo$yJU%;ocMQ4%mqR%054xxumckP>Y^}9)9*ikHIyI@yJy{CdC^$L|Z%ngl`TkC$JSWE3qV{vT241=FkU?P4Dxk1Nwsfl)'
            '=@x@=Au^D5Z9d-Q>}s7nS)^(I{1KPQ<(Y@P#v$J%;a&}E%B~J|vFJH)MtlPto6Dc!l_{kQ#K%~&a8G3M(p$=ZXYjAaGQKQni}x};'
            'T0EVGV_-!<$?3-Vg}7NkLriaCj+oxYtNDaXl4u(YvsB#;iD#GuPWdk2nkX}~K<Ax6_!C_UL||@plriE(n5imd3$jv@kVBH$J&d2l'
            'X!MK|IUsyaPsyi*dy;@m{RK!AnA-#9jvfWkc)proY@Q{yJRLJ{+9G&ASaic1q{qAzLjgN-yyrew71eCfkwppo)|<TKKnWW;JB<#='
            'rB%w{Q0E30Y8dps5xjI+U{3B~_kADrXaGE*2CyF_So|GKS`@sdw&oWNgY`FngmZLqq5S_`z8r}p8qHNG7m*Se&NrM$6b#Di;Q-^('
            'W6l-|;wBy%L=e&$=>fSGu?^97Fd}Uf8k9k=2jB<v%?HOG7BmX_S8r*7c9<$IZ33IWJiwRB!@=cw?+9sYzB9^hLN%+)Q_`a^!Uf1y'
            '>!oyq^0DRt7n<GU<YKiwd0r03pj*$>5_0-uatAr_x98}12Gf@IgeRw7V<mr3yYQ$C_7_y=B>y@P1K8i$9Pn=)Cv8K{(HjKRk<Y``'
            'gG#f;)lw+Q&EvIN5fRQJG~1m0qd_k%Nr>Mo214$Z9%UCrh7QE7AvJ6dQ-OYHnoj0T_RjRCQ4@5jH0Tf|8~e-7pYTT#-4v=knZ=y$'
            'QK2eUKGJ2Mpxf&_Lp|Q?z!MVrS-j!7v5VP1F#Re_hBEl`?bHw^F*I*=jagS4VyMszT2MhKl(kv%E+86FHqI)uG3;H3{ufAK0!S~@'
            '-xtS(2{Dkwx1#Y}eEY4i;Oe^ge82s@_znka^GKMjT|C!Em&g07FruW@1~lkhp@t1SQTOSj^9HrIt>lc3+Xw=|c#a;WlBM$>-qfd*'
            'x$0p-AFpTdTIEv`!k4?!{c?A@U*}TyX)h<8tF1maojJqY*%vVr*tQz#d43KhG}Ou4Fk;z7SdR9o!69`4B6Z`LaUvUrl<rEN@<RJm'
            '=dNTGrTU&5Whu%HjA9pCVTf`d<(aPLvY4VY=twbfD9=QLk3iEfk|G<RNuY&5@s%M?8I4&ij=_Oh8I@;ou-9T}Y<a`%Z18CumrCeO'
            'hOq5xW+yzO$|A8tE=g7>U3`W@Eu19vw=5MZ%fPeL@gvs9#Uegogbr7DkItpF!<5rS;zf2UGgXcN`><>>U^Q9YRT5;Gdosw-G1jn`'
            'p`}BSsg~%<b0wrjA0&eQ;o$gJanbvaOU_mXf1t>4-UC75^g;<_RLVfeNRM1t>WaC~mA_Gr4XN%puAVptR^%f>&%#%57se~b1yBiA'
            'VT5u&M@lpeuP{zGPp(Rwxsfe-Q$M02MoMyK5iaxR@l9JV4~x!YZa&;QlDrFL3l(Q#PK7Lr`BxK%i>#96`H)9?NDxD@&H<nD*K*OM'
            'l1?fjAyl5e%Cn`jYu0}F<^Hq1m-{cY6G%?%siumwfoV^wd!QWwp0?Lm1!|d^1HIJVBz046>e~dXe7nB0V@V(%^ZO4U^q2~$#HX9Z'
            'TJ#bHPR)w+LX%Kqn(Abv`f#b5Rq%~%%5-0JPkOe)r-ZIzL;1I4@oi<+fe%~^&fZ_NWj<pvSO)Tt?CDb60GZJmgXLxad^|dutQ+ch'
            '7MXM|1C-xoEZ=R)6%kKRADjB4gz6C&Ds#bif+X{z*sIRxD9tE}<DhDyej-ZfjK@1UkJeXP)wf}6UNzi|)hBey86!LNPF`wGIj)tQ'
            '_3~GpN;fk*sg72YI_zhW%Om=iA(W^eO(lQuKQ<?;_Ajb@+N_X1I^{=&@juB;*ofwn@*(m7%j3W1j!dB-UL`REox%-TIkj2EwV3je'
            'FI{vPt<;rAX`tmCZ)r9TQ{OqiWUK0|-d1#1E7u!PaIkTBC`zYD4a#s(JM5wRDZFO8R>k|8G;0nY+6n+0f<yW36ItGjSW-UMqX2WJ'
            'Ki!T?xt5wCACAUrl=v`PU3<Us8j}gprmM>0)>yBlA=jLpcAmGU^tfYPv^|Y6<|Q?~F%8amOs>=_$n*d@>x%4s94ZU&t!ZwCqdmQ}'
            '!nJJS!cUqE=(vS=@%LIheaTvym$^~be5)Q!8impVOmUmyMMbi=lnSDjGT>m5_W2#Ul#`r{HxBZVe4+(tTbDG|NUIhZFE_vNJK?PK'
            '7&QPbe)s+un1fw7$k0fdd(ywS=%2pbW?#1{B>Gi1f^Oa7G$h9a%gYJ`-38gW8-&&F6B$r*ZzkPNd7*p>zmzaf%vHSXV3?H2?X~t&'
            'Df=W=u_aOE%o5Mabhny>I1~{uUwf)@d{Z`ncWUZ%K{cPJ9-Os`Jg9Y^K;<A%J7lOj4789B$&Lnf$v?Fs{9KM)6RVo*nBp7KhX9Zx'
            'eRsHHe_NiGYHo2|m)rJ<UV801uT8;Uh{;RA&|wAV=VEyEYrR9(V`Xe(MZXYPZ@rY^$42<<Cl{G9=1_{{W(zSUxKQ?m0s$=<8Y&{O'
            'S$*i#UN9GHbL&l|_cb{d$!vW=Bb{?YHebtL<q_FRVHO==tA=UJz(Ak;Elc`Ee~IF3i`m0UlwbWPcZzC#gwu724wt9A2vUk9`()y6'
            'pJ6*ta;M%p*|Z}7)zxMm1Zao<eZGvZ>}4I>ZdM-JIA;cC*dh0Pg<Y45^D`*+m!;g*6r3QL6K%QEMVg)eQ9bUXyK&V#Q%buc=en`E'
            'DVm@iD!BKQJE60&V?jv4A6tCL6iyoTV0(M{bMN%H|4x9=bJRc3Mzk!r31P$u5S1)!i?^%0Aex6W3j2W%;l-$*F8moso;$T4m@=65'
            'QdNjT9FdsOKkpx(^p5&yftXNairl2f>ajL&MYVpinZ;}x?{men%JeL`h*ovb^5iSs%fV00S{c;+pCh>WOi8u!|Hq@ms@V*KQ)Xfk'
            '-y|!_&2Z~umH|vVLdI10pp<$<_N+fejy^sq#=T%xQw(4bp!fMREELn;cmz!+$=hJQWL~=~85d}byY8dtuF=gk{GBIJ<Do9|a-Ej~'
            'bN=r?#23BupL^%(7!Ud<y|c>!mq96uvVW<$%d4D9xqGENiT_pI+?TpHS!VMJiQ*^$H_A)f*MQA0{_(C6{=XWQWhJOq=&<up{SWj$'
            '4SN04pS#EXBM&g|{pEn5SD`5VMLavG`BJHEEv)p74JeV@+>cPzT{^zyby%8Lvsw|;Y^0KO({;~IA$OR(FSOG>OToW3p+(907xWT$'
            '=aYs1O2J<iF#SBUy|scuwvX)ufA{~m(WRz(P1$UliwpA4tz`wLjI`LJ^su65>1gBCN9nk4vhHlD5U6I-lz&ZyEl_K&Y-JhtiGBI^'
            'l?^L0E85viEP<wYQ^D+&3=eg9fPC^<<!JV;1}#V$Aki_Y<ehK4D^nhQH{TIgzI0;<qfPjEb<lctt?c#w72?|U^-i5MP`qIwvE;Ri'
            'yr~Ife(<k$X)kxLA3-qk$^N#RBQD5IPCi$)gtDGZR~B(J4rU?a26}yJ%RZXxot>xTTUg%Cb~38t?0gOR>PyH~HS*LaQ=#9lx!B~V'
            '%xnx|WHY8hCgU0k^_djuQ>oEo`AXsUY;%pCRECaud0elCZKFm_oouzRuC-+0%G6InOeWEIFzC~62}5?TL3aV=CRhYk{Gl@rXIUsN'
            'e?7W9zdSza9`)Ofkp%1aAaenUzj!j9+_kGEy-%dsl;YROx9bBtLrLe7i{451bkIM%_$tYH0uEK<x_+FGi~Q9nrp(Buf%TMuq@{+h'
            '*1unzoyrtmdw!DmYx16orxeblg4?7USQ;;=P(~{56Q4as!i>pkoKi%$FNVV*&O%JFVyxR1=V+%+MWrvCs<0?atgt_exLKj^nBfde'
            'm%VZ|Iz^-WyPTv^8pf->*KkI0$`x6q$F>YlDvrjtbyV{fJ<;3jxZOw`SY6|X16-i{!*A)gjrMmxxbO@J8z+l`wio)6ng%E(n4+g`'
            '^dw(4p1IZ+RpBXgKyHM@wL(%{X>L<E)D7nW1HX*FuN;@m_94bgw4S+nmNWQ6JKylY*kDUQ5s)&U(PK1^Z&rAdEuj{$pznX+i6uOf'
            'M?Ew7;2Nj)ofJ&a(;d3~rBkewwQ(B8=V$)Xj~4zSOx51m*V=U8jWVt{lPmrfeE8bBv#!6Z*fg6dPsd*;WEYZR0SFy10xX_GM2+w+'
            'eUqYGJJSYBlg&Izoinm^8UVWSZ7`nUPuEDx#~lsoBD!w|7y?C65qpx-<gcYm=Y{26${uqxdAtY2G|i^7DZM@r!v_bWkjXd3aUO>('
            '`ld4ZoeU9y{;>A>9Fw%o@tlew&wI$#)>FKk#B&o*sbeDob+U<a*Mz*+Bm`Xc2#n`0d<E!GDbG?NVVi<mAvxgY-BM|cc;!)$!vBR$'
            'f%TEmnvD@EYC3Jt!-;y{qeLz%ZC+uwtk4r~Bm+)(w5s@$x;e^3u)?2XLabRDV&pZA`G@4u2rIe6_!O;<Tv3Z_a~|BlNUmWP{H2$i'
            'R*7~Ir%TTi`Efsc&OlMiGe+nYHXgyln6jZ9b3B73B3d*`2vcA0xAMMm^9uY3&Mw}`&d~Kao{QZFobKZ;S%L-dr0R2B$lX%PbpL4|'
            ')dCcJiau|L<V(#{M*0wIm*<>rd7maACJm}{UoTP63ZhjdU6e0{^%HkH%UWRtYWfxk)MUPb1lLWn5M7(dK8DMaNfgDi{jZcj@z#I-'
            '{VN8S&Wx$^4Lp^}5~g-L4@Yetr7^-V>D%DLB)kD-qTz1LbJ?Jnq0u@EQunhGUSj^dB*P*uDVfHqlDBP0sYx;cRZtHas($)T{n`jP'
            '*5WJxVrlUPRkpSO$c^D^h0*v+6h1;uHC*Z1Nk(w6fpFXcIh+S}VgyPufb=Q^wXDGjWDjIZstMnC_5JrRUcrgOZLmoIPeX<8(MWcb'
            '7r!sBPKJWZ`yPp$4dr%=YK%&+@@-RCp<qk$JIpxg`=+DE=1Lh}T~+Z$`kkvLPlkib<_zh|0a>lwzPgT(VsNgDRt~FEeR%0=gwfT&'
            'bnSTbjbNa#Y43Ig1kUr9sIOw45;OQrMrE40lI6!YMKk#&nu6%W0XeKbw^WpS9@@hpIG}o=6<<d<DJ6Q3Fkp}D1bc|uduT#z$y_$l'
            'MGaXxN5*q_52Z53-2AKTHcrBiEI8f*=AxzBH#}xw9+&6GO;+8|&neJcl?goyaTKJJr{!U<EEFB0o0KEg{W%_twYW#0QW?I*QBakh'
            'J4074oGJ-SR}uUFa}@v)VBx)8-Wg2p0v|)G!h{2<J^Iq_YYNnkd1+*UVo{amsshM{<R|DY!)1WEirQ+o<qzYexx+&8ALZsHgEKlR'
            'N&`i!l6Q#C+a%Ab={l_3|7On9Y>^zUHVlMDN@>1vji_#Grkod40#BB-wm`QrN2qP<s-#NB*6fw{MF8{2f<^H0HEacDPXKkPlynLZ'
            'xs~}N5Q#zoI@Ds~i$?9<xG0dJYjw$CTF=M|jxY?5Wl-vu49xm4iE>wdV}>1oV!4dNC?mt7qQl1f{9#>->OhW~n2hq2J~u3*ZY0%g'
            '+X#)iaXv9rqh_pf(~9vbB0DM$P_dD_A@YM1kn$LIGuPc18?vFhiX^CtyeORIO#v9@-xSPVLpADyGJp;E3=`)dFM^8|JgjoG)zv%-'
            'ktF4>o9~+#Y2t2*dMWoWCTZFxCZ;?}O;dM^$eE`H<Cf-FmWU+<Bit-Q_P3g_>CSvoj<l-`Gb(?>62_<woBg0TtWD|78QNcPjj=Y)'
            '(JjP06$CVWu*$Ejqm%@oQ-<iC1l#VxXTmXV?VIyn&l~i6=iX6w(8V%S{llKouR8h>O<CH_X~UkYvNV(?brReJ30-k;ahgH*r<4Ba'
            'QUAQDs?w~3YzCA(l^KpF$eL6{lTwYUrOe!kU9zq#$8N!yTx@o!nYp?q%O7!-9(fn9ANu7mG+list!Q6DCcE!rO|BBPhWczJs>N4O'
            ';NJ&HD`%-FI>r$iG(q@Y06-_HR_L3k$StH|4?TN^KyGklLFI!m<-pl5r)0jG$YM*x<J%ys25XFU#>0{XD67L@<Ss*`B5BI7^n~ni'
            '%GZVhH?Y<>W=~1j*@nE6xK0hq?us?Qa1`IaSk413jQ3S65p>Eu6ggzlA<lS2nq_UWu&wfrZL~_&CLGJt>U64=w{O2?0DH50Ke<Vz'
            '_$mv0c514R@S<F!GPM1^yizwWzlMhKx=a#Qt{yx#I0yckX&~G_(N(7+(N$sisq;`C@A0?jx9EYt{8-GYgyAOD>E&v|sd<IPZ7%$6'
            'Z%qZVlpj@Xd+CWiZjKN7SF<uyk~1RFHnnTg$no7}Ctsn0YDfKON{Ds%ABhFZ^oTd%PZ(cq(`CV>rm-pM7K&~5Sc2?N)uS=nmIaNh'
            'z@3AtARTEkw~mfgf~?w+4ev6IctBNstS%!FCnK`6<EY=w(#ciz%T|#Z={C+5@q#@kP^XsXRwX?AR(9Mbhs-pLfc(RNEgn(uh@!!z'
            'bGpq!+O-afA*+QlMVtrYAiN7wA!7@H5G3%o^lPP?G0cKBiSJXfC(<z>8W0z(+7iQGAXTR7J{viM3LHZPCKin}fC(HQpZ(&UcYpCP'
            'tfK1;&O9yE6ms}g>{bC$FqN6Z>xAEOuPeGSGGXhTJn5zEhl%G=8(PC_iB`)4>)K5zzWf;=;i#$c9TK{HEU)ytYZslyq$KBBxMT1e'
            'r?}mY!F|51MG}U-qGXuW;`d^S+|PVOjU{>F-YqvlUYdUD9>_H<!U(W2y2);>O+e^k(p0$v>Km9IRr?IMSUdLR(=abVZ#$rmR`Z!B'
            'gY{(ozTW}%+wkVLbr;W9i@?m?f@z2p=zlC9&@ciDHCGHjK@Wl?z8Chapj(*DgFVioP1Z{?m1h2pvB6E_9CMDRdEiH@B~VeRV`^Vx'
            '+1{8Y-5>*hxI?eqIQm2sso$Ck7;m>ZcVXfK<p&OyER!Jny}JVs4$u1odT0-du0Z4#j(?`)G9c*VW^NL-fi6{eRZcx5^`@oXm{pIA'
            'vAL<W1edDiZ3h^Z7^uLo^xCnZ!s1Atxme{r`~DLlR+Si17P6)%5+sL;yIi6dl+0K9y#2kAN|N*d031pq6Pc>X7Z)eME=06ii~yFn'
            '2k}y7CQ~!lIqb14uD@m}CRw1GK?!Mn>=TYD-4<7ir^I41ZPUG!xkjsuoVeI8#5d}6(L{cYSA*3kz%!DBV-io~$_IEFQ_A&&=^#Uu'
            'nCOBFHeQ||r+ed&w=N~UM&2U_d|@%eOME#Utw=t_1U1wE8S<M*ogv3Wp%Nlg;*aTQs0a7Bt|Yrls@w0<+lFQKG=0I{x^e%xB0$xQ'
            'E8lKCu8qj-)owkqS}QQX`@29^fY14n^HyQ7pI}7XMfa$8*b}=WEWsPC04?xB>DC{SWn=;f5$@ux@Cq#=IvZ+Yxp!K)Gb4k_U9k2G'
            '2=oy?7ZC<w$Z90k@C%Eu^$t&Ze=j(M+D@IO=VQh0m@Wx;qUXAts&<1+4O}H2k}B`VVGhNs>vlf6;uR`8^&58o@zE)A8e<+bLg*WP'
            '+vkU3_*Qj4QN6_;UykWFhj>%`OE4M=bW2J_W0-%FlitHL?t@W#m4the-Ng$(jD}KjR9Wb+uJbId+JSRS*Nf=)W6&M7l#=h23AtK_'
            '1X+hbLK~Q65Ja+)6c(j&H|LPIoKZIBk=M*wr^YuE$d@<9DHU~7Zw@wB{;_WU<)?IQUh_-59o3}}V26x}VngT(4-175U=#CZ@EU6p'
            'Bw@K)0!|;>!k6$LOlb^YK6tvu52sZNqP9PoG@z?JvfQ){(L*?!1N>%HG!9}PytSs!4=X%>97fY(d->F@jM|YXCZ}0@(lx!@c+#<='
            '%n&{*voEV&L!&YSC;vq5n5HXqGpyoG65Y{3S&>Q*+`<LbkQ`C6!m&&hi75anBj>#jG=@GK<V9=c1o+nL3Z#<jEy{Xqe_(J3{KR^5'
            'ZI7yx!?;=iR5hXla`H!{=c8IwY5(O5g=^Hw+GW{5S%}48aG`{!BliF$q2Bv0Mw?1iTU$q^l#Ua?eIjQg@|jD7G<Lr2o%B!p9*pSy'
            'eAYkgU1*l){5@FSt`jQzuo}<s-@{u!AtBRwhc`s<@0%b>LTYmyOnm(J4EP=X+l>QVe$hDu)CrU1Hl|M9RTfA5vj-~l+@EllzP}7I'
            '{&~3=!HVb9dVUiIBVP@51X~R1DRl-e2#T_76hz^O3Q_o}4}XE0MAP!2#Uu!+Ll=-NiTp&3)tyt|627`X{PSrgZ~bwy`oQa+V~+$B'
            'EZIFw$X(#?z_X(OzquNPYP6p*?e!w`fs<433qYCi?T9~(f6PJK3g{CTtGPe*vBtH#qY^UfEx`B_g0J=JxO@8cvip|aW=U&s?(BSX'
            'Adcekip}6ka7Nr<yWxR&{!+|-`q&iDUbRLcS!sd0p~dC>;fZL#?=hEw3KHOu_>qh5$$Kn$iKx_p?%w|XzWL&p{?Xtkd_xZpf9jr|'
            '_Kp!{o)sp8vmoQ)2W1fA4;-GI4tjrs6`j57opNk6^nUT2J|s==ii!~~d*qX67nsRbfZkX1qG`B3$>aB5sq9Y}p%z!F`@@gD0xvfG'
            '6RE+LOw4Be+!fhJd33BgP|-3kMo^keStuyGH8S_N-5M+AhdS-6m+jH3my>|}9PxQoIwB%@yzSJaN%P)$G7tiBj$$$Z-+TpnCWv<='
            'I?M13MziI3;bpOR@83xjg9isfA(f95)I%eAr*JONl8p`^^R#20ff{c*InaGL$!<HvgTx;G_(jc$e@Xmhs*fn3pfAq*Z{F<v(*3!Y'
            'pH?YY8z)|d2T2Duj5@#pxTv!jwqcb+OnM`Wx{Y81P#Ry({WaeEvZIhpFiqhL!4kCX{Em&CL)qAagd+oS0aT=+iyr{h7)?~DY)j`l'
            'S`(<v(<xO&z|7Mzm5!k~*upfnB|uXKD-Cw>t%fuFG{Kk`gYx`&c!Lu4rj`s5#6Jfi>4@*-I@(JRLmJLs`aqChyrQDt5h4Kv<cu7I'
            '_;is8?2uRLEOh<pr#}3@gA}%jm#AtY76vu*HCAJZv6!Z{09enTT<iNIKGv0s(DSz(hk5zmA-)%5ZkWJoXJNMdgb|;I&a?gJFVV?!'
            '$zDMljzonqNMY(8p7ilv8h!+2Y{gc_y(o@aXj6v<fTv{nl@=ZLlmmPN=u0GjPZ2gGzTp0;1%+7}%%@F>e^C%jsJ6{?$_W6`dcsI4'
            'MO3=5knwDkem%?zh)5aIJMg{imVKvXQ8~0rQ{|Z80|=w&8h~%xVakw04xsM{a_0kmR*}NgWgT(|Iv$du^w}%Y7*u})+@2Bq3op=5'
            'v_6rMqoC|@p!%I2^ikg2!nEeRAY2EzNhQ2sH9GA-WrH+^V;`o<D{XVKM%Dv)7V;+`gQVUPQ;bl1`Qp)Z%x7?4DOg{ZbZVS+np%Dy'
            '^6KC~cE(BbBGF}we^uyQc|}n~uSuM)F}+bdU!@etE}_M&i#8lxHzliZO;BE$KvRhba^9v^q;bS*BK|5~zO>q@{Vg=><R>#Pdt|(Q'
            '`I2xR|GqkSd5st5is0$ze>!;fT)si-W?ZGKkK0&#EnU5HXBH+H<mgsQ>^%BE+-ojshg516NjjwoCQa!1{=th^*SQQp?o0@Pik^Md'
            '>E-^x%a_;g)&8|nm4<dg`Pe)Z5Yn2SiI;i>^(m{$IEzn=fb|@7_E9F=p$RYq{4?4Nh7J7nsys~L+|wu~TkLcyNwQ_pG22tx9K{~h'
            '#|!Zo@`%Hy@(tJ%Tr#MZB{ygldwam4Y#~k`3yXB*QR(V&DXW44L79SKXFW@mlY{91#`U8g3iWMMz+PIoOfpCY|M(o5;S?$m%dc~}'
            'Sf(H{Q(79DQEPP$?cWL1137I6bH7<r+~pB~ZS=>2l_<h^=c?+$>2i))bNRv&I(k`_Hsqf=m|b4sfWEG8Z(f3mRW|)x;Ap8hJ3aoD'
            'jiTyqt}VLi2AHB3rq~Q?{ZR;O7?EQbTkqJVOP0!8EGS4J4OFca!^exXLfEci-B)w^s=%cg^eu@UhlKs5W2~)Ya>X{+I#i6dN7{|p'
            'CfQUMJ<<^C<dr;A6m>B!m%>@w1m{NLG=BI?=c@gOAFkcs((hyxyL(_)LysDAR9DK8YAg!w29(t^<^#!%3)WVvzN>7a@3HNsdI{<~'
            'JEjB7K}};!p~rtbV(DV;`REiuUHG)gC7zt2@;9QE<(tQ9RE<}erv=cvI2`MOS|!^^C8eJG(aj1yp~prot}5F}W$y@6IuN~?go&!s'
            'E$ewX#yKJ`KnpWZROOIGm!7RIOLJvToAy|D!h;vzN0qic6(Y1sW?_1Zb>dVB&c@IJOJ|piM!Q2-`Emh0#R@wFZ(XZX14C$FV15dz'
            'j&f3qrK$w!%cyw}CRn+o40k|8Ebp+1XlStlzYT0sNeqHnGBTW|$oYb$AE?XW?)&%WXFqq34^*&N(cFd88y13&hN$ZLD<A?RtPBN>'
            'cXUY`zHohi-ut=V`vn$Bckv-ty@LkfbS>+VD!@l-MbJ}vB6H~1P>1CZ?bd$ELpTmC_K;HPL}1P6cHuU;LT~QYN)W@X*kgM;TFlu='
            'AdGHdh|DhiL`|`n%WggyyysSQSV#SE_#4vfDON>mVyPnO(kzCyTlJ6?QLzLWSlxi{Xt9sxZEH;G+$ya}{q0(BK}97xv6ul$HTaKI'
            'rH?}~-;>$otO(r=ETKXtkRd`&mZdV9t0JvZUb~}QQ=vjZC`OY|Vica$YstxO*s7$8RaA{qQ+1^_ereFXc;}s;T@HE|SG1h#stcoH'
            'z!HsU7gwpMV<Jn5=sM|7NfMRBk9SE{$p(C!h98iFS5Q&4UnuLK8?M8`%!Y<6O*z}RE*sO#3n(!ylh-Y!n~e<-eNzYh+oMIW1_{WZ'
            'EFjonk^>s|56L44y|a()^2n|u=P-`v97H8=u1H^#cUPb6I4y^2qZ5`{>f5@7<;^VBEZ~dj&Qp%R<!thGp1}5oe<wx1Dg!mT30Eg;'
            'Hp<_|6QHFTCkDH6x|c`&Gv~Tm4RZi60s3Tc72gA~ArrL4^6*HRe{az8XoHO}z_AU`Gd%N3Z}<UvGr*tp$Kzv7*5~r$$4yBrjr$Hz'
            'p_Q$E<84UQ-GR~YV5g=eLcIZ<h#ashX)&L~5%A^7$|+U?QOGHULF1!U<y{yZ#W%1aX;$>PNM#~Zb;-K(8JRhh`AE%6<-i&_QpeJ7'
            'J|T^ZBT0Y)2EfKCt*+t=J731BTsmx3x|K^#_i29Rg+eGg-`yWL^txP!h<BCDgxAFiBX*lSyHZulQno~W>w?C$;zdj`!)QUAxCdcK'
            'n%f$>lWU5cY|E%JbK4S1%`(v*x~XAdt@)o;TOtkI0Dl$I^_gUfEPdt^t8E3pq#hN|5$QGLRe$M<qs)o?r@bt8B2j{X3~(bz7!i*a'
            '!!0L*l~68s-T?Qdl-Bc%v)bCIAueT8kFz<HJo0i@ohMzCBK>Q@xf9XPlO5;^%0XLP00J7$co?Y2S<AvC4fa450Ld+)V10fhWb#uA'
            '+K;ms4skdWH^CTPo<G7q0HrVoTnCe2h7?2!^+tEBAo25g_xPl1JMbxUk4CC1WxcxITqmqpIG{djr8%~GT<bi@@lil+@-z_mhn;Sv'
            'tkQq0ADZb+Dr$z*GUbXGSPp$FYtuS%U28?HF-R-0L{xCKD#TYz=B4RM>CfHyJsP&k>SIIVD5DqM@edi&!@XfijTknyBJKZw-Cb>O'
            '8#faE?q5NuA1oD;6Zeo@YeNN`T2726PGKj9GYmtbv|D+hH&?55qqzR>H*e&UORg;YQXFvk&?YOA%N@QAXP$Y6TciuT#>m`@caLAa'
            '#9zDVQpn0J-w1ViAG9}6|AK~GVdZdbVBBz6BH}DNC1H_mjyZO^_9!wzqTNQIXGRpi?$s`-eY@8!XA*dE-d*=DvJ++0=~llkwi`J&'
            'DE|0vl$->l#o=SM5_73Vs%^>&>sHu7oarX}l4LzR#C-vFWY{%8(c?WPLG-UWJ-awMga7{0d*A!eyEs*EdcVFuz5IAyO^?y|XvssK'
            'M-bwEb^N0vOboGw8F|*;2{}ltgT{$TFVEhCpbyTEZu~o|;8_x>!Kbdz!N4ag6b=T~KZj|2V`jsa*n%6K2D+Uhjhp+%BQJ1j#*X!{'
            'eyo&7iV-`x!)BtzoOZrLgc#oQ73jh95gF~6dkakfs}ZGRhDP^zZwU8WJI&lSVJ}1YI{aF|+|GB^Zdo3UNquTM&81(vYK%@M_S!=)'
            'Za{VptNeHV{%xDutCI%4A~xu8VTg_C>R|u%wb7&M1#K)z-oHKW4v<cg-r1bUeM=i!&6!(qus56WMc^o%)Ytom7XNJgR+Xo2tL}^$'
            'E%H^r$dkUggaj#WZ3%|_hWT;-C~ym3_bH&Wzu$iq776-Xx3=BSa{ItaYgo$w%RRCjSBtFXX$UmPNKYyye6kWcYV*E1X-g6cZzvx&'
            'QaY&KY?vkqQGtnZA3-zEVka;gRPVzZ%yoA5wtsy3p?7w1`tb}7?|;7jF;apuqbd%ehsD!LO?Ns!G<FcV89qmIF9K=?c*YYraB#>N'
            'V)(ZjFH(!FVapIR819tsBts?~c_*j;CQvc&`3*yv+0V0CHq}e!eQF3>pW*T?e^h@dRx1t4b~MGD@JOpx)+?U{#hFS1M>i34s{1+$'
            'LzA2KU&@q-)h1AOeO3CI>OF*BFDq82kIXGUp_tQ%i{5I;F$rzr>VuO%o#*7WVz<<W8M2vD5SKHa<fMdE$O50`(<(n)-DR3)3w5eP'
            'FyOOK%8-LGrqHO39Yd=l+Br<>@+))C$))X}QD7mupE*JheM+r8H3(<+KztgjMaL%2$-RTmyouCtSOYK~u;t-l<_5nKne#0u25EB0'
            '7HcOG!unlkS8A7y^1v&xqoNb&7OpEKadq+Tr~`+Q1S{!3SsS*br!+tbBRI)bKd1~Dc~L4HeFP&w12oON>j)!hE~Lk#1(FYfp89;L'
            '*vbaKOF7VgpZjIJm=&ORz=awa_U8oVZ9ksY{xQNh&*9>H$91b6^|OQxZ{c^5)G+KTso5jxFskkf6wPj`qwfp>FIUzIkESXdn-&w{'
            'BZUuhvd;1+I~<}^AtaEvyI_N>fP32BiFpRh4b83yE!=;w4K(e;)`QDt@(hEpv-NaqB7~g54g%uJ-=1JD_JpNu9)*|@{Vv6f@p&25'
            'WE?xpriAIvYkI&Cr}Cqy0(=cWkB1ZRZ%_0F*bYEvC(zlVV*hr)FWi0$4axo}KJAq?>5D8!{cy2{?eN|<{~$#5>fY%@B;TXoXA!;e'
            'TwF2E?T(iaA+Wfm>sA%xBQ@ze32ZGB{Ru9Gok(^juc-$$&X!5W32E=$w?4=?@N{a_IFbx2I9GbECt@=TYhKKnaj)!(r6>tc<de4Y'
            'XLu*bliV7vvC~csGIaWL(JNTpXQ_PkVO<T*5k`OYim)i~-YH1nE05H$Zl4aDAeugFvZrEBo)TY83k<&WnBPw|YfZaw(ZeXDQq64Y'
            'wi=RRM{DB1)c~~hsNr9*TiULJ;U==&M(`S?*KmO^zG8xe$x72kxtPv0Y?d;`F|Qb{`lakQ$%4RcFz>#o@70>5_@%9Aqx_oPa>uzS'
            '>S%rKl>Wk{DVU}O(+5uES7phelw8|(snMZVl4QWmcGXLk1lR4lN#I1taN>J(=Fs;SsNhlq-%C{!hBRN?rBid6E<?=vynx%*!rQl-'
            'k3T%1=k2Qw0yShj-Mf<>YgF}YBcM3m7DG^bZqnpMNwX~{mT24idoL-c<ch35(i+FEGczbLl_KtT&frd!6rcs8^1P>xQ&^g)aU+ol'
            'H8;-gQYUMs*?nWlOsd}+NOoT&J7gMr%&rqo;!uVQ<oe*HM}Kv*+*!GL%Y=Z+*gTp>DA&+F{sl$Tp|`g{JA7@%5L5ZsTo8RLjeNs!'
            'B8?B?u@+Of_|b5}_q^Qu>4y-Zdu-Z6V6r&+^k})ra|9u;#0F)^8DtO#=0#!78Kg%W;+=&1g!!i=c#NXG*(Rqnp|120rYARO`#V5='
            '3*Rd<gK}i~<7pc4Vz#}vx<gP7Gc5u{t!y!<(t8j0PSt<zI<oaOR_>OK>2FY%2qr>ZYj`#Zo&?k;j&(Ke9&@QRrA3T}Uw4I>YIdMH'
            'vn#i)Yco;APSGE(-EX6}PtlypQ?X`l{P-%I8*i6snylM2GwiYxswZZ)aZDpkMs0Vd8G^R$J31JcnUs`XQo{t_;~oF{k&&!^K`D0P'
            'H%g2Z!G|#MdkDm2URhNx!--&qU#eU+MyLQUvV_>xYKPwWu&c%lm5rs7m67@%hN4O&3c1aLx~=;5%PyhZqPkRbgdr64f{xlph#V-O'
            'FZhil90>-%!!6QoIQ`9^Fm<RM0xy)uPc~e+FL0F#j9kX3Ts=k<a4jYQA_mcRQEu1LO@HxRxpha5Odt5$-6?S`OULf2eR#Wu6SFBN'
            'rz{c1D#~x}viZlW$dUOQmoxSo#b9P6)ar0nCE8_qKcPfTsa|$$7wmy)mW*^ojwJug;Om8yXtg5eOp}ilQv@c4^C@!7uo?LX4w&&$'
            '-{ILVcI1=R0R;ptGs1+!vix{*;+n|nId0KHsmm5E2~^*0FQ5j4ZXr|j_PIoYKa}r$0d32zf$f%Cr#|^6XnXe)J+WGHGPu3K9xun~'
            'vG8O20Q>XLXMN0A`S$$iU#Fig`{%uj-qHCl?~GIi%=I>dnM}-KAaM9pPO?WBc9U~M?nKN4i=`R)ExMZbjNM{B0A1WqZZS#^qEQX-'
            'Q%?-6g|cmO)nU(8OmnYvzEegyyQw?LyFUd5?+BGXTnEtppgNKRH7t^Giq=YM2W5Tq>ErpOgy{-?A<w4Fv<B-aN3ugyJQBU$%>@A`'
            'Qa@eg4z^zq6$5wZ2C#7kppB?*X{K#p!>lB(6@zAMUDTU4jq52wCtg}T-F`QsrAlLGvr#v<7z3l4cY2xYHJpm`)&V7sUFXQsv(yM+'
            'nrZb|A{9*eD_|#tuD8`qHX3G22DO)v_Pr|o(0>PLkBYU5aDOJcz0yc^Zel*=%A*K-JCR{kiYZcZR?X^j3PpRlb)J~5wp+IQ=utLN'
            'I2u7$hQ(@>=0UO;NjI1jOQg}wL5%=ae^{)G39l%%Dkhk^0!KF2lU9%=M%HaN{-KuJej|(c&s}Nef4Im5o%7i8`n_SimimU^1S}<7'
            'Q6Svv_jm94yB;22RWd%%qRzARVtM#~y`Weu?lSroK8MXw3Xeo6)HodiHg(<(n)_`t^*ftqcf=v_41o9MS8QP?b#M*w#Y3i3Hzley'
            'p>9oz9@tnT3CDxUaIMVhujEWOnuHO*XcAPRycEV@>D@Y5x(j>tPSq{Iua!;txOdjOyy(4G+SLIysD8y^h!_5hD7_h~P%zSnEjFo5'
            '+SLu25W=t>7c=JmWMJ_u{b~=|JYB=Cu+A{!ss0*l$n{uUci_t;hz=gNq5y9X?et-twG(Uu&eJ=}^ca2kjR3i6(48S{6X~=JLbFC)'
            '^1+sm0v3Jz;Z=b9^g!oju^=6sKU>R!(RJV@M5`94_GsG`^7m(pIa#J@2h$2M?TO@192a_-m^?8*efzwx#O#e|hYyG!{d|j=$=@It'
            'I!&7A#T>blNt=>W1HtgJ91s@2T=h%rsYY&Ds7C)UpYv4}jbxVl+=6W;dOg<@&Op$hg;O%Q!VTg){k<Uep#?_TCq&FOTHLOe+4u(U'
            'OVmF8O3}R_KSlBbT@0Cx@F7%RN?LWkH6azk?TSLE<(l&zsfpCd$?P^ABkzVr!$;@On+y)iB|P|TeSwJ-j>-B9|DI;U<syfjJVky`'
            '2`pabyz%O}p00C3i@^FjWVA?qmWDQwQ$yL=E#g4%Jue8Ew`1+GLRnxL^(@=$c!6Kec+BXXt2z!9O*ua1JHB^xQ$h&73mvF?XJj8#'
            'F@mK68nBw^)eOW@-I^~y|N6dnhQ!Il?G;U$g(0izvY23`{Y2s34Z2*8B&K@3)Z;+^MumN(L2pO=_BdUplWC@I^kfPbGSDh-vRk#7'
            'CdC*JgH^hQt8lvVwtTV3*r^QX?mU~1n|=ac*#jIc;Pd6t`RT#Nv>Aj7+EefJjHmevhh^Q$$_ZLwv1M}{U_|d`1iE=lSZ@lrx<Fx&'
            '4Oc_KIjz#YW}{>h^d~{D;NLbP=0BPWT9N'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
script_generation = _local_pipeline_modules['script_generation']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']
google_ai = _local_pipeline_modules['google_ai']



## `local_pipeline` kanonik modüllerini yükle

In [ ]:
HOT_MOMENTS_PATH = ''  # elle yol vermek istersen buraya yaz

CANDIDATE_HOT_MOMENTS_PATHS = [
    HOT_MOMENTS_PATH,
    str(RESULTS_DIR / 'hot_moments.json'),
    'results/hot_moments.json',
    '../results/hot_moments.json',
    '/content/results/hot_moments.json',
]

def find_hot_moments_path(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None

hot_moments_path = find_hot_moments_path(CANDIDATE_HOT_MOMENTS_PATHS)
if hot_moments_path is None:
    raise FileNotFoundError(
        'hot_moments.json bulunamadi. Aranan yollar: '
        + ', '.join(p for p in CANDIDATE_HOT_MOMENTS_PATHS if p)
        + ". Once Notebook 01'i calistir veya HOT_MOMENTS_PATH degiskenini elle ayarla."
    )

with open(hot_moments_path, 'r', encoding='utf-8') as f:
    hot_moments_data = json.load(f)

# Kanonik sema dogrulamasi (Notebook 06 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(
        f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors)
    )

print('Yuklendi:', hot_moments_path)
print('Uretim yontemi:', hot_moments_data.get('method'))
print('Kaynak durum:', hot_moments_data.get('status'), '-', hot_moments_data.get('status_reason', ''))
print(f"{len(hot_moments_data['hot_moments'])} aday an bulundu.")
if hot_moments_data.get('transcript_available') is False:
    print("UYARI: Notebook 01 bos transcript bildirdi (konusma tespit edilemedi); "
          "uretilecek prompt'ta diyalog metni olmayacak.")


## 1. Notebook 01 çıktısını yükle: `hot_moments.json`

Aşağıdaki sırayla aranır: `HOT_MOMENTS_PATH` (elle ayarlanmışsa) → `results/hot_moments.json` → `../results/hot_moments.json` → `/content/results/hot_moments.json`. Bulunamazsa veya şema geçersizse notebook **açık bir hata** ile durur (sessizce örnek veriye düşmez).

In [ ]:
HOT_MOMENTS_PATH = ''  # elle yol vermek istersen buraya yaz

CANDIDATE_HOT_MOMENTS_PATHS = [
    HOT_MOMENTS_PATH,
    str(RESULTS_DIR / 'hot_moments.json'),
    'results/hot_moments.json',
    '../results/hot_moments.json',
    '/content/results/hot_moments.json',
]

def find_hot_moments_path(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None

hot_moments_path = find_hot_moments_path(CANDIDATE_HOT_MOMENTS_PATHS)
if hot_moments_path is None:
    raise FileNotFoundError(
        'hot_moments.json bulunamadi. Aranan yollar: '
        + ', '.join(p for p in CANDIDATE_HOT_MOMENTS_PATHS if p)
        + ". Once Notebook 01'i calistir veya HOT_MOMENTS_PATH degiskenini elle ayarla."
    )

with open(hot_moments_path, 'r', encoding='utf-8') as f:
    hot_moments_data = json.load(f)

# Kanonik sema dogrulamasi (Notebook 06 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(
        f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors)
    )

print('Yuklendi:', hot_moments_path)
print('Uretim yontemi:', hot_moments_data.get('method'))
print('Kaynak durum:', hot_moments_data.get('status'), '-', hot_moments_data.get('status_reason', ''))
print(f"{len(hot_moments_data['hot_moments'])} aday an bulundu.")
if hot_moments_data.get('transcript_available') is False:
    print("UYARI: Notebook 01 bos transcript bildirdi (konusma tespit edilemedi); "
          "uretilecek prompt'ta diyalog metni olmayacak.")


## 2. Aday anları görüntüle

In [ ]:
moments_view = pd.DataFrame([
    {
        'rank': m.get('rank'),
        'source_start_sec': m.get('source_start_sec', m.get('aligned_start_sec')),
        'source_end_sec': m.get('source_end_sec', m.get('aligned_end_sec')),
        'score': m.get('score'),
        'reason': m.get('reason'),
        'transcript': (m.get('transcript') or '')[:200],
        'preview_path': m.get('preview_path'),
        'review_status': hot_moments_data.get('status', 'review_required'),
    }
    for m in hot_moments_data['hot_moments']
]).sort_values('rank')
display(moments_view)
print('Kalite politikasi hatirlatma: kaynak durum ->', hot_moments_data.get('status'))
print('Bu durum insan veya LLM onayi olmadan otomatik olarak \'approved\' sayilmaz.')

## 3. An seçimi

`SELECTED_MOMENT_RANK = None` bırakılırsa en yüksek skorlu (rank=1) an otomatik seçilir. İstersen belirli bir rank ver.

In [ ]:
SELECTED_MOMENT_RANK = None  # None = en yuksek skorlu an; istersen ornek: 2

moments_by_rank = {m['rank']: m for m in hot_moments_data['hot_moments']}
if SELECTED_MOMENT_RANK is None:
    SELECTED_MOMENT = min(hot_moments_data['hot_moments'], key=lambda m: m['rank'])
else:
    if SELECTED_MOMENT_RANK not in moments_by_rank:
        raise ValueError(f'SELECTED_MOMENT_RANK={SELECTED_MOMENT_RANK} hot_moments icinde bulunamadi. Mevcut ranklar: {sorted(moments_by_rank)}')
    SELECTED_MOMENT = moments_by_rank[SELECTED_MOMENT_RANK]

print('Secilen an (rank', SELECTED_MOMENT['rank'], '):')
print(' Zaman araligi:', SELECTED_MOMENT.get('source_start_sec', SELECTED_MOMENT.get('aligned_start_sec')), '-', SELECTED_MOMENT.get('source_end_sec', SELECTED_MOMENT.get('aligned_end_sec')), 'sn')
print(' Skor:', SELECTED_MOMENT.get('score'))
print(' Gerekce:', SELECTED_MOMENT.get('reason'))
print(' Transcript:', (SELECTED_MOMENT.get('transcript') or '')[:300])

## 4. İnsan onay kapısı (elle doldurulacak)

**Kalite politikası:** `hot_moments.json` durumu (`review_required`) tek başına yeterli değildir; script üretimi API çağrıları yalnızca `MOMENT_APPROVED = True` olduğunda çalışır. Bu hücreyi izleme/önizleme klibini kontrol ettikten sonra düzenle.

In [ ]:
CONTENT_MODE = 'spoiler-free-recap'  # 'spoiler-free-recap', 'dramatic-retelling', 'comedic-commentary'
TONE = 'gergin ve merak uyandirici'
TARGET_DURATION_SEC = 60

# Kanonik prompt olusturma (master notebook ile ayni fonksiyon).
USER_PROMPT_TEXT = script_generation.build_user_prompt(
    SELECTED_MOMENT, content_mode=CONTENT_MODE, tone=TONE, target_duration_sec=TARGET_DURATION_SEC,
)
print(USER_PROMPT_TEXT)


## 5. Kullanıcı promptunu oluştur

In [ ]:
scripts = {}
script_meta = {}
attempts = []  # her zaman tanimli olsun (disa aktarim hucresi bunu okur)

if not MOMENT_APPROVED:
    print('MOMENT_APPROVED=False oldugu icin API tabanli script uretimi ATLANDI. '
          'Asagidaki disa aktarim hucresi yine de prompt paketini kaydedecek.')
elif not (OPENAI_API_KEY or ANTHROPIC_API_KEY or GOOGLE_API_KEY):
    print('Hicbir API anahtari yok. Sahte/ornek script uretilmeyecek; sadece prompt paketi disa aktarilacak.')
else:
    # Bu notebook'un katkisi: AYNI kanonik call_*_script() fonksiyonlarini her
    # saglayicinin BIRDEN FAZLA modeliyle calistirip karsilastirmak.
    attempts = []
    if OPENAI_API_KEY:
        from openai import OpenAI
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        for model_name, price_in, price_out in script_generation.OPENAI_SCRIPT_MODELS:
            attempts.append(script_generation.call_openai_script(
                openai_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('OpenAI key yok, atlandi.')

    if ANTHROPIC_API_KEY:
        import anthropic
        anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        for model_name, price_in, price_out in script_generation.ANTHROPIC_SCRIPT_MODELS:
            attempts.append(script_generation.call_anthropic_script(
                anthropic_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('Anthropic key yok, atlandi.')

    if GOOGLE_API_KEY:
        # Karsilastirma hucresi bile kanonik router'i kullanir: istemci TEK bir
        # kimlik bilgisinden kurulur ve call_google_script() her modeli
        # local_pipeline/google_ai.py uzerinden cagirir (kota defteri, hata
        # siniflandirma ve sanitize edilmis hatalar dahil).
        google_client = google_ai.build_google_client(
            google_ai.resolve_secret('google_api', get_secret))
        for model_name, price_in, price_out in script_generation.GOOGLE_SCRIPT_MODELS:
            attempts.append(script_generation.call_google_script(
                google_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('Google key yok, atlandi.')

    for attempt in attempts:
        name = attempt['candidate']
        if 'script' in attempt:
            scripts[name] = attempt['script']
            script_meta[name] = {
                'provider': attempt['provider'], 'model': attempt['model'],
                'cost_usd': attempt['cost_usd'], 'latency_sec': attempt['latency_sec'],
            }
            log_result('script-generation', name, None, attempt['cost_usd'], attempt['latency_sec'],
                       f"{len(attempt['script'])} karakter", status='review_required')
        else:
            log_result('script-generation', name, None, 0.0, attempt.get('latency_sec', 0.0),
                       attempt['error'], status='failed')

print(f'{len(scripts)} script uretildi.')


## 6. Aday LLM'ler — Script Üretimi (sadece an onaylandıysa çalışır)

Yapılandırılmış JSON çıktı (OpenAI için `response_format=json_object`) ve sağlam JSON ayrıştırma kullanılır; ayrıştırma başarısız olursa aday atlanır ve neden loglanır (sessizce yanlış veri üretilmez).

In [ ]:
scripts = {}
script_meta = {}

if not MOMENT_APPROVED:
    print('MOMENT_APPROVED=False oldugu icin API tabanli script uretimi ATLANDI. '
          'Asagidaki disa aktarim hucresi yine de prompt paketini kaydedecek.')
elif not (OPENAI_API_KEY or ANTHROPIC_API_KEY or GOOGLE_API_KEY):
    print('Hicbir API anahtari yok. Sahte/ornek script uretilmeyecek; sadece prompt paketi disa aktarilacak.')
else:
    # Bu notebook'un katkisi: AYNI kanonik call_*_script() fonksiyonlarini her
    # saglayicinin BIRDEN FAZLA modeliyle calistirip karsilastirmak.
    attempts = []
    if OPENAI_API_KEY:
        from openai import OpenAI
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        for model_name, price_in, price_out in script_generation.OPENAI_SCRIPT_MODELS:
            attempts.append(script_generation.call_openai_script(
                openai_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('OpenAI key yok, atlandi.')

    if ANTHROPIC_API_KEY:
        import anthropic
        anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        for model_name, price_in, price_out in script_generation.ANTHROPIC_SCRIPT_MODELS:
            attempts.append(script_generation.call_anthropic_script(
                anthropic_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('Anthropic key yok, atlandi.')

    if GOOGLE_API_KEY:
        # Karsilastirma hucresi bile kanonik router'i kullanir: istemci TEK bir
        # kimlik bilgisinden kurulur ve call_google_script() her modeli
        # local_pipeline/google_ai.py uzerinden cagirir (kota defteri, hata
        # siniflandirma ve sanitize edilmis hatalar dahil).
        google_client = google_ai.build_google_client(
            google_ai.resolve_secret('google_api', get_secret))
        for model_name, price_in, price_out in script_generation.GOOGLE_SCRIPT_MODELS:
            attempts.append(script_generation.call_google_script(
                google_client, model_name, price_in, price_out, USER_PROMPT_TEXT))
    else:
        print('Google key yok, atlandi.')

    for attempt in attempts:
        name = attempt['candidate']
        if 'script' in attempt:
            scripts[name] = attempt['script']
            script_meta[name] = {
                'provider': attempt['provider'], 'model': attempt['model'],
                'cost_usd': attempt['cost_usd'], 'latency_sec': attempt['latency_sec'],
            }
            log_result('script-generation', name, None, attempt['cost_usd'], attempt['latency_sec'],
                       f"{len(attempt['script'])} karakter", status='review_required')
        else:
            log_result('script-generation', name, None, 0.0, attempt.get('latency_sec', 0.0),
                       attempt['error'], status='failed')

print(f'{len(scripts)} script uretildi.')


## 8. Final script seçimi

`SELECTED_SCRIPT_CANDIDATE = None` bırakılırsa en yüksek `quality_score_1_5` puanlı aday (yoksa ilk
üretilen) otomatik seçilir.

**Onay burada verilmez.** Dışa aktarım her zaman `review_required` yazar; onay, bölüm 10'daki ayrı
onay hücresinde, scripti okuyup SHA-256'sını doğruladıktan sonra verilir.

In [ ]:
SELECTED_SCRIPT_CANDIDATE = None  # None = otomatik en iyi aday

if scripts:
    if SELECTED_SCRIPT_CANDIDATE is None:
        scored = [r for r in results if r['task'] == 'script-generation'
                  and r['candidate'] in scripts and r['quality_score_1_5'] is not None]
        if scored:
            SELECTED_SCRIPT_CANDIDATE = max(scored, key=lambda r: r['quality_score_1_5'])['candidate']
        else:
            SELECTED_SCRIPT_CANDIDATE = next(iter(scripts))
    if SELECTED_SCRIPT_CANDIDATE not in scripts:
        raise ValueError(
            f"SELECTED_SCRIPT_CANDIDATE='{SELECTED_SCRIPT_CANDIDATE}' scripts icinde yok. "
            f'Mevcutlar: {list(scripts)}'
        )
    print('Secilen script adayi:', SELECTED_SCRIPT_CANDIDATE)
    print()
    print(scripts[SELECTED_SCRIPT_CANDIDATE])
else:
    print('Uretilmis script yok (an onaylanmadi ya da API anahtari yok). '
          'Sadece prompt paketi disa aktarilacak.')


## 8. Final script seçimi ve onayı

`SELECTED_SCRIPT_CANDIDATE = None` bırakılırsa en yüksek `quality_score_1_5` puanlı aday (yoksa ilk üretilen) otomatik seçilir. **Bu script'i yayına/TTS'e vermeden önce `SCRIPT_APPROVED = True` yapman gerekir.**

In [ ]:
SCRIPTS_DIR = RESULTS_DIR / 'scripts'
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
for name, text in scripts.items():
    (SCRIPTS_DIR / f'{name}.txt').write_text(text, encoding='utf-8')

df = pd.DataFrame(results)
out_path = RESULTS_DIR / '02_script_generation_sonuclar.csv'
if not df.empty:
    display(df)
    df.to_csv(out_path, index=False)
else:
    pd.DataFrame(columns=['task', 'candidate', 'quality_score_1_5', 'cost_usd_estimate',
                           'latency_sec', 'notes', 'status']).to_csv(out_path, index=False)
print('Kaydedildi:', out_path)

# Kanonik artefakt olusturma (master notebook ile ayni fonksiyon). Bu fonksiyon
# onay argumani ALMAZ ve asla 'approved' uretemez -- yeni uretilen script her
# zaman 'review_required' olur. Onay bolum 10'daki ayri hucrede verilir.
benchmark_attempts = [
    {'provider': a.get('provider'), 'model': a.get('model'),
     'ok': 'script' in a, 'error': a.get('error')}
    for a in attempts
]

if scripts and SELECTED_SCRIPT_CANDIDATE:
    meta = script_meta[SELECTED_SCRIPT_CANDIDATE]
    quality_score = next((r['quality_score_1_5'] for r in results
                          if r['candidate'] == SELECTED_SCRIPT_CANDIDATE), None)
    selected_script = script_generation.build_selected_script_artifact(
        scripts[SELECTED_SCRIPT_CANDIDATE], meta['provider'], meta['model'],
        meta['cost_usd'], meta['latency_sec'], SELECTED_MOMENT, moment_review_record,
        hot_moments_data.get('source') or {}, hot_moments_path,
        quality_score_1_5=quality_score, attempts=benchmark_attempts,
    )
else:
    selected_script = script_generation.build_selected_script_artifact(
        None, None, None, 0.0, 0.0, SELECTED_MOMENT, moment_review_record,
        hot_moments_data.get('source') or {}, hot_moments_path,
        prompt_package={'user_prompt': USER_PROMPT_TEXT, 'content_mode': CONTENT_MODE,
                         'tone': TONE, 'target_duration_sec': TARGET_DURATION_SEC},
        status_reason_override=(
            'moment not approved (MOMENT_APPROVED=False)' if not MOMENT_APPROVED
            else 'no API keys available or all attempts failed; prompt package exported'
        ),
        attempts=benchmark_attempts,
    )

selected_script_path = RESULTS_DIR / 'selected_script.json'
with open(selected_script_path, 'w', encoding='utf-8') as f:
    json.dump(selected_script, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', selected_script_path, '(status:', selected_script['status'], ')')

zip_path = RESULTS_DIR / '02_script_generation_paylasim_paketi.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(selected_script_path, arcname='selected_script.json')
    if out_path.exists():
        archive.write(out_path, arcname=out_path.name)
    for script_file in SCRIPTS_DIR.glob('*.txt'):
        archive.write(script_file, arcname=f'scripts/{script_file.name}')
print('Paylasim paketi hazir:', zip_path)


## 10. İnsan onayı (ayrı adım, üretimden sonra)

Yukarıdaki scripti **okuduktan sonra** inceleme çıktısının yazdırdığı SHA-256'yı
`SCRIPT_REVIEWED_SHA256` alanına yapıştır. Hash eşleşmezse onay reddedilir; script sonradan
değişirse onay otomatik geçersizleşir.

In [ ]:
if selected_script.get('script_text'):
    script_review_info = approval.describe_for_review('script', selected_script)
    print('Inceledigin scriptin SHA-256 degeri (asagiya yapistir):')
    print('   ', script_review_info['content_sha256'])
else:
    print('Onaylanacak bir script yok (sadece prompt paketi).')

SCRIPT_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
SCRIPT_APPROVED_BY = ''       # ornek: 'insan: birkan'
SCRIPT_APPROVAL_NOTES = ''

if not selected_script.get('script_text'):
    pass
elif not SCRIPT_REVIEWED_SHA256 or not SCRIPT_APPROVED_BY:
    print('Onay ATLANDI: SCRIPT_REVIEWED_SHA256 ve SCRIPT_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', selected_script['status'])
else:
    try:
        selected_script = approval.approve_artifact(
            selected_script, 'script', SCRIPT_REVIEWED_SHA256,
            SCRIPT_APPROVED_BY, SCRIPT_APPROVAL_NOTES,
        )
        with open(selected_script_path, 'w', encoding='utf-8') as f:
            json.dump(selected_script, f, ensure_ascii=False, indent=2)
        print('Script ONAYLANDI ->', selected_script_path, '(status:', selected_script['status'], ')')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', selected_script['status'])


## 9. Dışa aktarım